In [0]:
"""
id: source_3f9a1c2e
template: source
templateVersion: 3.0.0
name: customer_ind
parentId: group_0
position:
  x: 30
  y: 155.5
codegenHash: f836ccce6b5bf6bc
updatedAt: 1792940614983
config:
  table_source:
    tableName: workspace.visual_data_prep.customer_ind
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(payload: Dict[str, Any], file_url: str) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(result: Dict[str, Any], file_url: str, spark):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(result, mcp_file_source.get("fileUrl", ""), spark)

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "workspace.visual_data_prep.customer_ind"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_3f9a1c2e.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_3f9a1c2e" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_3f9a1c2e.data"])
if "source_3f9a1c2e" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_3f9a1c2e", "counts": {"data": int(ctx["source_3f9a1c2e.data"].count())}}}, raw=True)

In [0]:
"""
id: source_51f8d2a6
template: source
templateVersion: 3.0.0
name: products
parentId: group_e7a14c3b
position:
  x: 30
  y: 155.5
codegenHash: f836ccce6b5bf6bc
updatedAt: 1792940615159
config:
  table_source:
    tableName: workspace.visual_data_prep.products
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(payload: Dict[str, Any], file_url: str) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(result: Dict[str, Any], file_url: str, spark):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(result, mcp_file_source.get("fileUrl", ""), spark)

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "workspace.visual_data_prep.products"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_51f8d2a6.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_51f8d2a6" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_51f8d2a6.data"])
if "source_51f8d2a6" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_51f8d2a6", "counts": {"data": int(ctx["source_51f8d2a6.data"].count())}}}, raw=True)

In [0]:
"""
id: source_8e3a6c15
template: source
templateVersion: 3.0.0
name: transactions
parentId: group_2b6f9e0d
position:
  x: 30
  y: 155.5
codegenHash: f836ccce6b5bf6bc
updatedAt: 1792940615248
config:
  table_source:
    tableName: workspace.visual_data_prep.transactions
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(payload: Dict[str, Any], file_url: str) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(result: Dict[str, Any], file_url: str, spark):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(result, mcp_file_source.get("fileUrl", ""), spark)

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "workspace.visual_data_prep.transactions"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_8e3a6c15.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_8e3a6c15" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_8e3a6c15.data"])
if "source_8e3a6c15" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_8e3a6c15", "counts": {"data": int(ctx["source_8e3a6c15.data"].count())}}}, raw=True)

In [0]:
"""
id: source_a4c71b09
template: source
templateVersion: 3.0.0
name: customer_usa
parentId: group_5d2e8a1f
position:
  x: 30
  y: 155.5
description:
  text: Read data from the 'workspace.visual_data_prep.customer_usa' table source.
  hash: 8603d4e9
previewCodeHash: 4b37b81767d0d85e
codegenHash: f836ccce6b5bf6bc
updatedAt: 1792940647159
config:
  table_source:
    tableName: workspace.visual_data_prep.customer_usa
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(payload: Dict[str, Any], file_url: str) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(result: Dict[str, Any], file_url: str, spark):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(result, mcp_file_source.get("fileUrl", ""), spark)

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "workspace.visual_data_prep.customer_usa"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_a4c71b09.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_a4c71b09" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_a4c71b09.data"])
if "source_a4c71b09" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_a4c71b09", "counts": {"data": int(ctx["source_a4c71b09.data"].count())}}}, raw=True)

In [0]:
"""
id: source_d6e29a40
template: source
templateVersion: 3.0.0
name: customer_uk
parentId: group_8c3f1d6a
position:
  x: 30
  y: 155.5
codegenHash: f836ccce6b5bf6bc
updatedAt: 1792940615068
config:
  table_source:
    tableName: workspace.visual_data_prep.customer_uk
checkpoint:
  mode: auto
input: []
"""

# generated from the system
import atexit
import base64
import csv
import hashlib
import io
import json
import os
import re
import shutil
import tempfile
import time
from typing import Dict, Any, List, Optional
from urllib.parse import parse_qs, unquote, urlparse

_MANAGED_MCP_SERVICES = {
    "google_drive": "system.ai.google_drive",
    "microsoft_365": "system.ai.microsoft_365",
}
_GOOGLE_SHEETS_MIME_TYPE = "application/vnd.google-apps.spreadsheet"
_EXCEL_WORKBOOK_MIME_TYPE = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
_GOOGLE_DELIMITED_FILE_MIME_TYPES = {
    "application/csv",
    "text/csv",
    "text/tab-separated-values",
}
_MICROSOFT_CSV_FILE_EXTENSION = ".csv"
_MICROSOFT_SAFE_TEXT_PAGE_CHARS = 20_000
_MICROSOFT_CONTENT_TOO_LARGE_ERROR = (
    "Content too large to return even after truncation. "
    "Use smaller limit, offset, or spreadsheet_range for targeted reads."
)
_MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES = 16 * 1024 * 1024
_MANAGED_SOURCE_CACHE_MAX_DISK_BYTES = 256 * 1024 * 1024
_MANAGED_SOURCE_CACHE_TTL_SECONDS = 15 * 60
_MANAGED_SOURCE_MAX_PAGES = 1_000
_MANAGED_SOURCE_CACHE_GLOBAL = "_lb_mcp_source_payload_cache"
_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL = "_lb_mcp_source_payload_cache_directory"

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def _decode_jsonrpc_response(response: Any) -> Dict[str, Any]:
    if isinstance(response, dict) and "contents" in response:
        stream = response["contents"]
        try:
            response = stream.read()
        finally:
            stream.close()
    if isinstance(response, dict):
        return response
    if isinstance(response, bytes):
        response = response.decode("utf-8")
    if not isinstance(response, str):
        raise ValueError("Managed MCP service returned an unsupported response")
    if response.lstrip().startswith("{"):
        parsed = json.loads(response)
        if isinstance(parsed, dict):
            return parsed
    data_parts = [
        line.strip()[len("data:"):].strip()
        for line in response.splitlines()
        if line.strip().startswith("data:")
    ]
    if data_parts:
        parsed = json.loads("\n".join(data_parts))
        if isinstance(parsed, dict) and ("result" in parsed or "error" in parsed):
            return parsed
    raise ValueError("Managed MCP service returned an invalid JSON-RPC response")

def _post_mcp_jsonrpc(api_client, service_name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    response = api_client.do(
        method="POST",
        path=f"/ai-gateway/mcp-services/{service_name}",
        body=payload,
        headers={
            "Accept": "application/json, text/event-stream",
            "Accept-Encoding": "identity",
            "Content-Type": "application/json",
        },
        raw=True,
    )
    decoded = _decode_jsonrpc_response(response)
    if decoded.get("error"):
        raise ValueError(f"Managed MCP request failed: {decoded['error']}")
    return decoded

def _call_mcp_tool(service_name: str, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    from databricks.sdk import WorkspaceClient

    api_client = WorkspaceClient().api_client
    reply = _post_mcp_jsonrpc(
        api_client,
        service_name,
        {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": arguments},
        },
    )
    result = reply.get("result")
    if not isinstance(result, dict):
        raise ValueError("Managed MCP tool returned no result")
    text_parts = [
        item.get("text", "")
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    if result.get("isError"):
        detail = " ".join(part for part in text_parts if part)
        raise ValueError(f"Managed MCP tool failed: {detail or tool_name}")
    structured = result.get("structuredContent")
    if isinstance(structured, dict):
        return structured
    for text in text_parts:
        try:
            payload = json.loads(text)
        except (TypeError, json.JSONDecodeError):
            continue
        if isinstance(payload, dict):
            return payload
    raise ValueError("Managed MCP tool returned no structured spreadsheet content")

def _managed_source_cache_identity():
    try:
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        current_user = client.current_user.me()
        user_id = getattr(current_user, "id", None)
        host = getattr(client.config, "host", None)
        if user_id and host:
            return (str(host), str(user_id))
    except Exception:
        pass
    return None

def _managed_source_cache_key(identity, *parts: Any) -> str:
    return json.dumps((identity, *parts), ensure_ascii=False, separators=(",", ":"), default=str)

def _managed_source_cache_remove(cache: Dict[str, Any], key: str) -> None:
    entry = cache.pop(key, None)
    path = entry.get("path") if isinstance(entry, dict) else None
    if isinstance(path, str):
        try:
            os.remove(path)
        except OSError:
            pass

def _managed_source_cache_directory() -> str:
    existing = globals().get(_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL)
    if isinstance(existing, str) and os.path.isdir(existing):
        return existing
    directory = tempfile.mkdtemp(prefix="lakebuilder-mcp-")
    globals()[_MANAGED_SOURCE_CACHE_DIRECTORY_GLOBAL] = directory
    atexit.register(shutil.rmtree, directory, ignore_errors=True)
    return directory

def _managed_source_cache_cleanup(cache: Dict[str, Any], now: float) -> None:
    expired_keys = [
        key
        for key, entry in cache.items()
        if not isinstance(entry, dict)
        or now - entry.get("last_accessed", 0) >= _MANAGED_SOURCE_CACHE_TTL_SECONDS
    ]
    for key in expired_keys:
        _managed_source_cache_remove(cache, key)

def _managed_source_cache_get(key: str, version: Any = None):
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return None
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    entry = cache.get(key)
    if not isinstance(entry, dict):
        cache.pop(key, None)
        return None
    if globals().get("_lb_force", False) or entry.get("version") != version:
        _managed_source_cache_remove(cache, key)
        return None
    value = entry.get("value")
    if isinstance(value, dict):
        entry["last_accessed"] = now
        return value
    path = entry.get("path")
    if isinstance(path, str):
        try:
            with open(path, encoding="utf-8") as handle:
                value = json.load(handle)
                if isinstance(value, dict):
                    entry["last_accessed"] = now
                    return value
        except (OSError, json.JSONDecodeError):
            pass
    _managed_source_cache_remove(cache, key)
    return None

def _managed_source_cache_put(key: str, value: Dict[str, Any], version: Any = None) -> None:
    if not globals().get("_lb_mcp_preview_cache_enabled", False):
        return
    encoded = json.dumps(value, ensure_ascii=False, default=str).encode("utf-8")
    cache = globals().setdefault(_MANAGED_SOURCE_CACHE_GLOBAL, {})
    now = time.monotonic()
    _managed_source_cache_cleanup(cache, now)
    _managed_source_cache_remove(cache, key)
    memory_size = sum(
        entry.get("size", 0)
        for entry in cache.values()
        if isinstance(entry, dict) and isinstance(entry.get("value"), dict)
    )
    if memory_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_MEMORY_BYTES:
        cache[key] = {
            "size": len(encoded),
            "value": value,
            "version": version,
            "last_accessed": now,
        }
        return
    if len(encoded) > _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
        return
    disk_entries = sorted(
        (
            (entry.get("last_accessed", 0), entry_key, entry.get("size", 0))
            for entry_key, entry in cache.items()
            if isinstance(entry, dict) and isinstance(entry.get("path"), str)
        ),
    )
    disk_size = sum(size for _, _, size in disk_entries)
    for _, entry_key, size in disk_entries:
        if disk_size + len(encoded) <= _MANAGED_SOURCE_CACHE_MAX_DISK_BYTES:
            break
        _managed_source_cache_remove(cache, entry_key)
        disk_size -= size
    temporary_path = None
    try:
        directory = _managed_source_cache_directory()
        path = os.path.join(directory, hashlib.sha256(key.encode("utf-8")).hexdigest() + ".json")
        file_descriptor, temporary_path = tempfile.mkstemp(prefix=".tmp-", dir=directory)
        with os.fdopen(file_descriptor, "wb") as handle:
            handle.write(encoded)
        os.replace(temporary_path, path)
    except OSError:
        if temporary_path is not None:
            try:
                os.remove(temporary_path)
            except OSError:
                pass
        return
    cache[key] = {
        "size": len(encoded),
        "path": path,
        "version": version,
        "last_accessed": now,
    }

def _extract_google_file_id(file_url: str) -> str:
    file_url = file_url.strip()
    match = re.search(r"/(?:spreadsheets|file)(?:/u/[0-9]+)?/d/([^/?#]+)", file_url)
    if match:
        return match.group(1)
    query_id = parse_qs(urlparse(file_url).query).get("id", [None])[0]
    if query_id:
        return query_id
    if re.fullmatch(r"[A-Za-z0-9_-]{5,}", file_url):
        return file_url
    raise ValueError("Google Drive source requires a valid Google Drive URL or file ID")

def _collect_sharepoint_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if isinstance(value, dict):
        parent = value.get("parentReference")
        drive_id = parent.get("driveId") if isinstance(parent, dict) else None
        item_id = value.get("id")
        if drive_id and item_id:
            candidates.append({
                "drive_id": str(drive_id),
                "item_id": str(item_id),
                "name": str(value.get("name") or ""),
                "web_url": str(value.get("webUrl") or ""),
            })
        for child in value.values():
            _collect_sharepoint_candidates(child, candidates)
    elif isinstance(value, list):
        for child in value:
            _collect_sharepoint_candidates(child, candidates)

def _collect_sharepoint_search_candidates(value: Any, candidates: List[Dict[str, str]]) -> None:
    if not isinstance(value, dict):
        return
    for search_response in value.get("value", []):
        if not isinstance(search_response, dict):
            continue
        for container in search_response.get("hitsContainers", []):
            if not isinstance(container, dict):
                continue
            for hit in container.get("hits", []):
                if not isinstance(hit, dict):
                    continue
                resource = hit.get("resource")
                if isinstance(resource, dict):
                    parent = resource.get("parentReference")
                    drive_id = parent.get("driveId") if isinstance(parent, dict) else None
                    item_id = resource.get("id")
                    if drive_id and item_id:
                        candidates.append({
                            "drive_id": str(drive_id),
                            "item_id": str(item_id),
                            "name": str(resource.get("name") or ""),
                            "web_url": str(resource.get("webUrl") or ""),
                        })

def _normalized_url_path(value: str) -> str:
    parsed = urlparse(value)
    return unquote(parsed.path).rstrip("/").casefold()

def _sharepoint_search_name(file_url: str) -> str:
    parsed = urlparse(file_url)
    query = parse_qs(parsed.query)
    for key in ("id", "file", "filename"):
        value = query.get(key, [None])[0]
        if value:
            name = unquote(value).rstrip("/").split("/")[-1]
            if name:
                return name
    name = unquote(parsed.path).rstrip("/").split("/")[-1]
    if name.casefold() not in {"allitems.aspx", "doc.aspx", "view.aspx"} and name:
        return name
    raise ValueError("SharePoint source URL does not identify a supported file; use its Preview URL")

def _microsoft_share_id(file_url: str) -> str:
    parsed = urlparse(file_url)
    host = (parsed.hostname or "").casefold()
    sharepoint_url = host.endswith(".sharepoint.com") and bool(
        re.match(r"^/:[a-z]:/", parsed.path, re.IGNORECASE)
    )
    onedrive_url = host == "1drv.ms" or host.endswith(".1drv.ms") or host == "onedrive.live.com"
    if not sharepoint_url and not onedrive_url:
        return ""
    encoded = base64.urlsafe_b64encode(file_url.encode("utf-8")).decode("ascii").rstrip("=")
    return f"u!{encoded}"

def _resource_uri_from_drive_item(value: Any, error_message: str) -> str:
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_candidates(value, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    if len(deduplicated) != 1:
        raise ValueError(error_message)
    drive_id, item_id = next(iter(deduplicated))
    return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

def _resolve_microsoft_resource_uri(file_url: str) -> str:
    direct = re.search(r"/v1\.0/drives/([^/]+)/items/([^/?#]+)", file_url)
    if direct:
        return f"/v1.0/drives/{direct.group(1)}/items/{direct.group(2)}/content"

    parsed = urlparse(file_url)
    query = {key.casefold(): values for key, values in parse_qs(parsed.query).items()}
    drive_id = query.get("driveid", [None])[0]
    item_id = query.get("itemid", [None])[0]
    if drive_id and item_id:
        return f"/v1.0/drives/{drive_id}/items/{item_id}/content"

    share_id = _microsoft_share_id(file_url)
    if share_id:
        drive_item = _call_mcp_tool(
            "system.ai.microsoft_365",
            "read_resource",
            {"resource_uri": f"/v1.0/shares/{share_id}/driveItem"},
        )
        return _resource_uri_from_drive_item(
            drive_item,
            "Could not resolve the SharePoint or OneDrive sharing URL to a supported file.",
        )

    file_name = _sharepoint_search_name(file_url)
    search_payload = _call_mcp_tool(
        "system.ai.microsoft_365",
        "sharepoint_search",
        {"query": file_name, "max_results": 100},
    )
    candidates: List[Dict[str, str]] = []
    _collect_sharepoint_search_candidates(search_payload, candidates)
    deduplicated = {
        (candidate["drive_id"], candidate["item_id"]): candidate
        for candidate in candidates
    }
    candidates = list(deduplicated.values())
    input_path = _normalized_url_path(file_url)
    exact = [
        candidate
        for candidate in candidates
        if candidate["web_url"]
        and (
            input_path.endswith(_normalized_url_path(candidate["web_url"]))
            or _normalized_url_path(candidate["web_url"]).endswith(input_path)
        )
    ]
    matches = exact or [candidate for candidate in candidates if candidate["name"].casefold() == file_name.casefold()]
    if len(matches) != 1:
        raise ValueError(
            "Could not uniquely resolve the SharePoint file URL. Open the file in Preview and copy the browser URL."
        )
    match = matches[0]
    return f"/v1.0/drives/{match['drive_id']}/items/{match['item_id']}/content"

def _read_google_delimited_source(service_name: str, file_id: str) -> Dict[str, Any]:
    chunks: List[str] = []
    offset = 0
    metadata: Dict[str, Any] = {}
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        try:
            payload = _call_mcp_tool(
                service_name,
                "google_file_read",
                {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": offset},
            )
        except ValueError:
            if offset == 0:
                return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
            raise
        page_start = payload.get("start_char_offset")
        delivered = payload.get("reading_length_in_chars")
        if offset == 0 and page_start is None and delivered is None:
            return _call_mcp_tool(service_name, "google_file_download", {"file_id": file_id})
        content = payload.get("content")
        if not isinstance(content, str):
            raise ValueError("Managed MCP delimited-text response has no text content")
        truncated = payload.get("truncated")
        if page_start != offset or delivered != len(content) or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            metadata = page_metadata
        chunks.append(content)
        if not truncated:
            return {
                "content": "".join(chunks),
                "name": metadata.get("name"),
                "mime_type": metadata.get("mime_type"),
                "truncated": False,
            }
        if delivered == 0:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += delivered
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _sheet_name(range_name: str) -> str:
    name = range_name.split("!", 1)[0].strip()
    if len(name) >= 2 and name[0] == "'" and name[-1] == "'":
        name = name[1:-1].replace("''", "'")
    return name

def _target_spreadsheet_sheet(payload: Dict[str, Any], file_url: str):
    gid = parse_qs(urlparse(file_url).fragment).get("gid", [None])[0]
    metadata = payload.get("metadata")
    tabs = metadata.get("sheet_tabs", []) if isinstance(metadata, dict) else []
    if gid:
        for tab in tabs:
            if isinstance(tab, dict) and str(tab.get("sheetId")) == str(gid):
                return tab.get("title")
    return None

def _read_microsoft_spreadsheet_source(
    service_name: str,
    arguments: Dict[str, Any],
    initial_payload: Dict[str, Any],
    file_url: str,
) -> Dict[str, Any]:
    pages: List[List[Any]] = []
    range_name = None
    offset = 0
    first_page = None
    metadata_version = None
    read_arguments = dict(arguments)
    pending_payload = initial_payload
    target_sheet = _target_spreadsheet_sheet(initial_payload, file_url)
    if target_sheet is not None and "spreadsheet_range" not in read_arguments:
        read_arguments["spreadsheet_range"] = target_sheet
        pending_payload = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        if pending_payload is not None:
            payload = pending_payload
            pending_payload = None
        else:
            payload = _call_mcp_tool(
                service_name,
                "read_resource",
                {**read_arguments, "content_offset": offset},
            )
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
        truncated = payload.get("truncated")
        metadata = payload.get("metadata")
        if isinstance(metadata, dict):
            page_metadata_version = (
                metadata.get("id"),
                metadata.get("size"),
                metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 workbook changed during pagination")
        elif metadata_version is not None:
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if truncated is False and first_page is None:
            return payload
        content = payload.get("content")
        if not isinstance(content, dict) or not isinstance(metadata, dict):
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        sheet_names = metadata.get("sheet_names")
        if (
            first_page is None
            and "spreadsheet_range" not in read_arguments
            and (len(content) != 1 or isinstance(sheet_names, list) and len(sheet_names) > 1)
        ):
            selected_range = next(
                (
                    page_range
                    for page_range, page_content in content.items()
                    if isinstance(page_content, dict)
                    and not page_content.get("error")
                    and isinstance(page_content.get("values"), list)
                    and page_content["values"]
                ),
                None,
            )
            if selected_range is None:
                raise ValueError("Managed MCP spreadsheet contains no readable sheet rows")
            read_arguments["spreadsheet_range"] = _sheet_name(str(selected_range))
            continue
        if len(content) != 1:
            raise ValueError(
                "Managed MCP spreadsheet content was truncated without valid pagination metadata. "
                "Configure spreadsheetRange to read a single range."
            )
        page_range, page_content = next(iter(content.items()))
        values = page_content.get("values") if isinstance(page_content, dict) else None
        next_offset = metadata.get("next_offset")
        has_more = metadata.get("has_more")
        if (
            not isinstance(values, list)
            or payload.get("offset") != offset
            or isinstance(next_offset, bool)
            or not isinstance(next_offset, int)
            or next_offset != offset + len(values)
            or not isinstance(has_more, bool)
            or has_more != truncated
            or (range_name is not None and page_range != range_name)
        ):
            raise ValueError("Managed MCP spreadsheet response has invalid pagination metadata")
        if first_page is None:
            first_page = payload
            range_name = page_range
        pages.extend(values)
        if not has_more:
            combined = {**first_page, **payload}
            first_content = first_page["content"][range_name]
            combined_content = {
                range_name: {
                    **first_content,
                    **page_content,
                    "values": pages,
                    "row_count": len(pages),
                }
            }
            combined["content"] = combined_content
            combined_metadata = {
                **first_page["metadata"],
                **metadata,
                "content_length": len(json.dumps(combined_content, indent=2)),
                "returned_rows": len(pages),
            }
            combined["metadata"] = combined_metadata
            combined["truncated"] = False
            combined["offset"] = 0
            combined.pop("hint", None)
            return combined
        if next_offset <= offset:
            raise ValueError("Managed MCP spreadsheet pagination made no progress")
        offset = next_offset
    raise ValueError("Managed MCP spreadsheet read exceeded the pagination limit")

def _read_microsoft_source(
    service_name: str,
    resource_uri: str,
    spreadsheet_range: Optional[str],
    file_url: str,
):
    chunks: List[str] = []
    offset = 0
    content_limit = -1
    metadata: Dict[str, Any] = {}
    metadata_version = None
    for _ in range(_MANAGED_SOURCE_MAX_PAGES):
        arguments: Dict[str, Any] = {
            "resource_uri": resource_uri,
            "content_limit": content_limit,
            "content_offset": offset,
        }
        if spreadsheet_range:
            arguments["spreadsheet_range"] = spreadsheet_range
        payload = _call_mcp_tool(service_name, "read_resource", arguments)
        if payload.get("error") or payload.get("error_code"):
            raise ValueError(f"Managed MCP file read failed: {payload}")
        content = payload.get("content")
        if isinstance(content, dict):
            if offset != 0:
                raise ValueError("Managed Microsoft 365 response changed content type during pagination")
            return "spreadsheet", _read_microsoft_spreadsheet_source(
                service_name,
                arguments,
                payload,
                file_url,
            )
        if not isinstance(content, str):
            raise ValueError("Managed Microsoft 365 file response has no supported content")

        page_metadata = payload.get("metadata")
        if isinstance(page_metadata, dict):
            page_metadata_version = (
                page_metadata.get("id"),
                page_metadata.get("size"),
                page_metadata.get("lastModifiedDateTime"),
            )
            if metadata_version is None:
                metadata_version = page_metadata_version
            elif page_metadata_version != metadata_version:
                raise ValueError("Managed Microsoft 365 file changed during pagination")
            metadata = page_metadata
        file_name = str(metadata.get("name") or "")
        if not file_name.casefold().endswith(_MICROSOFT_CSV_FILE_EXTENSION):
            raise ValueError("Managed Microsoft 365 sources support Excel and CSV files")
        if spreadsheet_range:
            raise ValueError("Managed Microsoft 365 CSV sources do not support spreadsheetRange")

        decode_error = (
            f"File '{file_name}' could not be decoded as UTF-8 text. "
            "The file may use a different encoding. "
            "Please access it directly via the web link."
        )
        if content == decode_error:
            raise ValueError("Managed Microsoft 365 CSV source must be UTF-8 encoded")
        if content == _MICROSOFT_CONTENT_TOO_LARGE_ERROR:
            if content_limit == -1:
                content_limit = _MICROSOFT_SAFE_TEXT_PAGE_CHARS
                continue
            raise ValueError("Managed Microsoft 365 CSV page exceeds the response size limit")

        page_start = payload.get("offset")
        truncated = payload.get("truncated")
        if page_start != offset or not isinstance(truncated, bool):
            raise ValueError("Managed MCP delimited-text response has invalid pagination metadata")
        chunks.append(content)
        if not truncated:
            return "delimited_text", {
                "metadata": metadata,
                "content": "".join(chunks),
                "name": file_name,
                "truncated": False,
            }
        if not content:
            raise ValueError("Managed MCP delimited-text pagination made no progress")
        offset += len(content)
    raise ValueError("Managed MCP delimited-text read exceeded the pagination limit")

def _read_google_excel_source(
    service_name: str, arguments: Dict[str, Any]
) -> Dict[str, Any]:
    read_arguments = dict(arguments)
    payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
    metadata = payload.get("metadata")
    if not isinstance(metadata, dict):
        return payload

    has_more_present = "has_more" in metadata
    next_offset_present = "next_offset" in metadata
    if not has_more_present and not next_offset_present:
        content = payload.get("content")
        if (
            payload.get("truncated") is not True
            or "spreadsheet_range" in read_arguments
            or not isinstance(content, dict)
        ):
            return payload
        selected_range = next(
            (
                page_range
                for page_range, page_content in content.items()
                if isinstance(page_content, dict)
                and not page_content.get("error")
                and isinstance(page_content.get("values"), list)
                and page_content["values"]
            ),
            None,
        )
        if selected_range is None:
            raise ValueError("Managed MCP Excel workbook contains no readable sheet rows")
        read_arguments["spreadsheet_range"] = selected_range
        read_arguments["offset"] = 0
        payload = _call_mcp_tool(service_name, "google_file_read", read_arguments)
        metadata = payload.get("metadata")
        if not isinstance(metadata, dict):
            return payload

    has_more = metadata.get("has_more")
    next_offset = metadata.get("next_offset")
    content = payload.get("content")
    if (
        not isinstance(has_more, bool)
        or not isinstance(next_offset, int)
        or isinstance(next_offset, bool)
        or not isinstance(content, dict)
        or len(content) != 1
    ):
        raise ValueError("Managed MCP Excel response has invalid pagination metadata")

    range_name, aggregate_entry = next(iter(content.items()))
    if not isinstance(aggregate_entry, dict) or not isinstance(aggregate_entry.get("values"), list):
        raise ValueError("Managed MCP Excel response has invalid pagination content")
    aggregate_values = aggregate_entry["values"]
    if next_offset != len(aggregate_values):
        raise ValueError("Managed MCP Excel pagination has a gap or overlap")
    if not has_more:
        return payload

    offset = next_offset
    for _ in range(_MANAGED_SOURCE_MAX_PAGES - 1):
        page_arguments = dict(read_arguments)
        page_arguments["offset"] = offset
        page = _call_mcp_tool(service_name, "google_file_read", page_arguments)
        if page.get("error") or page.get("error_code"):
            raise ValueError(f"Managed MCP spreadsheet read failed: {page}")

        page_metadata = page.get("metadata")
        page_content = page.get("content")
        if not isinstance(page_metadata, dict) or not isinstance(page_content, dict):
            raise ValueError("Managed MCP Excel response has invalid pagination metadata")
        page_has_more = page_metadata.get("has_more")
        page_next_offset = page_metadata.get("next_offset")
        page_entry = page_content.get(range_name)
        page_values = page_entry.get("values") if isinstance(page_entry, dict) else None
        if (
            not isinstance(page_has_more, bool)
            or not isinstance(page_next_offset, int)
            or isinstance(page_next_offset, bool)
            or not isinstance(page_values, list)
            or page_next_offset != offset + len(page_values)
        ):
            raise ValueError("Managed MCP Excel pagination has a gap or overlap")

        aggregate_values.extend(page_values)
        aggregate_entry["row_count"] = len(aggregate_values)
        metadata.update(page_metadata)
        if not page_has_more:
            metadata["reading_length_in_chars"] = len(json.dumps(content, indent=2))
            payload["truncated"] = False
            return payload
        if page_next_offset == offset:
            raise ValueError("Managed MCP Excel pagination made no progress")
        offset = page_next_offset

    raise ValueError("Managed MCP Excel read exceeded the pagination limit")

def _read_managed_source(config: Dict[str, Any]) -> Dict[str, Any]:
    provider = config.get("provider")
    service_name = config.get("serviceName")
    file_url = config.get("fileUrl")
    spreadsheet_range = config.get("spreadsheetRange")
    expected_service = _MANAGED_MCP_SERVICES.get(provider)
    if not expected_service or service_name != expected_service:
        raise ValueError("Source: unsupported managed MCP provider or service")
    if not file_url:
        raise ValueError("Source: 'fileUrl' is required for managed MCP source")
    cache_identity = (
        _managed_source_cache_identity()
        if globals().get("_lb_mcp_preview_cache_enabled", False)
        else None
    )

    if provider == "google_drive":
        file_id = _extract_google_file_id(file_url)
        try:
            metadata = _call_mcp_tool(service_name, "google_file_metadata", {"file_id": file_id})
        except ValueError as error:
            detail = str(error).casefold()
            if "404" in detail or "not found" in detail:
                raise ValueError(
                    "Google Drive file was not found or is not accessible to the connected Google account. "
                    "Reconnect with an account that can access the file, or update its sharing permissions."
                ) from error
            raise
        mime_type = str(metadata.get("mimeType") or "").casefold()
        file_name = str(metadata.get("name") or "").casefold()
        cache_version = (
            metadata.get("version"),
            metadata.get("modifiedTime"),
            metadata.get("md5Checksum"),
        )
        if mime_type in {_GOOGLE_SHEETS_MIME_TYPE, _EXCEL_WORKBOOK_MIME_TYPE}:
            arguments: Dict[str, Any] = {
                "file_id": file_id,
                "file_type": "spreadsheet",
                "limit": -1,
            }
            if spreadsheet_range:
                arguments["spreadsheet_range"] = spreadsheet_range
            source_kind = "spreadsheet"
            tool_name = "google_file_read"
        elif mime_type in _GOOGLE_DELIMITED_FILE_MIME_TYPES or file_name.endswith((".csv", ".tsv")):
            if spreadsheet_range:
                raise ValueError("Managed Google Drive CSV and TSV sources do not support spreadsheetRange")
            source_kind = "delimited_text"
            tool_name = "google_file_read"
            arguments = {"file_id": file_id, "file_type": "csv", "limit": -1, "offset": 0}
        else:
            raise ValueError("Managed Google Drive sources support Google Sheets, Excel, CSV, and TSV files")
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            file_id,
            spreadsheet_range,
            source_kind,
        )
    else:
        resource_uri = _resolve_microsoft_resource_uri(file_url)
        source_kind = "microsoft_file"
        cache_key = _managed_source_cache_key(
            cache_identity,
            provider,
            service_name,
            resource_uri,
            spreadsheet_range,
            source_kind,
        )
        cache_version = None

    cached = _managed_source_cache_get(cache_key, cache_version) if cache_identity is not None else None
    if cached is not None:
        return cached
    if provider == "google_drive" and source_kind == "delimited_text":
        payload = _read_google_delimited_source(service_name, file_id)
    elif provider == "google_drive" and mime_type == _EXCEL_WORKBOOK_MIME_TYPE:
        payload = _read_google_excel_source(service_name, arguments)
    elif provider == "microsoft_365":
        source_kind, payload = _read_microsoft_source(
            service_name,
            resource_uri,
            spreadsheet_range,
            file_url,
        )
    else:
        payload = _call_mcp_tool(service_name, tool_name, arguments)

    if payload.get("error") or payload.get("error_code"):
        raise ValueError(f"Managed MCP spreadsheet read failed: {payload}")
    if payload.get("truncated"):
        raise ValueError(
            "Managed MCP spreadsheet content was truncated. Configure spreadsheetRange to read a smaller range."
        )
    result = {"kind": source_kind, "payload": payload}
    if cache_identity is not None:
        _managed_source_cache_put(cache_key, result, cache_version)
    return result

def _select_spreadsheet_values(payload: Dict[str, Any], file_url: str) -> List[List[Any]]:
    content = payload.get("content")
    if not isinstance(content, dict):
        raise ValueError("Managed MCP spreadsheet response has no sheet content")

    target_sheet = _target_spreadsheet_sheet(payload, file_url)

    candidates = []
    for range_name, sheet in content.items():
        if not isinstance(sheet, dict):
            continue
        if sheet.get("error"):
            continue
        values = sheet.get("values")
        if isinstance(values, list) and values:
            candidates.append((str(range_name), values))
    if target_sheet:
        for range_name, values in candidates:
            if _sheet_name(range_name) == target_sheet:
                return values
        raise ValueError(f"Managed MCP response did not contain the selected sheet: {target_sheet}")
    if not candidates:
        raise ValueError("Managed MCP spreadsheet contains no rows")
    return candidates[0][1]

def _normalized_headers(header_row: List[Any], width: int) -> List[str]:
    headers: List[str] = []
    used = set()
    for index in range(width):
        raw = header_row[index] if index < len(header_row) else None
        base = str(raw).strip() if raw is not None else ""
        if not base:
            base = f"_c{index + 1}"
        candidate = base
        suffix = 2
        while candidate.casefold() in used:
            candidate = f"{base}_{suffix}"
            suffix += 1
        headers.append(candidate)
        used.add(candidate.casefold())
    return headers

def _values_dataframe(values: List[List[Any]], spark):
    from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructField, StructType

    if not all(isinstance(row, list) for row in values):
        raise ValueError("Managed MCP tabular source returned malformed rows")
    width = max((len(row) for row in values), default=0)
    if width == 0:
        raise ValueError("Managed MCP tabular source contains no columns")
    headers = _normalized_headers(values[0], width)
    columns = [
        [row[index] if index < len(row) else None for row in values[1:]]
        for index in range(width)
    ]
    fields = []
    normalized_columns = []
    for header, column in zip(headers, columns):
        present = [value for value in column if value is not None]
        if present and all(isinstance(value, bool) for value in present):
            data_type = BooleanType()
            normalized = column
        elif present and all(isinstance(value, int) and not isinstance(value, bool) for value in present):
            data_type = LongType()
            normalized = column
        elif present and all(
            isinstance(value, (int, float)) and not isinstance(value, bool)
            for value in present
        ):
            data_type = DoubleType()
            normalized = [None if value is None else float(value) for value in column]
        elif not present or all(isinstance(value, str) for value in present):
            data_type = StringType()
            normalized = column
        else:
            data_type = StringType()
            normalized = [None if value is None else str(value) for value in column]
        fields.append(StructField(header, data_type, True))
        normalized_columns.append(normalized)
    rows = [list(row) for row in zip(*normalized_columns)]
    schema = StructType(fields)
    return spark.createDataFrame(rows, schema=schema)

def _managed_source_dataframe(result: Dict[str, Any], file_url: str, spark):
    payload = result.get("payload")
    if not isinstance(payload, dict):
        raise ValueError("Managed MCP source returned no content")
    if result.get("kind") == "spreadsheet":
        values = _select_spreadsheet_values(payload, file_url)
    elif result.get("kind") == "delimited_text":
        content = payload.get("content")
        if not isinstance(content, str) or not content:
            raise ValueError("Managed MCP file contains no text")
        delimiter = "	" if str(payload.get("name") or "").casefold().endswith(".tsv") else ","
        try:
            dialect = csv.Sniffer().sniff(content[:4096], delimiters=",	;|")
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), dialect))
        except csv.Error:
            values = list(csv.reader(io.StringIO(content.lstrip("﻿")), delimiter=delimiter))
    else:
        raise ValueError("Managed MCP source returned an unsupported content type")
    return _values_dataframe(values, spark)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")
    mcp_file_source = config.get("mcp_file_source")
    source_modes = [file_source, table_source, mcp_file_source]
    if sum(mode is not None for mode in source_modes) != 1:
        raise ValueError("Source: exactly one of 'file_source', 'table_source', or 'mcp_file_source' must be configured")

    if file_source is not None:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source is not None:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    elif mcp_file_source is not None:
        result = _read_managed_source(mcp_file_source)
        out = _managed_source_dataframe(result, mcp_file_source.get("fileUrl", ""), spark)

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "workspace.visual_data_prep.customer_uk"
    }
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["source_d6e29a40.data"] = out["data"]
if globals().get("ld_display_outputs", False) or "source_d6e29a40" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["source_d6e29a40.data"])
if "source_d6e29a40" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "source_d6e29a40", "counts": {"data": int(ctx["source_d6e29a40.data"].count())}}}, raw=True)

In [0]:
"""
id: guardrails_7b4d2e91
template: guardrails
templateVersion: 1.0.0
name: customer_ind quality checks
parentId: group_0
position:
  x: 330
  y: 155.5
previewCodeHash: e5dd20426e93e8e0
codegenHash: e904b99093c61cc4
updatedAt: 1792940614985
config:
  on_failure:
    mode: warn
  checks:
    - type: not_null
      column: customer_id
    - type: not_null
      column: first_name
    - type: not_null
      column: last_name
    - type: not_null
      column: email
    - type: not_null
      column: phone
    - type: unique
      columns: customer_id
    - type: regex_match
      column: email
      pattern: ^[^@]+@[^@]+\.[^@]+$
checkpoint:
  mode: auto
input:
  - node: source_3f9a1c2e
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)

CheckResult = Optional[Tuple[str, Any]]

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    return F.col(_quoted_ident(name))

def _require_column(df, column: str, check_type: str) -> None:
    if not column:
        raise ValueError(f"{check_type}: 'column' is required")
    if column.lower() not in {c.lower() for c in df.columns}:
        raise ValueError(
            f"{check_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _split_list(raw: str) -> List[str]:
    if not raw:
        return []
    return [item.strip() for item in raw.split(",") if item.strip()]

def _parse_bound(raw: Any, check_type: str, field: str) -> Optional[float]:
    if raw is None:
        return None
    text = str(raw).strip()
    if text == "":
        return None
    try:
        return int(text)
    except ValueError:
        pass
    try:
        return float(text)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{check_type}: '{field}' must be a number, got {raw!r}"
        ) from exc

def _fmt_bound(bound: Any) -> str:
    return f"{bound:g}" if isinstance(bound, float) else str(bound)

def _check_required_columns(df, check: Dict[str, Any]) -> CheckResult:
    columns = _split_list(check.get("columns", ""))
    if not columns:
        raise ValueError("required_columns: 'columns' is required")
    available = {c.lower() for c in df.columns}
    missing = [c for c in columns if c.lower() not in available]
    if missing:
        return (
            f"missing required column(s) {missing}. "
            f"Available columns: {df.columns}",
            F.lit(True),
        )
    return None

def _check_row_count(df, check: Dict[str, Any]) -> CheckResult:
    min_bound = _parse_bound(check.get("min"), "row_count", "min")
    max_bound = _parse_bound(check.get("max"), "row_count", "max")
    if min_bound is None and max_bound is None:
        raise ValueError("row_count: at least one of 'min' or 'max' is required")
    count = df.count()
    if min_bound is not None and count < min_bound:
        return (
            f"row count {count} is below minimum {_fmt_bound(min_bound)}",
            F.lit(True),
        )
    if max_bound is not None and count > max_bound:
        return (
            f"row count {count} is above maximum {_fmt_bound(max_bound)}",
            F.lit(True),
        )
    return None

def _check_not_null(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "not_null")
    predicate = _col(column).isNull()
    violations = df.filter(predicate).count()
    if violations > 0:
        return (f"column '{column}' has {violations} NULL value(s)", predicate)
    return None

def _check_unique(df, check: Dict[str, Any]) -> CheckResult:
    columns = _split_list(check.get("columns", ""))
    if not columns:
        raise ValueError("unique: 'columns' is required")
    for column in columns:
        _require_column(df, column, "unique")
    window = Window.partitionBy(*[_col(c) for c in columns])
    predicate = F.count(F.lit(1)).over(window) > 1
    violations = (
        df.select(predicate.alias("is_duplicate"))
        .filter(F.col("is_duplicate"))
        .count()
    )
    if violations > 0:
        return (
            f"columns {columns} contain {violations} duplicated row(s) "
            f"(expected unique)",
            predicate,
        )
    return None

def _resolved_name(df, column: str) -> str:
    lowered = column.lower()
    for existing in df.columns:
        if existing.lower() == lowered:
            return existing
    return column

def _matches_any(df, column: str, values: List[str]):
    col = _col(column)
    dtype = df.schema[_resolved_name(df, column)].dataType
    matched = F.lit(False)
    for value in values:
        matched = matched | (col == F.lit(value).cast(dtype))
    return F.coalesce(matched, F.lit(False))

def _check_accepted_values(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "accepted_values")
    values = _split_list(check.get("values", ""))
    if not values:
        raise ValueError("accepted_values: 'values' is required")
    col = _col(column)
    predicate = col.isNotNull() & ~_matches_any(df, column, values)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) outside the "
            f"accepted set {values}",
            predicate,
        )
    return None

def _check_disallowed_values(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "disallowed_values")
    values = _split_list(check.get("values", ""))
    if not values:
        raise ValueError("disallowed_values: 'values' is required")
    col = _col(column)
    predicate = col.isNotNull() & _matches_any(df, column, values)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) in the "
            f"disallowed set {values}",
            predicate,
        )
    return None

def _check_value_range(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "value_range")
    min_bound = _parse_bound(check.get("min"), "value_range", "min")
    max_bound = _parse_bound(check.get("max"), "value_range", "max")
    if min_bound is None and max_bound is None:
        raise ValueError(
            "value_range: at least one of 'min' or 'max' is required"
        )
    col = _col(column)
    out_of_range = None
    if min_bound is not None:
        out_of_range = col < F.lit(min_bound)
    if max_bound is not None:
        above = col > F.lit(max_bound)
        out_of_range = above if out_of_range is None else (out_of_range | above)
    predicate = col.isNotNull() & out_of_range
    violations = df.filter(predicate).count()
    if violations > 0:
        bounds = []
        if min_bound is not None:
            bounds.append(f"min={_fmt_bound(min_bound)}")
        if max_bound is not None:
            bounds.append(f"max={_fmt_bound(max_bound)}")
        return (
            f"column '{column}' has {violations} value(s) outside range "
            f"[{', '.join(bounds)}]",
            predicate,
        )
    return None

def _check_regex_match(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    pattern = check.get("pattern", "")
    _require_column(df, column, "regex_match")
    if not pattern:
        raise ValueError("regex_match: 'pattern' is required")
    as_string = _col(column).cast("string")
    predicate = as_string.isNotNull() & ~as_string.rlike(pattern)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) not matching "
            f"pattern {pattern!r}",
            predicate,
        )
    return None

def _check_expression(df, check: Dict[str, Any]) -> CheckResult:
    expression = check.get("expression", "")
    if not expression:
        raise ValueError("expression: 'expression' is required")
    predicate = ~F.coalesce(F.expr(expression), F.lit(False))
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"{violations} row(s) failed expression {expression!r}",
            predicate,
        )
    return None

CHECK_DISPATCH: Dict[str, Callable] = {
    "required_columns": _check_required_columns,
    "row_count": _check_row_count,
    "not_null": _check_not_null,
    "unique": _check_unique,
    "accepted_values": _check_accepted_values,
    "disallowed_values": _check_disallowed_values,
    "value_range": _check_value_range,
    "regex_match": _check_regex_match,
    "expression": _check_expression,
}

_FAILED_COL = "__gr_failed__"

_ERROR_SCHEMA = StructType([
    StructField("check_index", IntegerType(), True),
    StructField("check_type", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("message", StringType(), True),
])

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        raise ValueError(
            "Guardrails: no input dataset. Connect an upstream node to "
            "the 'data' input port before running this check."
        )
    checks: List[Dict[str, Any]] = config.get("checks", []) or []
    raw_on_failure = config.get("on_failure", "warn")
    if isinstance(raw_on_failure, dict):
        mode = raw_on_failure.get("mode", "warn")
    else:
        mode = raw_on_failure or "warn"
    if mode not in ("warn", "block", "stop"):
        mode = "warn"

    error_rows: List[tuple] = []
    marked = df
    failed_cols: List[str] = []
    taken_lower = {c.lower() for c in df.columns}
    for index, check in enumerate(checks):
        if not isinstance(check, dict):
            check_type = ""
            result = (
                f"expected an object, got {type(check).__name__}",
                F.lit(True),
            )
        elif check.get("enabled", True) is False:
            continue
        else:
            check_type = check.get("type", "")
            fn = CHECK_DISPATCH.get(check_type)
            if fn is None:
                result = (
                    f"unsupported check type {check_type!r}. "
                    f"Choose one of: {sorted(CHECK_DISPATCH.keys())}",
                    F.lit(True),
                )
            else:
                try:
                    result = fn(df, check)
                except ValueError as exc:
                    result = (str(exc), F.lit(True))
                except Exception as exc:
                    result = (f"{type(exc).__name__}: {exc}", F.lit(True))
        if result is not None:
            error, predicate = result
            message = f"checks[{index}] ({check_type}): {error}"
            severity = "failure" if mode == "stop" else "warning"
            error_rows.append((index, check_type, severity, message))
            if mode == "stop":
                print(f"[guardrails] FAILURE: {message}")
            else:
                print(f"[guardrails] WARNING: {message}")
            failed_col = f"__gr_failed_{index}__"
            while failed_col.lower() in taken_lower:
                failed_col += "_"
            taken_lower.add(failed_col.lower())
            marked = marked.withColumn(
                failed_col, F.coalesce(predicate, F.lit(False))
            )
            failed_cols.append(failed_col)

    marker_col = _FAILED_COL
    while marker_col.lower() in taken_lower:
        marker_col += "_"
    taken_lower.add(marker_col.lower())
    failed_marker = F.lit(False)
    for failed_col in failed_cols:
        failed_marker = failed_marker | F.col(failed_col)
    marked = marked.withColumn(marker_col, failed_marker)
    drop_cols = [marker_col, *failed_cols]
    passed_df = marked.filter(~F.col(marker_col)).drop(*drop_cols)
    failed_df = marked.filter(F.col(marker_col)).drop(*drop_cols)
    errors_df = spark.createDataFrame(error_rows, _ERROR_SCHEMA)

    meta_state = config.get("meta_state")
    if not isinstance(meta_state, dict):
        meta_state = {}
    is_focused_preview = meta_state.get("is_focused_preview") is True

    if error_rows and mode == "block":
        passed_df = passed_df.limit(0)
    elif error_rows and mode == "stop":
        if not is_focused_preview:
            detail = "\n".join(f"[guardrails] {r[3]}" for r in error_rows)
            raise ValueError(
                f"Guardrails failed the workflow: {len(error_rows)} check(s) "
                f"failed and the failure mode is 'stop' (Fail the workflow)."
                f"\n{detail}"
            )

    return {"passed": passed_df, "failed": failed_df, "errors": errors_df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "on_failure": {
        "mode": "warn"
    },
    "checks": [
        {
            "type": "not_null",
            "column": "customer_id"
        },
        {
            "type": "not_null",
            "column": "first_name"
        },
        {
            "type": "not_null",
            "column": "last_name"
        },
        {
            "type": "not_null",
            "column": "email"
        },
        {
            "type": "not_null",
            "column": "phone"
        },
        {
            "type": "unique",
            "columns": "customer_id"
        },
        {
            "type": "regex_match",
            "column": "email",
            "pattern": "^[^@]+@[^@]+\\.[^@]+$"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_3f9a1c2e.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["guardrails_7b4d2e91.passed"] = out["passed"]
ctx["guardrails_7b4d2e91.failed"] = out["failed"]
ctx["guardrails_7b4d2e91.errors"] = out["errors"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["guardrails_7b4d2e91.passed"], "localCheckpoint", None)) and callable(getattr(ctx["guardrails_7b4d2e91.failed"], "localCheckpoint", None)) and callable(getattr(ctx["guardrails_7b4d2e91.errors"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["guardrails_7b4d2e91.passed"]
        _lb_cp_original_1 = ctx["guardrails_7b4d2e91.failed"]
        _lb_cp_original_2 = ctx["guardrails_7b4d2e91.errors"]
        try:
            ctx["guardrails_7b4d2e91.passed"] = ctx["guardrails_7b4d2e91.passed"].localCheckpoint(eager=True)
            ctx["guardrails_7b4d2e91.failed"] = ctx["guardrails_7b4d2e91.failed"].localCheckpoint(eager=True)
            ctx["guardrails_7b4d2e91.errors"] = ctx["guardrails_7b4d2e91.errors"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["guardrails_7b4d2e91.passed"] is not _lb_cp_original_0:
                try:
                    ctx["guardrails_7b4d2e91.passed"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_7b4d2e91.passed"] = _lb_cp_original_0
            if ctx["guardrails_7b4d2e91.failed"] is not _lb_cp_original_1:
                try:
                    ctx["guardrails_7b4d2e91.failed"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_7b4d2e91.failed"] = _lb_cp_original_1
            if ctx["guardrails_7b4d2e91.errors"] is not _lb_cp_original_2:
                try:
                    ctx["guardrails_7b4d2e91.errors"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_7b4d2e91.errors"] = _lb_cp_original_2
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "guardrails_7b4d2e91" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["guardrails_7b4d2e91.passed"])
    display(ctx["guardrails_7b4d2e91.failed"])
    display(ctx["guardrails_7b4d2e91.errors"])
if "guardrails_7b4d2e91" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "guardrails_7b4d2e91", "counts": {"passed": int(ctx["guardrails_7b4d2e91.passed"].count()), "failed": int(ctx["guardrails_7b4d2e91.failed"].count()), "errors": int(ctx["guardrails_7b4d2e91.errors"].count())}}}, raw=True)

In [0]:
"""
id: guardrails_c0d93b7e
template: guardrails
templateVersion: 1.0.0
name: products quality checks
parentId: group_e7a14c3b
position:
  x: 330
  y: 155.5
previewCodeHash: 7a188bd5940cb283
codegenHash: e904b99093c61cc4
updatedAt: 1792940615159
config:
  on_failure:
    mode: warn
  checks:
    - type: not_null
      column: product_id
    - type: not_null
      column: product_name
    - type: not_null
      column: category
    - type: not_null
      column: brand
    - type: not_null
      column: unit_price
    - type: unique
      columns: product_id
    - type: value_range
      column: unit_price
      min: "0"
checkpoint:
  mode: auto
input:
  - node: source_51f8d2a6
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)

CheckResult = Optional[Tuple[str, Any]]

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    return F.col(_quoted_ident(name))

def _require_column(df, column: str, check_type: str) -> None:
    if not column:
        raise ValueError(f"{check_type}: 'column' is required")
    if column.lower() not in {c.lower() for c in df.columns}:
        raise ValueError(
            f"{check_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _split_list(raw: str) -> List[str]:
    if not raw:
        return []
    return [item.strip() for item in raw.split(",") if item.strip()]

def _parse_bound(raw: Any, check_type: str, field: str) -> Optional[float]:
    if raw is None:
        return None
    text = str(raw).strip()
    if text == "":
        return None
    try:
        return int(text)
    except ValueError:
        pass
    try:
        return float(text)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{check_type}: '{field}' must be a number, got {raw!r}"
        ) from exc

def _fmt_bound(bound: Any) -> str:
    return f"{bound:g}" if isinstance(bound, float) else str(bound)

def _check_required_columns(df, check: Dict[str, Any]) -> CheckResult:
    columns = _split_list(check.get("columns", ""))
    if not columns:
        raise ValueError("required_columns: 'columns' is required")
    available = {c.lower() for c in df.columns}
    missing = [c for c in columns if c.lower() not in available]
    if missing:
        return (
            f"missing required column(s) {missing}. "
            f"Available columns: {df.columns}",
            F.lit(True),
        )
    return None

def _check_row_count(df, check: Dict[str, Any]) -> CheckResult:
    min_bound = _parse_bound(check.get("min"), "row_count", "min")
    max_bound = _parse_bound(check.get("max"), "row_count", "max")
    if min_bound is None and max_bound is None:
        raise ValueError("row_count: at least one of 'min' or 'max' is required")
    count = df.count()
    if min_bound is not None and count < min_bound:
        return (
            f"row count {count} is below minimum {_fmt_bound(min_bound)}",
            F.lit(True),
        )
    if max_bound is not None and count > max_bound:
        return (
            f"row count {count} is above maximum {_fmt_bound(max_bound)}",
            F.lit(True),
        )
    return None

def _check_not_null(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "not_null")
    predicate = _col(column).isNull()
    violations = df.filter(predicate).count()
    if violations > 0:
        return (f"column '{column}' has {violations} NULL value(s)", predicate)
    return None

def _check_unique(df, check: Dict[str, Any]) -> CheckResult:
    columns = _split_list(check.get("columns", ""))
    if not columns:
        raise ValueError("unique: 'columns' is required")
    for column in columns:
        _require_column(df, column, "unique")
    window = Window.partitionBy(*[_col(c) for c in columns])
    predicate = F.count(F.lit(1)).over(window) > 1
    violations = (
        df.select(predicate.alias("is_duplicate"))
        .filter(F.col("is_duplicate"))
        .count()
    )
    if violations > 0:
        return (
            f"columns {columns} contain {violations} duplicated row(s) "
            f"(expected unique)",
            predicate,
        )
    return None

def _resolved_name(df, column: str) -> str:
    lowered = column.lower()
    for existing in df.columns:
        if existing.lower() == lowered:
            return existing
    return column

def _matches_any(df, column: str, values: List[str]):
    col = _col(column)
    dtype = df.schema[_resolved_name(df, column)].dataType
    matched = F.lit(False)
    for value in values:
        matched = matched | (col == F.lit(value).cast(dtype))
    return F.coalesce(matched, F.lit(False))

def _check_accepted_values(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "accepted_values")
    values = _split_list(check.get("values", ""))
    if not values:
        raise ValueError("accepted_values: 'values' is required")
    col = _col(column)
    predicate = col.isNotNull() & ~_matches_any(df, column, values)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) outside the "
            f"accepted set {values}",
            predicate,
        )
    return None

def _check_disallowed_values(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "disallowed_values")
    values = _split_list(check.get("values", ""))
    if not values:
        raise ValueError("disallowed_values: 'values' is required")
    col = _col(column)
    predicate = col.isNotNull() & _matches_any(df, column, values)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) in the "
            f"disallowed set {values}",
            predicate,
        )
    return None

def _check_value_range(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "value_range")
    min_bound = _parse_bound(check.get("min"), "value_range", "min")
    max_bound = _parse_bound(check.get("max"), "value_range", "max")
    if min_bound is None and max_bound is None:
        raise ValueError(
            "value_range: at least one of 'min' or 'max' is required"
        )
    col = _col(column)
    out_of_range = None
    if min_bound is not None:
        out_of_range = col < F.lit(min_bound)
    if max_bound is not None:
        above = col > F.lit(max_bound)
        out_of_range = above if out_of_range is None else (out_of_range | above)
    predicate = col.isNotNull() & out_of_range
    violations = df.filter(predicate).count()
    if violations > 0:
        bounds = []
        if min_bound is not None:
            bounds.append(f"min={_fmt_bound(min_bound)}")
        if max_bound is not None:
            bounds.append(f"max={_fmt_bound(max_bound)}")
        return (
            f"column '{column}' has {violations} value(s) outside range "
            f"[{', '.join(bounds)}]",
            predicate,
        )
    return None

def _check_regex_match(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    pattern = check.get("pattern", "")
    _require_column(df, column, "regex_match")
    if not pattern:
        raise ValueError("regex_match: 'pattern' is required")
    as_string = _col(column).cast("string")
    predicate = as_string.isNotNull() & ~as_string.rlike(pattern)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) not matching "
            f"pattern {pattern!r}",
            predicate,
        )
    return None

def _check_expression(df, check: Dict[str, Any]) -> CheckResult:
    expression = check.get("expression", "")
    if not expression:
        raise ValueError("expression: 'expression' is required")
    predicate = ~F.coalesce(F.expr(expression), F.lit(False))
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"{violations} row(s) failed expression {expression!r}",
            predicate,
        )
    return None

CHECK_DISPATCH: Dict[str, Callable] = {
    "required_columns": _check_required_columns,
    "row_count": _check_row_count,
    "not_null": _check_not_null,
    "unique": _check_unique,
    "accepted_values": _check_accepted_values,
    "disallowed_values": _check_disallowed_values,
    "value_range": _check_value_range,
    "regex_match": _check_regex_match,
    "expression": _check_expression,
}

_FAILED_COL = "__gr_failed__"

_ERROR_SCHEMA = StructType([
    StructField("check_index", IntegerType(), True),
    StructField("check_type", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("message", StringType(), True),
])

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        raise ValueError(
            "Guardrails: no input dataset. Connect an upstream node to "
            "the 'data' input port before running this check."
        )
    checks: List[Dict[str, Any]] = config.get("checks", []) or []
    raw_on_failure = config.get("on_failure", "warn")
    if isinstance(raw_on_failure, dict):
        mode = raw_on_failure.get("mode", "warn")
    else:
        mode = raw_on_failure or "warn"
    if mode not in ("warn", "block", "stop"):
        mode = "warn"

    error_rows: List[tuple] = []
    marked = df
    failed_cols: List[str] = []
    taken_lower = {c.lower() for c in df.columns}
    for index, check in enumerate(checks):
        if not isinstance(check, dict):
            check_type = ""
            result = (
                f"expected an object, got {type(check).__name__}",
                F.lit(True),
            )
        elif check.get("enabled", True) is False:
            continue
        else:
            check_type = check.get("type", "")
            fn = CHECK_DISPATCH.get(check_type)
            if fn is None:
                result = (
                    f"unsupported check type {check_type!r}. "
                    f"Choose one of: {sorted(CHECK_DISPATCH.keys())}",
                    F.lit(True),
                )
            else:
                try:
                    result = fn(df, check)
                except ValueError as exc:
                    result = (str(exc), F.lit(True))
                except Exception as exc:
                    result = (f"{type(exc).__name__}: {exc}", F.lit(True))
        if result is not None:
            error, predicate = result
            message = f"checks[{index}] ({check_type}): {error}"
            severity = "failure" if mode == "stop" else "warning"
            error_rows.append((index, check_type, severity, message))
            if mode == "stop":
                print(f"[guardrails] FAILURE: {message}")
            else:
                print(f"[guardrails] WARNING: {message}")
            failed_col = f"__gr_failed_{index}__"
            while failed_col.lower() in taken_lower:
                failed_col += "_"
            taken_lower.add(failed_col.lower())
            marked = marked.withColumn(
                failed_col, F.coalesce(predicate, F.lit(False))
            )
            failed_cols.append(failed_col)

    marker_col = _FAILED_COL
    while marker_col.lower() in taken_lower:
        marker_col += "_"
    taken_lower.add(marker_col.lower())
    failed_marker = F.lit(False)
    for failed_col in failed_cols:
        failed_marker = failed_marker | F.col(failed_col)
    marked = marked.withColumn(marker_col, failed_marker)
    drop_cols = [marker_col, *failed_cols]
    passed_df = marked.filter(~F.col(marker_col)).drop(*drop_cols)
    failed_df = marked.filter(F.col(marker_col)).drop(*drop_cols)
    errors_df = spark.createDataFrame(error_rows, _ERROR_SCHEMA)

    meta_state = config.get("meta_state")
    if not isinstance(meta_state, dict):
        meta_state = {}
    is_focused_preview = meta_state.get("is_focused_preview") is True

    if error_rows and mode == "block":
        passed_df = passed_df.limit(0)
    elif error_rows and mode == "stop":
        if not is_focused_preview:
            detail = "\n".join(f"[guardrails] {r[3]}" for r in error_rows)
            raise ValueError(
                f"Guardrails failed the workflow: {len(error_rows)} check(s) "
                f"failed and the failure mode is 'stop' (Fail the workflow)."
                f"\n{detail}"
            )

    return {"passed": passed_df, "failed": failed_df, "errors": errors_df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "on_failure": {
        "mode": "warn"
    },
    "checks": [
        {
            "type": "not_null",
            "column": "product_id"
        },
        {
            "type": "not_null",
            "column": "product_name"
        },
        {
            "type": "not_null",
            "column": "category"
        },
        {
            "type": "not_null",
            "column": "brand"
        },
        {
            "type": "not_null",
            "column": "unit_price"
        },
        {
            "type": "unique",
            "columns": "product_id"
        },
        {
            "type": "value_range",
            "column": "unit_price",
            "min": "0"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_51f8d2a6.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["guardrails_c0d93b7e.passed"] = out["passed"]
ctx["guardrails_c0d93b7e.failed"] = out["failed"]
ctx["guardrails_c0d93b7e.errors"] = out["errors"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["guardrails_c0d93b7e.passed"], "localCheckpoint", None)) and callable(getattr(ctx["guardrails_c0d93b7e.failed"], "localCheckpoint", None)) and callable(getattr(ctx["guardrails_c0d93b7e.errors"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["guardrails_c0d93b7e.passed"]
        _lb_cp_original_1 = ctx["guardrails_c0d93b7e.failed"]
        _lb_cp_original_2 = ctx["guardrails_c0d93b7e.errors"]
        try:
            ctx["guardrails_c0d93b7e.passed"] = ctx["guardrails_c0d93b7e.passed"].localCheckpoint(eager=True)
            ctx["guardrails_c0d93b7e.failed"] = ctx["guardrails_c0d93b7e.failed"].localCheckpoint(eager=True)
            ctx["guardrails_c0d93b7e.errors"] = ctx["guardrails_c0d93b7e.errors"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["guardrails_c0d93b7e.passed"] is not _lb_cp_original_0:
                try:
                    ctx["guardrails_c0d93b7e.passed"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_c0d93b7e.passed"] = _lb_cp_original_0
            if ctx["guardrails_c0d93b7e.failed"] is not _lb_cp_original_1:
                try:
                    ctx["guardrails_c0d93b7e.failed"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_c0d93b7e.failed"] = _lb_cp_original_1
            if ctx["guardrails_c0d93b7e.errors"] is not _lb_cp_original_2:
                try:
                    ctx["guardrails_c0d93b7e.errors"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_c0d93b7e.errors"] = _lb_cp_original_2
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "guardrails_c0d93b7e" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["guardrails_c0d93b7e.passed"])
    display(ctx["guardrails_c0d93b7e.failed"])
    display(ctx["guardrails_c0d93b7e.errors"])
if "guardrails_c0d93b7e" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "guardrails_c0d93b7e", "counts": {"passed": int(ctx["guardrails_c0d93b7e.passed"].count()), "failed": int(ctx["guardrails_c0d93b7e.failed"].count()), "errors": int(ctx["guardrails_c0d93b7e.errors"].count())}}}, raw=True)

In [0]:
"""
id: guardrails_f5b28d4a
template: guardrails
templateVersion: 1.0.0
name: transactions quality checks
parentId: group_2b6f9e0d
position:
  x: 330
  y: 155.5
previewCodeHash: 771044d0b0601f6f
codegenHash: e904b99093c61cc4
updatedAt: 1792940615248
config:
  on_failure:
    mode: warn
  checks:
    - type: not_null
      column: transaction_id
    - type: not_null
      column: customer_id
    - type: not_null
      column: product_id
    - type: not_null
      column: quantity
    - type: not_null
      column: total_amount
    - type: not_null
      column: order_timestamp
    - type: unique
      columns: transaction_id
    - type: value_range
      column: quantity
      min: "1"
    - type: value_range
      column: total_amount
      min: "0"
checkpoint:
  mode: auto
input:
  - node: source_8e3a6c15
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)

CheckResult = Optional[Tuple[str, Any]]

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    return F.col(_quoted_ident(name))

def _require_column(df, column: str, check_type: str) -> None:
    if not column:
        raise ValueError(f"{check_type}: 'column' is required")
    if column.lower() not in {c.lower() for c in df.columns}:
        raise ValueError(
            f"{check_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _split_list(raw: str) -> List[str]:
    if not raw:
        return []
    return [item.strip() for item in raw.split(",") if item.strip()]

def _parse_bound(raw: Any, check_type: str, field: str) -> Optional[float]:
    if raw is None:
        return None
    text = str(raw).strip()
    if text == "":
        return None
    try:
        return int(text)
    except ValueError:
        pass
    try:
        return float(text)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{check_type}: '{field}' must be a number, got {raw!r}"
        ) from exc

def _fmt_bound(bound: Any) -> str:
    return f"{bound:g}" if isinstance(bound, float) else str(bound)

def _check_required_columns(df, check: Dict[str, Any]) -> CheckResult:
    columns = _split_list(check.get("columns", ""))
    if not columns:
        raise ValueError("required_columns: 'columns' is required")
    available = {c.lower() for c in df.columns}
    missing = [c for c in columns if c.lower() not in available]
    if missing:
        return (
            f"missing required column(s) {missing}. "
            f"Available columns: {df.columns}",
            F.lit(True),
        )
    return None

def _check_row_count(df, check: Dict[str, Any]) -> CheckResult:
    min_bound = _parse_bound(check.get("min"), "row_count", "min")
    max_bound = _parse_bound(check.get("max"), "row_count", "max")
    if min_bound is None and max_bound is None:
        raise ValueError("row_count: at least one of 'min' or 'max' is required")
    count = df.count()
    if min_bound is not None and count < min_bound:
        return (
            f"row count {count} is below minimum {_fmt_bound(min_bound)}",
            F.lit(True),
        )
    if max_bound is not None and count > max_bound:
        return (
            f"row count {count} is above maximum {_fmt_bound(max_bound)}",
            F.lit(True),
        )
    return None

def _check_not_null(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "not_null")
    predicate = _col(column).isNull()
    violations = df.filter(predicate).count()
    if violations > 0:
        return (f"column '{column}' has {violations} NULL value(s)", predicate)
    return None

def _check_unique(df, check: Dict[str, Any]) -> CheckResult:
    columns = _split_list(check.get("columns", ""))
    if not columns:
        raise ValueError("unique: 'columns' is required")
    for column in columns:
        _require_column(df, column, "unique")
    window = Window.partitionBy(*[_col(c) for c in columns])
    predicate = F.count(F.lit(1)).over(window) > 1
    violations = (
        df.select(predicate.alias("is_duplicate"))
        .filter(F.col("is_duplicate"))
        .count()
    )
    if violations > 0:
        return (
            f"columns {columns} contain {violations} duplicated row(s) "
            f"(expected unique)",
            predicate,
        )
    return None

def _resolved_name(df, column: str) -> str:
    lowered = column.lower()
    for existing in df.columns:
        if existing.lower() == lowered:
            return existing
    return column

def _matches_any(df, column: str, values: List[str]):
    col = _col(column)
    dtype = df.schema[_resolved_name(df, column)].dataType
    matched = F.lit(False)
    for value in values:
        matched = matched | (col == F.lit(value).cast(dtype))
    return F.coalesce(matched, F.lit(False))

def _check_accepted_values(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "accepted_values")
    values = _split_list(check.get("values", ""))
    if not values:
        raise ValueError("accepted_values: 'values' is required")
    col = _col(column)
    predicate = col.isNotNull() & ~_matches_any(df, column, values)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) outside the "
            f"accepted set {values}",
            predicate,
        )
    return None

def _check_disallowed_values(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "disallowed_values")
    values = _split_list(check.get("values", ""))
    if not values:
        raise ValueError("disallowed_values: 'values' is required")
    col = _col(column)
    predicate = col.isNotNull() & _matches_any(df, column, values)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) in the "
            f"disallowed set {values}",
            predicate,
        )
    return None

def _check_value_range(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "value_range")
    min_bound = _parse_bound(check.get("min"), "value_range", "min")
    max_bound = _parse_bound(check.get("max"), "value_range", "max")
    if min_bound is None and max_bound is None:
        raise ValueError(
            "value_range: at least one of 'min' or 'max' is required"
        )
    col = _col(column)
    out_of_range = None
    if min_bound is not None:
        out_of_range = col < F.lit(min_bound)
    if max_bound is not None:
        above = col > F.lit(max_bound)
        out_of_range = above if out_of_range is None else (out_of_range | above)
    predicate = col.isNotNull() & out_of_range
    violations = df.filter(predicate).count()
    if violations > 0:
        bounds = []
        if min_bound is not None:
            bounds.append(f"min={_fmt_bound(min_bound)}")
        if max_bound is not None:
            bounds.append(f"max={_fmt_bound(max_bound)}")
        return (
            f"column '{column}' has {violations} value(s) outside range "
            f"[{', '.join(bounds)}]",
            predicate,
        )
    return None

def _check_regex_match(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    pattern = check.get("pattern", "")
    _require_column(df, column, "regex_match")
    if not pattern:
        raise ValueError("regex_match: 'pattern' is required")
    as_string = _col(column).cast("string")
    predicate = as_string.isNotNull() & ~as_string.rlike(pattern)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) not matching "
            f"pattern {pattern!r}",
            predicate,
        )
    return None

def _check_expression(df, check: Dict[str, Any]) -> CheckResult:
    expression = check.get("expression", "")
    if not expression:
        raise ValueError("expression: 'expression' is required")
    predicate = ~F.coalesce(F.expr(expression), F.lit(False))
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"{violations} row(s) failed expression {expression!r}",
            predicate,
        )
    return None

CHECK_DISPATCH: Dict[str, Callable] = {
    "required_columns": _check_required_columns,
    "row_count": _check_row_count,
    "not_null": _check_not_null,
    "unique": _check_unique,
    "accepted_values": _check_accepted_values,
    "disallowed_values": _check_disallowed_values,
    "value_range": _check_value_range,
    "regex_match": _check_regex_match,
    "expression": _check_expression,
}

_FAILED_COL = "__gr_failed__"

_ERROR_SCHEMA = StructType([
    StructField("check_index", IntegerType(), True),
    StructField("check_type", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("message", StringType(), True),
])

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        raise ValueError(
            "Guardrails: no input dataset. Connect an upstream node to "
            "the 'data' input port before running this check."
        )
    checks: List[Dict[str, Any]] = config.get("checks", []) or []
    raw_on_failure = config.get("on_failure", "warn")
    if isinstance(raw_on_failure, dict):
        mode = raw_on_failure.get("mode", "warn")
    else:
        mode = raw_on_failure or "warn"
    if mode not in ("warn", "block", "stop"):
        mode = "warn"

    error_rows: List[tuple] = []
    marked = df
    failed_cols: List[str] = []
    taken_lower = {c.lower() for c in df.columns}
    for index, check in enumerate(checks):
        if not isinstance(check, dict):
            check_type = ""
            result = (
                f"expected an object, got {type(check).__name__}",
                F.lit(True),
            )
        elif check.get("enabled", True) is False:
            continue
        else:
            check_type = check.get("type", "")
            fn = CHECK_DISPATCH.get(check_type)
            if fn is None:
                result = (
                    f"unsupported check type {check_type!r}. "
                    f"Choose one of: {sorted(CHECK_DISPATCH.keys())}",
                    F.lit(True),
                )
            else:
                try:
                    result = fn(df, check)
                except ValueError as exc:
                    result = (str(exc), F.lit(True))
                except Exception as exc:
                    result = (f"{type(exc).__name__}: {exc}", F.lit(True))
        if result is not None:
            error, predicate = result
            message = f"checks[{index}] ({check_type}): {error}"
            severity = "failure" if mode == "stop" else "warning"
            error_rows.append((index, check_type, severity, message))
            if mode == "stop":
                print(f"[guardrails] FAILURE: {message}")
            else:
                print(f"[guardrails] WARNING: {message}")
            failed_col = f"__gr_failed_{index}__"
            while failed_col.lower() in taken_lower:
                failed_col += "_"
            taken_lower.add(failed_col.lower())
            marked = marked.withColumn(
                failed_col, F.coalesce(predicate, F.lit(False))
            )
            failed_cols.append(failed_col)

    marker_col = _FAILED_COL
    while marker_col.lower() in taken_lower:
        marker_col += "_"
    taken_lower.add(marker_col.lower())
    failed_marker = F.lit(False)
    for failed_col in failed_cols:
        failed_marker = failed_marker | F.col(failed_col)
    marked = marked.withColumn(marker_col, failed_marker)
    drop_cols = [marker_col, *failed_cols]
    passed_df = marked.filter(~F.col(marker_col)).drop(*drop_cols)
    failed_df = marked.filter(F.col(marker_col)).drop(*drop_cols)
    errors_df = spark.createDataFrame(error_rows, _ERROR_SCHEMA)

    meta_state = config.get("meta_state")
    if not isinstance(meta_state, dict):
        meta_state = {}
    is_focused_preview = meta_state.get("is_focused_preview") is True

    if error_rows and mode == "block":
        passed_df = passed_df.limit(0)
    elif error_rows and mode == "stop":
        if not is_focused_preview:
            detail = "\n".join(f"[guardrails] {r[3]}" for r in error_rows)
            raise ValueError(
                f"Guardrails failed the workflow: {len(error_rows)} check(s) "
                f"failed and the failure mode is 'stop' (Fail the workflow)."
                f"\n{detail}"
            )

    return {"passed": passed_df, "failed": failed_df, "errors": errors_df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "on_failure": {
        "mode": "warn"
    },
    "checks": [
        {
            "type": "not_null",
            "column": "transaction_id"
        },
        {
            "type": "not_null",
            "column": "customer_id"
        },
        {
            "type": "not_null",
            "column": "product_id"
        },
        {
            "type": "not_null",
            "column": "quantity"
        },
        {
            "type": "not_null",
            "column": "total_amount"
        },
        {
            "type": "not_null",
            "column": "order_timestamp"
        },
        {
            "type": "unique",
            "columns": "transaction_id"
        },
        {
            "type": "value_range",
            "column": "quantity",
            "min": "1"
        },
        {
            "type": "value_range",
            "column": "total_amount",
            "min": "0"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_8e3a6c15.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["guardrails_f5b28d4a.passed"] = out["passed"]
ctx["guardrails_f5b28d4a.failed"] = out["failed"]
ctx["guardrails_f5b28d4a.errors"] = out["errors"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["guardrails_f5b28d4a.passed"], "localCheckpoint", None)) and callable(getattr(ctx["guardrails_f5b28d4a.failed"], "localCheckpoint", None)) and callable(getattr(ctx["guardrails_f5b28d4a.errors"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["guardrails_f5b28d4a.passed"]
        _lb_cp_original_1 = ctx["guardrails_f5b28d4a.failed"]
        _lb_cp_original_2 = ctx["guardrails_f5b28d4a.errors"]
        try:
            ctx["guardrails_f5b28d4a.passed"] = ctx["guardrails_f5b28d4a.passed"].localCheckpoint(eager=True)
            ctx["guardrails_f5b28d4a.failed"] = ctx["guardrails_f5b28d4a.failed"].localCheckpoint(eager=True)
            ctx["guardrails_f5b28d4a.errors"] = ctx["guardrails_f5b28d4a.errors"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["guardrails_f5b28d4a.passed"] is not _lb_cp_original_0:
                try:
                    ctx["guardrails_f5b28d4a.passed"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_f5b28d4a.passed"] = _lb_cp_original_0
            if ctx["guardrails_f5b28d4a.failed"] is not _lb_cp_original_1:
                try:
                    ctx["guardrails_f5b28d4a.failed"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_f5b28d4a.failed"] = _lb_cp_original_1
            if ctx["guardrails_f5b28d4a.errors"] is not _lb_cp_original_2:
                try:
                    ctx["guardrails_f5b28d4a.errors"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_f5b28d4a.errors"] = _lb_cp_original_2
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "guardrails_f5b28d4a" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["guardrails_f5b28d4a.passed"])
    display(ctx["guardrails_f5b28d4a.failed"])
    display(ctx["guardrails_f5b28d4a.errors"])
if "guardrails_f5b28d4a" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "guardrails_f5b28d4a", "counts": {"passed": int(ctx["guardrails_f5b28d4a.passed"].count()), "failed": int(ctx["guardrails_f5b28d4a.failed"].count()), "errors": int(ctx["guardrails_f5b28d4a.errors"].count())}}}, raw=True)

In [0]:
"""
id: guardrails_e3f05a6d
template: guardrails
templateVersion: 1.0.0
name: customer_usa quality checks
parentId: group_5d2e8a1f
position:
  x: 330
  y: 155.5
description:
  text: Check customer data for missing required fields, duplicate customer IDs, and valid email format.
  hash: a206d4fb
previewCodeHash: 061d1e4c2585f046
codegenHash: e904b99093c61cc4
updatedAt: 1792940647159
config:
  on_failure:
    mode: warn
  checks:
    - type: not_null
      column: customer_id
    - type: not_null
      column: first_name
    - type: not_null
      column: last_name
    - type: not_null
      column: email
    - type: not_null
      column: phone
    - type: unique
      columns: customer_id
    - type: regex_match
      column: email
      pattern: ^[^@]+@[^@]+\.[^@]+$
checkpoint:
  mode: auto
input:
  - node: source_a4c71b09
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)

CheckResult = Optional[Tuple[str, Any]]

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    return F.col(_quoted_ident(name))

def _require_column(df, column: str, check_type: str) -> None:
    if not column:
        raise ValueError(f"{check_type}: 'column' is required")
    if column.lower() not in {c.lower() for c in df.columns}:
        raise ValueError(
            f"{check_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _split_list(raw: str) -> List[str]:
    if not raw:
        return []
    return [item.strip() for item in raw.split(",") if item.strip()]

def _parse_bound(raw: Any, check_type: str, field: str) -> Optional[float]:
    if raw is None:
        return None
    text = str(raw).strip()
    if text == "":
        return None
    try:
        return int(text)
    except ValueError:
        pass
    try:
        return float(text)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{check_type}: '{field}' must be a number, got {raw!r}"
        ) from exc

def _fmt_bound(bound: Any) -> str:
    return f"{bound:g}" if isinstance(bound, float) else str(bound)

def _check_required_columns(df, check: Dict[str, Any]) -> CheckResult:
    columns = _split_list(check.get("columns", ""))
    if not columns:
        raise ValueError("required_columns: 'columns' is required")
    available = {c.lower() for c in df.columns}
    missing = [c for c in columns if c.lower() not in available]
    if missing:
        return (
            f"missing required column(s) {missing}. "
            f"Available columns: {df.columns}",
            F.lit(True),
        )
    return None

def _check_row_count(df, check: Dict[str, Any]) -> CheckResult:
    min_bound = _parse_bound(check.get("min"), "row_count", "min")
    max_bound = _parse_bound(check.get("max"), "row_count", "max")
    if min_bound is None and max_bound is None:
        raise ValueError("row_count: at least one of 'min' or 'max' is required")
    count = df.count()
    if min_bound is not None and count < min_bound:
        return (
            f"row count {count} is below minimum {_fmt_bound(min_bound)}",
            F.lit(True),
        )
    if max_bound is not None and count > max_bound:
        return (
            f"row count {count} is above maximum {_fmt_bound(max_bound)}",
            F.lit(True),
        )
    return None

def _check_not_null(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "not_null")
    predicate = _col(column).isNull()
    violations = df.filter(predicate).count()
    if violations > 0:
        return (f"column '{column}' has {violations} NULL value(s)", predicate)
    return None

def _check_unique(df, check: Dict[str, Any]) -> CheckResult:
    columns = _split_list(check.get("columns", ""))
    if not columns:
        raise ValueError("unique: 'columns' is required")
    for column in columns:
        _require_column(df, column, "unique")
    window = Window.partitionBy(*[_col(c) for c in columns])
    predicate = F.count(F.lit(1)).over(window) > 1
    violations = (
        df.select(predicate.alias("is_duplicate"))
        .filter(F.col("is_duplicate"))
        .count()
    )
    if violations > 0:
        return (
            f"columns {columns} contain {violations} duplicated row(s) "
            f"(expected unique)",
            predicate,
        )
    return None

def _resolved_name(df, column: str) -> str:
    lowered = column.lower()
    for existing in df.columns:
        if existing.lower() == lowered:
            return existing
    return column

def _matches_any(df, column: str, values: List[str]):
    col = _col(column)
    dtype = df.schema[_resolved_name(df, column)].dataType
    matched = F.lit(False)
    for value in values:
        matched = matched | (col == F.lit(value).cast(dtype))
    return F.coalesce(matched, F.lit(False))

def _check_accepted_values(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "accepted_values")
    values = _split_list(check.get("values", ""))
    if not values:
        raise ValueError("accepted_values: 'values' is required")
    col = _col(column)
    predicate = col.isNotNull() & ~_matches_any(df, column, values)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) outside the "
            f"accepted set {values}",
            predicate,
        )
    return None

def _check_disallowed_values(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "disallowed_values")
    values = _split_list(check.get("values", ""))
    if not values:
        raise ValueError("disallowed_values: 'values' is required")
    col = _col(column)
    predicate = col.isNotNull() & _matches_any(df, column, values)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) in the "
            f"disallowed set {values}",
            predicate,
        )
    return None

def _check_value_range(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "value_range")
    min_bound = _parse_bound(check.get("min"), "value_range", "min")
    max_bound = _parse_bound(check.get("max"), "value_range", "max")
    if min_bound is None and max_bound is None:
        raise ValueError(
            "value_range: at least one of 'min' or 'max' is required"
        )
    col = _col(column)
    out_of_range = None
    if min_bound is not None:
        out_of_range = col < F.lit(min_bound)
    if max_bound is not None:
        above = col > F.lit(max_bound)
        out_of_range = above if out_of_range is None else (out_of_range | above)
    predicate = col.isNotNull() & out_of_range
    violations = df.filter(predicate).count()
    if violations > 0:
        bounds = []
        if min_bound is not None:
            bounds.append(f"min={_fmt_bound(min_bound)}")
        if max_bound is not None:
            bounds.append(f"max={_fmt_bound(max_bound)}")
        return (
            f"column '{column}' has {violations} value(s) outside range "
            f"[{', '.join(bounds)}]",
            predicate,
        )
    return None

def _check_regex_match(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    pattern = check.get("pattern", "")
    _require_column(df, column, "regex_match")
    if not pattern:
        raise ValueError("regex_match: 'pattern' is required")
    as_string = _col(column).cast("string")
    predicate = as_string.isNotNull() & ~as_string.rlike(pattern)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) not matching "
            f"pattern {pattern!r}",
            predicate,
        )
    return None

def _check_expression(df, check: Dict[str, Any]) -> CheckResult:
    expression = check.get("expression", "")
    if not expression:
        raise ValueError("expression: 'expression' is required")
    predicate = ~F.coalesce(F.expr(expression), F.lit(False))
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"{violations} row(s) failed expression {expression!r}",
            predicate,
        )
    return None

CHECK_DISPATCH: Dict[str, Callable] = {
    "required_columns": _check_required_columns,
    "row_count": _check_row_count,
    "not_null": _check_not_null,
    "unique": _check_unique,
    "accepted_values": _check_accepted_values,
    "disallowed_values": _check_disallowed_values,
    "value_range": _check_value_range,
    "regex_match": _check_regex_match,
    "expression": _check_expression,
}

_FAILED_COL = "__gr_failed__"

_ERROR_SCHEMA = StructType([
    StructField("check_index", IntegerType(), True),
    StructField("check_type", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("message", StringType(), True),
])

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        raise ValueError(
            "Guardrails: no input dataset. Connect an upstream node to "
            "the 'data' input port before running this check."
        )
    checks: List[Dict[str, Any]] = config.get("checks", []) or []
    raw_on_failure = config.get("on_failure", "warn")
    if isinstance(raw_on_failure, dict):
        mode = raw_on_failure.get("mode", "warn")
    else:
        mode = raw_on_failure or "warn"
    if mode not in ("warn", "block", "stop"):
        mode = "warn"

    error_rows: List[tuple] = []
    marked = df
    failed_cols: List[str] = []
    taken_lower = {c.lower() for c in df.columns}
    for index, check in enumerate(checks):
        if not isinstance(check, dict):
            check_type = ""
            result = (
                f"expected an object, got {type(check).__name__}",
                F.lit(True),
            )
        elif check.get("enabled", True) is False:
            continue
        else:
            check_type = check.get("type", "")
            fn = CHECK_DISPATCH.get(check_type)
            if fn is None:
                result = (
                    f"unsupported check type {check_type!r}. "
                    f"Choose one of: {sorted(CHECK_DISPATCH.keys())}",
                    F.lit(True),
                )
            else:
                try:
                    result = fn(df, check)
                except ValueError as exc:
                    result = (str(exc), F.lit(True))
                except Exception as exc:
                    result = (f"{type(exc).__name__}: {exc}", F.lit(True))
        if result is not None:
            error, predicate = result
            message = f"checks[{index}] ({check_type}): {error}"
            severity = "failure" if mode == "stop" else "warning"
            error_rows.append((index, check_type, severity, message))
            if mode == "stop":
                print(f"[guardrails] FAILURE: {message}")
            else:
                print(f"[guardrails] WARNING: {message}")
            failed_col = f"__gr_failed_{index}__"
            while failed_col.lower() in taken_lower:
                failed_col += "_"
            taken_lower.add(failed_col.lower())
            marked = marked.withColumn(
                failed_col, F.coalesce(predicate, F.lit(False))
            )
            failed_cols.append(failed_col)

    marker_col = _FAILED_COL
    while marker_col.lower() in taken_lower:
        marker_col += "_"
    taken_lower.add(marker_col.lower())
    failed_marker = F.lit(False)
    for failed_col in failed_cols:
        failed_marker = failed_marker | F.col(failed_col)
    marked = marked.withColumn(marker_col, failed_marker)
    drop_cols = [marker_col, *failed_cols]
    passed_df = marked.filter(~F.col(marker_col)).drop(*drop_cols)
    failed_df = marked.filter(F.col(marker_col)).drop(*drop_cols)
    errors_df = spark.createDataFrame(error_rows, _ERROR_SCHEMA)

    meta_state = config.get("meta_state")
    if not isinstance(meta_state, dict):
        meta_state = {}
    is_focused_preview = meta_state.get("is_focused_preview") is True

    if error_rows and mode == "block":
        passed_df = passed_df.limit(0)
    elif error_rows and mode == "stop":
        if not is_focused_preview:
            detail = "\n".join(f"[guardrails] {r[3]}" for r in error_rows)
            raise ValueError(
                f"Guardrails failed the workflow: {len(error_rows)} check(s) "
                f"failed and the failure mode is 'stop' (Fail the workflow)."
                f"\n{detail}"
            )

    return {"passed": passed_df, "failed": failed_df, "errors": errors_df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "on_failure": {
        "mode": "warn"
    },
    "checks": [
        {
            "type": "not_null",
            "column": "customer_id"
        },
        {
            "type": "not_null",
            "column": "first_name"
        },
        {
            "type": "not_null",
            "column": "last_name"
        },
        {
            "type": "not_null",
            "column": "email"
        },
        {
            "type": "not_null",
            "column": "phone"
        },
        {
            "type": "unique",
            "columns": "customer_id"
        },
        {
            "type": "regex_match",
            "column": "email",
            "pattern": "^[^@]+@[^@]+\\.[^@]+$"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_a4c71b09.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["guardrails_e3f05a6d.passed"] = out["passed"]
ctx["guardrails_e3f05a6d.failed"] = out["failed"]
ctx["guardrails_e3f05a6d.errors"] = out["errors"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["guardrails_e3f05a6d.passed"], "localCheckpoint", None)) and callable(getattr(ctx["guardrails_e3f05a6d.failed"], "localCheckpoint", None)) and callable(getattr(ctx["guardrails_e3f05a6d.errors"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["guardrails_e3f05a6d.passed"]
        _lb_cp_original_1 = ctx["guardrails_e3f05a6d.failed"]
        _lb_cp_original_2 = ctx["guardrails_e3f05a6d.errors"]
        try:
            ctx["guardrails_e3f05a6d.passed"] = ctx["guardrails_e3f05a6d.passed"].localCheckpoint(eager=True)
            ctx["guardrails_e3f05a6d.failed"] = ctx["guardrails_e3f05a6d.failed"].localCheckpoint(eager=True)
            ctx["guardrails_e3f05a6d.errors"] = ctx["guardrails_e3f05a6d.errors"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["guardrails_e3f05a6d.passed"] is not _lb_cp_original_0:
                try:
                    ctx["guardrails_e3f05a6d.passed"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_e3f05a6d.passed"] = _lb_cp_original_0
            if ctx["guardrails_e3f05a6d.failed"] is not _lb_cp_original_1:
                try:
                    ctx["guardrails_e3f05a6d.failed"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_e3f05a6d.failed"] = _lb_cp_original_1
            if ctx["guardrails_e3f05a6d.errors"] is not _lb_cp_original_2:
                try:
                    ctx["guardrails_e3f05a6d.errors"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_e3f05a6d.errors"] = _lb_cp_original_2
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "guardrails_e3f05a6d" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["guardrails_e3f05a6d.passed"])
    display(ctx["guardrails_e3f05a6d.failed"])
    display(ctx["guardrails_e3f05a6d.errors"])
if "guardrails_e3f05a6d" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "guardrails_e3f05a6d", "counts": {"passed": int(ctx["guardrails_e3f05a6d.passed"].count()), "failed": int(ctx["guardrails_e3f05a6d.failed"].count()), "errors": int(ctx["guardrails_e3f05a6d.errors"].count())}}}, raw=True)

In [0]:
"""
id: guardrails_4a7c0e8b
template: guardrails
templateVersion: 1.0.0
name: customer_uk quality checks
parentId: group_8c3f1d6a
position:
  x: 330
  y: 155.5
codegenHash: e904b99093c61cc4
updatedAt: 1792940615068
config:
  on_failure:
    mode: warn
  checks:
    - type: not_null
      column: customer_id
    - type: not_null
      column: first_name
    - type: not_null
      column: last_name
    - type: not_null
      column: email
    - type: not_null
      column: phone
    - type: unique
      columns: customer_id
    - type: regex_match
      column: email
      pattern: ^[^@]+@[^@]+\.[^@]+$
checkpoint:
  mode: auto
input:
  - node: source_d6e29a40
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)

CheckResult = Optional[Tuple[str, Any]]

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    return F.col(_quoted_ident(name))

def _require_column(df, column: str, check_type: str) -> None:
    if not column:
        raise ValueError(f"{check_type}: 'column' is required")
    if column.lower() not in {c.lower() for c in df.columns}:
        raise ValueError(
            f"{check_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _split_list(raw: str) -> List[str]:
    if not raw:
        return []
    return [item.strip() for item in raw.split(",") if item.strip()]

def _parse_bound(raw: Any, check_type: str, field: str) -> Optional[float]:
    if raw is None:
        return None
    text = str(raw).strip()
    if text == "":
        return None
    try:
        return int(text)
    except ValueError:
        pass
    try:
        return float(text)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"{check_type}: '{field}' must be a number, got {raw!r}"
        ) from exc

def _fmt_bound(bound: Any) -> str:
    return f"{bound:g}" if isinstance(bound, float) else str(bound)

def _check_required_columns(df, check: Dict[str, Any]) -> CheckResult:
    columns = _split_list(check.get("columns", ""))
    if not columns:
        raise ValueError("required_columns: 'columns' is required")
    available = {c.lower() for c in df.columns}
    missing = [c for c in columns if c.lower() not in available]
    if missing:
        return (
            f"missing required column(s) {missing}. "
            f"Available columns: {df.columns}",
            F.lit(True),
        )
    return None

def _check_row_count(df, check: Dict[str, Any]) -> CheckResult:
    min_bound = _parse_bound(check.get("min"), "row_count", "min")
    max_bound = _parse_bound(check.get("max"), "row_count", "max")
    if min_bound is None and max_bound is None:
        raise ValueError("row_count: at least one of 'min' or 'max' is required")
    count = df.count()
    if min_bound is not None and count < min_bound:
        return (
            f"row count {count} is below minimum {_fmt_bound(min_bound)}",
            F.lit(True),
        )
    if max_bound is not None and count > max_bound:
        return (
            f"row count {count} is above maximum {_fmt_bound(max_bound)}",
            F.lit(True),
        )
    return None

def _check_not_null(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "not_null")
    predicate = _col(column).isNull()
    violations = df.filter(predicate).count()
    if violations > 0:
        return (f"column '{column}' has {violations} NULL value(s)", predicate)
    return None

def _check_unique(df, check: Dict[str, Any]) -> CheckResult:
    columns = _split_list(check.get("columns", ""))
    if not columns:
        raise ValueError("unique: 'columns' is required")
    for column in columns:
        _require_column(df, column, "unique")
    window = Window.partitionBy(*[_col(c) for c in columns])
    predicate = F.count(F.lit(1)).over(window) > 1
    violations = (
        df.select(predicate.alias("is_duplicate"))
        .filter(F.col("is_duplicate"))
        .count()
    )
    if violations > 0:
        return (
            f"columns {columns} contain {violations} duplicated row(s) "
            f"(expected unique)",
            predicate,
        )
    return None

def _resolved_name(df, column: str) -> str:
    lowered = column.lower()
    for existing in df.columns:
        if existing.lower() == lowered:
            return existing
    return column

def _matches_any(df, column: str, values: List[str]):
    col = _col(column)
    dtype = df.schema[_resolved_name(df, column)].dataType
    matched = F.lit(False)
    for value in values:
        matched = matched | (col == F.lit(value).cast(dtype))
    return F.coalesce(matched, F.lit(False))

def _check_accepted_values(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "accepted_values")
    values = _split_list(check.get("values", ""))
    if not values:
        raise ValueError("accepted_values: 'values' is required")
    col = _col(column)
    predicate = col.isNotNull() & ~_matches_any(df, column, values)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) outside the "
            f"accepted set {values}",
            predicate,
        )
    return None

def _check_disallowed_values(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "disallowed_values")
    values = _split_list(check.get("values", ""))
    if not values:
        raise ValueError("disallowed_values: 'values' is required")
    col = _col(column)
    predicate = col.isNotNull() & _matches_any(df, column, values)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) in the "
            f"disallowed set {values}",
            predicate,
        )
    return None

def _check_value_range(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    _require_column(df, column, "value_range")
    min_bound = _parse_bound(check.get("min"), "value_range", "min")
    max_bound = _parse_bound(check.get("max"), "value_range", "max")
    if min_bound is None and max_bound is None:
        raise ValueError(
            "value_range: at least one of 'min' or 'max' is required"
        )
    col = _col(column)
    out_of_range = None
    if min_bound is not None:
        out_of_range = col < F.lit(min_bound)
    if max_bound is not None:
        above = col > F.lit(max_bound)
        out_of_range = above if out_of_range is None else (out_of_range | above)
    predicate = col.isNotNull() & out_of_range
    violations = df.filter(predicate).count()
    if violations > 0:
        bounds = []
        if min_bound is not None:
            bounds.append(f"min={_fmt_bound(min_bound)}")
        if max_bound is not None:
            bounds.append(f"max={_fmt_bound(max_bound)}")
        return (
            f"column '{column}' has {violations} value(s) outside range "
            f"[{', '.join(bounds)}]",
            predicate,
        )
    return None

def _check_regex_match(df, check: Dict[str, Any]) -> CheckResult:
    column = check.get("column", "")
    pattern = check.get("pattern", "")
    _require_column(df, column, "regex_match")
    if not pattern:
        raise ValueError("regex_match: 'pattern' is required")
    as_string = _col(column).cast("string")
    predicate = as_string.isNotNull() & ~as_string.rlike(pattern)
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"column '{column}' has {violations} value(s) not matching "
            f"pattern {pattern!r}",
            predicate,
        )
    return None

def _check_expression(df, check: Dict[str, Any]) -> CheckResult:
    expression = check.get("expression", "")
    if not expression:
        raise ValueError("expression: 'expression' is required")
    predicate = ~F.coalesce(F.expr(expression), F.lit(False))
    violations = df.filter(predicate).count()
    if violations > 0:
        return (
            f"{violations} row(s) failed expression {expression!r}",
            predicate,
        )
    return None

CHECK_DISPATCH: Dict[str, Callable] = {
    "required_columns": _check_required_columns,
    "row_count": _check_row_count,
    "not_null": _check_not_null,
    "unique": _check_unique,
    "accepted_values": _check_accepted_values,
    "disallowed_values": _check_disallowed_values,
    "value_range": _check_value_range,
    "regex_match": _check_regex_match,
    "expression": _check_expression,
}

_FAILED_COL = "__gr_failed__"

_ERROR_SCHEMA = StructType([
    StructField("check_index", IntegerType(), True),
    StructField("check_type", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("message", StringType(), True),
])

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    if df is None:
        raise ValueError(
            "Guardrails: no input dataset. Connect an upstream node to "
            "the 'data' input port before running this check."
        )
    checks: List[Dict[str, Any]] = config.get("checks", []) or []
    raw_on_failure = config.get("on_failure", "warn")
    if isinstance(raw_on_failure, dict):
        mode = raw_on_failure.get("mode", "warn")
    else:
        mode = raw_on_failure or "warn"
    if mode not in ("warn", "block", "stop"):
        mode = "warn"

    error_rows: List[tuple] = []
    marked = df
    failed_cols: List[str] = []
    taken_lower = {c.lower() for c in df.columns}
    for index, check in enumerate(checks):
        if not isinstance(check, dict):
            check_type = ""
            result = (
                f"expected an object, got {type(check).__name__}",
                F.lit(True),
            )
        elif check.get("enabled", True) is False:
            continue
        else:
            check_type = check.get("type", "")
            fn = CHECK_DISPATCH.get(check_type)
            if fn is None:
                result = (
                    f"unsupported check type {check_type!r}. "
                    f"Choose one of: {sorted(CHECK_DISPATCH.keys())}",
                    F.lit(True),
                )
            else:
                try:
                    result = fn(df, check)
                except ValueError as exc:
                    result = (str(exc), F.lit(True))
                except Exception as exc:
                    result = (f"{type(exc).__name__}: {exc}", F.lit(True))
        if result is not None:
            error, predicate = result
            message = f"checks[{index}] ({check_type}): {error}"
            severity = "failure" if mode == "stop" else "warning"
            error_rows.append((index, check_type, severity, message))
            if mode == "stop":
                print(f"[guardrails] FAILURE: {message}")
            else:
                print(f"[guardrails] WARNING: {message}")
            failed_col = f"__gr_failed_{index}__"
            while failed_col.lower() in taken_lower:
                failed_col += "_"
            taken_lower.add(failed_col.lower())
            marked = marked.withColumn(
                failed_col, F.coalesce(predicate, F.lit(False))
            )
            failed_cols.append(failed_col)

    marker_col = _FAILED_COL
    while marker_col.lower() in taken_lower:
        marker_col += "_"
    taken_lower.add(marker_col.lower())
    failed_marker = F.lit(False)
    for failed_col in failed_cols:
        failed_marker = failed_marker | F.col(failed_col)
    marked = marked.withColumn(marker_col, failed_marker)
    drop_cols = [marker_col, *failed_cols]
    passed_df = marked.filter(~F.col(marker_col)).drop(*drop_cols)
    failed_df = marked.filter(F.col(marker_col)).drop(*drop_cols)
    errors_df = spark.createDataFrame(error_rows, _ERROR_SCHEMA)

    meta_state = config.get("meta_state")
    if not isinstance(meta_state, dict):
        meta_state = {}
    is_focused_preview = meta_state.get("is_focused_preview") is True

    if error_rows and mode == "block":
        passed_df = passed_df.limit(0)
    elif error_rows and mode == "stop":
        if not is_focused_preview:
            detail = "\n".join(f"[guardrails] {r[3]}" for r in error_rows)
            raise ValueError(
                f"Guardrails failed the workflow: {len(error_rows)} check(s) "
                f"failed and the failure mode is 'stop' (Fail the workflow)."
                f"\n{detail}"
            )

    return {"passed": passed_df, "failed": failed_df, "errors": errors_df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "on_failure": {
        "mode": "warn"
    },
    "checks": [
        {
            "type": "not_null",
            "column": "customer_id"
        },
        {
            "type": "not_null",
            "column": "first_name"
        },
        {
            "type": "not_null",
            "column": "last_name"
        },
        {
            "type": "not_null",
            "column": "email"
        },
        {
            "type": "not_null",
            "column": "phone"
        },
        {
            "type": "unique",
            "columns": "customer_id"
        },
        {
            "type": "regex_match",
            "column": "email",
            "pattern": "^[^@]+@[^@]+\\.[^@]+$"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["source_d6e29a40.data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["guardrails_4a7c0e8b.passed"] = out["passed"]
ctx["guardrails_4a7c0e8b.failed"] = out["failed"]
ctx["guardrails_4a7c0e8b.errors"] = out["errors"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["guardrails_4a7c0e8b.passed"], "localCheckpoint", None)) and callable(getattr(ctx["guardrails_4a7c0e8b.failed"], "localCheckpoint", None)) and callable(getattr(ctx["guardrails_4a7c0e8b.errors"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["guardrails_4a7c0e8b.passed"]
        _lb_cp_original_1 = ctx["guardrails_4a7c0e8b.failed"]
        _lb_cp_original_2 = ctx["guardrails_4a7c0e8b.errors"]
        try:
            ctx["guardrails_4a7c0e8b.passed"] = ctx["guardrails_4a7c0e8b.passed"].localCheckpoint(eager=True)
            ctx["guardrails_4a7c0e8b.failed"] = ctx["guardrails_4a7c0e8b.failed"].localCheckpoint(eager=True)
            ctx["guardrails_4a7c0e8b.errors"] = ctx["guardrails_4a7c0e8b.errors"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["guardrails_4a7c0e8b.passed"] is not _lb_cp_original_0:
                try:
                    ctx["guardrails_4a7c0e8b.passed"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_4a7c0e8b.passed"] = _lb_cp_original_0
            if ctx["guardrails_4a7c0e8b.failed"] is not _lb_cp_original_1:
                try:
                    ctx["guardrails_4a7c0e8b.failed"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_4a7c0e8b.failed"] = _lb_cp_original_1
            if ctx["guardrails_4a7c0e8b.errors"] is not _lb_cp_original_2:
                try:
                    ctx["guardrails_4a7c0e8b.errors"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["guardrails_4a7c0e8b.errors"] = _lb_cp_original_2
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "guardrails_4a7c0e8b" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["guardrails_4a7c0e8b.passed"])
    display(ctx["guardrails_4a7c0e8b.failed"])
    display(ctx["guardrails_4a7c0e8b.errors"])
if "guardrails_4a7c0e8b" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "guardrails_4a7c0e8b", "counts": {"passed": int(ctx["guardrails_4a7c0e8b.passed"].count()), "failed": int(ctx["guardrails_4a7c0e8b.failed"].count()), "errors": int(ctx["guardrails_4a7c0e8b.errors"].count())}}}, raw=True)

In [0]:
"""
id: output_2e6b9d47
template: output
templateVersion: 4.0.0
name: customer_ind clean
parentId: group_0
position:
  x: 630
  y: 78
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792940614985
config:
  output_type: table
  catalog: workspace
  schema: visual_data_prep
  table_name: customer_ind_clean
  write_mode: append
checkpoint:
  mode: auto
input:
  - node: guardrails_7b4d2e91
    input_port: data
    output_port: passed
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "visual_data_prep",
    "table_name": "customer_ind_clean",
    "write_mode": "append",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["guardrails_7b4d2e91.passed"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM workspace.visual_data_prep.customer_ind_clean").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_2e6b9d47.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_2e6b9d47" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_2e6b9d47.result"])
if "output_2e6b9d47" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_2e6b9d47", "counts": {"result": int(ctx["output_2e6b9d47.result"].count())}}}, raw=True)

In [0]:
"""
id: prepare_c81a5f03
template: prepare
templateVersion: 2.0.0
name: customer_ind failure reasons
parentId: group_0
position:
  x: 630
  y: 233
codegenHash: 7b3643f91c41b910
updatedAt: 1792940614985
config:
  actions:
    - type: formula
      id: a_f3I40slN
      enabled: true
      target: quality_check_reason
      expression: |
        CONCAT_WS('; ',
          CASE WHEN customer_id IS NULL THEN 'customer_id is missing' END,
          CASE WHEN first_name IS NULL THEN 'first_name is missing' END,
          CASE WHEN last_name IS NULL THEN 'last_name is missing' END,
          CASE WHEN email IS NULL THEN 'email is missing' END,
          CASE WHEN phone IS NULL THEN 'phone is missing' END,
          CASE WHEN NOT email RLIKE '^[^@]+@[^@]+\\.[^@]+$' THEN 'email is invalid' END
        )
checkpoint:
  mode: auto
input:
  - node: guardrails_7b4d2e91
    input_port: data
    output_port: failed
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
from uuid import uuid4
import pyspark.sql.functions as F
import pyspark.sql.types as T

def _lb_run_bound_sql(df, spark, make_query):
    view_name = "_lb_input_" + uuid4().hex
    df.createOrReplaceTempView(view_name)
    try:
        query = make_query(view_name)
        args = {}
        if "dbutils" in globals() and hasattr(dbutils, "widgets") and hasattr(dbutils.widgets, "getAll"):
            args = dict(dbutils.widgets.getAll())
        evaluated = spark.sql(query, args=args)
    except Exception:
        spark.catalog.dropTempView(view_name)
        raise
    globals().setdefault("_lb_views", set()).add(view_name)
    return evaluated

def _lb_apply_bound_expressions(df, spark, assignments):
    occupied = set(df.columns) | set(assignments)
    temporary = {}
    for target in assignments:
        name = "_lb_expression_" + uuid4().hex
        while name in occupied:
            name = "_lb_expression_" + uuid4().hex
        occupied.add(name)
        temporary[target] = name
    expressions = [
        "(" + expression + "\n) AS " + chr(96) + temporary[target] + chr(96)
        for target, expression in assignments.items()
    ]
    evaluated = _lb_run_bound_sql(
        df, spark, lambda view: "SELECT *, " + ", ".join(expressions) + " FROM " + view
    )
    return evaluated.withColumns({target: F.col(name) for target, name in temporary.items()}).drop(*temporary.values())

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Coerce a user-provided value to the target column's type.

    UI always feeds text per the operator spec. try_cast, not cast, so a
    value that does not fit the column's type yields NULL instead of
    aborting the query under ANSI mode -- the empty default fill value
    against a numeric column then leaves that column's nulls in place, as
    the fill-null UI promises, rather than raising CAST_INVALID_INPUT.
    """
    if value is None:
        return F.lit(None).cast(target_dtype)
    escaped = str(value).replace(chr(92), chr(92) * 2).replace("'", chr(92) + "'")
    return F.expr("try_cast('" + escaped + "' as " + target_dtype + ")")

def _apply_formula(df, action: Dict[str, Any], spark):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, {target: expression})
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any], spark):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = rendered if action.get("__lb_bind_expression") else F.expr(rendered)
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, assignments)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action, spark)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action, spark)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "id": "a_f3I40slN",
            "enabled": True,
            "target": "quality_check_reason",
            "expression": "CONCAT_WS('; ',\n  CASE WHEN customer_id IS NULL THEN 'customer_id is missing' END,\n  CASE WHEN first_name IS NULL THEN 'first_name is missing' END,\n  CASE WHEN last_name IS NULL THEN 'last_name is missing' END,\n  CASE WHEN email IS NULL THEN 'email is missing' END,\n  CASE WHEN phone IS NULL THEN 'phone is missing' END,\n  CASE WHEN NOT email RLIKE '^[^@]+@[^@]+\\\\.[^@]+$' THEN 'email is invalid' END\n)\n"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["guardrails_7b4d2e91.failed"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_c81a5f03.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_c81a5f03" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_c81a5f03.prepared_data"])
if "prepare_c81a5f03" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_c81a5f03", "counts": {"prepared_data": int(ctx["prepare_c81a5f03.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: output_3b8e5c90
template: output
templateVersion: 4.0.0
name: products clean
parentId: group_e7a14c3b
position:
  x: 630
  y: 78
previewCodeHash: f4831a874b1b58a9
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792940615159
config:
  output_type: table
  catalog: workspace
  schema: visual_data_prep
  table_name: products_clean
  write_mode: append
checkpoint:
  mode: auto
input:
  - node: guardrails_c0d93b7e
    input_port: data
    output_port: passed
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "visual_data_prep",
    "table_name": "products_clean",
    "write_mode": "append",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["guardrails_c0d93b7e.passed"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM workspace.visual_data_prep.products_clean").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_3b8e5c90.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_3b8e5c90" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_3b8e5c90.result"])
if "output_3b8e5c90" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_3b8e5c90", "counts": {"result": int(ctx["output_3b8e5c90.result"].count())}}}, raw=True)

In [0]:
"""
id: prepare_6e2a7f41
template: prepare
templateVersion: 2.0.0
name: products failure reasons
parentId: group_e7a14c3b
position:
  x: 630
  y: 233
previewCodeHash: d52f58868e428974
codegenHash: 7b3643f91c41b910
updatedAt: 1792940615159
config:
  actions:
    - type: formula
      id: a_dNSaR9lE
      enabled: true
      target: quality_check_reason
      expression: |
        CONCAT_WS('; ',
          CASE WHEN product_id IS NULL THEN 'product_id is missing' END,
          CASE WHEN product_name IS NULL THEN 'product_name is missing' END,
          CASE WHEN category IS NULL THEN 'category is missing' END,
          CASE WHEN brand IS NULL THEN 'brand is missing' END,
          CASE WHEN unit_price IS NULL THEN 'unit_price is missing' END,
          CASE WHEN unit_price < 0 THEN 'unit_price is negative' END
        )
checkpoint:
  mode: auto
input:
  - node: guardrails_c0d93b7e
    input_port: data
    output_port: failed
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
from uuid import uuid4
import pyspark.sql.functions as F
import pyspark.sql.types as T

def _lb_run_bound_sql(df, spark, make_query):
    view_name = "_lb_input_" + uuid4().hex
    df.createOrReplaceTempView(view_name)
    try:
        query = make_query(view_name)
        args = {}
        if "dbutils" in globals() and hasattr(dbutils, "widgets") and hasattr(dbutils.widgets, "getAll"):
            args = dict(dbutils.widgets.getAll())
        evaluated = spark.sql(query, args=args)
    except Exception:
        spark.catalog.dropTempView(view_name)
        raise
    globals().setdefault("_lb_views", set()).add(view_name)
    return evaluated

def _lb_apply_bound_expressions(df, spark, assignments):
    occupied = set(df.columns) | set(assignments)
    temporary = {}
    for target in assignments:
        name = "_lb_expression_" + uuid4().hex
        while name in occupied:
            name = "_lb_expression_" + uuid4().hex
        occupied.add(name)
        temporary[target] = name
    expressions = [
        "(" + expression + "\n) AS " + chr(96) + temporary[target] + chr(96)
        for target, expression in assignments.items()
    ]
    evaluated = _lb_run_bound_sql(
        df, spark, lambda view: "SELECT *, " + ", ".join(expressions) + " FROM " + view
    )
    return evaluated.withColumns({target: F.col(name) for target, name in temporary.items()}).drop(*temporary.values())

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Coerce a user-provided value to the target column's type.

    UI always feeds text per the operator spec. try_cast, not cast, so a
    value that does not fit the column's type yields NULL instead of
    aborting the query under ANSI mode -- the empty default fill value
    against a numeric column then leaves that column's nulls in place, as
    the fill-null UI promises, rather than raising CAST_INVALID_INPUT.
    """
    if value is None:
        return F.lit(None).cast(target_dtype)
    escaped = str(value).replace(chr(92), chr(92) * 2).replace("'", chr(92) + "'")
    return F.expr("try_cast('" + escaped + "' as " + target_dtype + ")")

def _apply_formula(df, action: Dict[str, Any], spark):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, {target: expression})
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any], spark):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = rendered if action.get("__lb_bind_expression") else F.expr(rendered)
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, assignments)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action, spark)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action, spark)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "id": "a_dNSaR9lE",
            "enabled": True,
            "target": "quality_check_reason",
            "expression": "CONCAT_WS('; ',\n  CASE WHEN product_id IS NULL THEN 'product_id is missing' END,\n  CASE WHEN product_name IS NULL THEN 'product_name is missing' END,\n  CASE WHEN category IS NULL THEN 'category is missing' END,\n  CASE WHEN brand IS NULL THEN 'brand is missing' END,\n  CASE WHEN unit_price IS NULL THEN 'unit_price is missing' END,\n  CASE WHEN unit_price < 0 THEN 'unit_price is negative' END\n)\n"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["guardrails_c0d93b7e.failed"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_6e2a7f41.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_6e2a7f41" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_6e2a7f41.prepared_data"])
if "prepare_6e2a7f41" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_6e2a7f41", "counts": {"prepared_data": int(ctx["prepare_6e2a7f41.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: output_4c1e9b78
template: output
templateVersion: 4.0.0
name: transactions clean
parentId: group_2b6f9e0d
position:
  x: 630
  y: 78
previewCodeHash: 6897bdfa3d46bf48
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792940615249
config:
  output_type: table
  catalog: workspace
  schema: visual_data_prep
  table_name: transactions_clean
  write_mode: append
checkpoint:
  mode: auto
input:
  - node: guardrails_f5b28d4a
    input_port: data
    output_port: passed
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "visual_data_prep",
    "table_name": "transactions_clean",
    "write_mode": "append",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["guardrails_f5b28d4a.passed"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM workspace.visual_data_prep.transactions_clean").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_4c1e9b78.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_4c1e9b78" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_4c1e9b78.result"])
if "output_4c1e9b78" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_4c1e9b78", "counts": {"result": int(ctx["output_4c1e9b78.result"].count())}}}, raw=True)

In [0]:
"""
id: prepare_a7d03e6b
template: prepare
templateVersion: 2.0.0
name: transactions failure reasons
parentId: group_2b6f9e0d
position:
  x: 630
  y: 233
previewCodeHash: 7bd515681a6f1963
codegenHash: 7b3643f91c41b910
updatedAt: 1792940615248
config:
  actions:
    - type: formula
      id: a_u4Yzy99s
      enabled: true
      target: quality_check_reason
      expression: |
        CONCAT_WS('; ',
          CASE WHEN transaction_id IS NULL THEN 'transaction_id is missing' END,
          CASE WHEN customer_id IS NULL THEN 'customer_id is missing' END,
          CASE WHEN product_id IS NULL THEN 'product_id is missing' END,
          CASE WHEN quantity IS NULL THEN 'quantity is missing' END,
          CASE WHEN total_amount IS NULL THEN 'total_amount is missing' END,
          CASE WHEN order_timestamp IS NULL THEN 'order_timestamp is missing' END,
          CASE WHEN quantity < 1 THEN 'quantity is less than 1' END,
          CASE WHEN total_amount < 0 THEN 'total_amount is negative' END
        )
checkpoint:
  mode: auto
input:
  - node: guardrails_f5b28d4a
    input_port: data
    output_port: failed
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
from uuid import uuid4
import pyspark.sql.functions as F
import pyspark.sql.types as T

def _lb_run_bound_sql(df, spark, make_query):
    view_name = "_lb_input_" + uuid4().hex
    df.createOrReplaceTempView(view_name)
    try:
        query = make_query(view_name)
        args = {}
        if "dbutils" in globals() and hasattr(dbutils, "widgets") and hasattr(dbutils.widgets, "getAll"):
            args = dict(dbutils.widgets.getAll())
        evaluated = spark.sql(query, args=args)
    except Exception:
        spark.catalog.dropTempView(view_name)
        raise
    globals().setdefault("_lb_views", set()).add(view_name)
    return evaluated

def _lb_apply_bound_expressions(df, spark, assignments):
    occupied = set(df.columns) | set(assignments)
    temporary = {}
    for target in assignments:
        name = "_lb_expression_" + uuid4().hex
        while name in occupied:
            name = "_lb_expression_" + uuid4().hex
        occupied.add(name)
        temporary[target] = name
    expressions = [
        "(" + expression + "\n) AS " + chr(96) + temporary[target] + chr(96)
        for target, expression in assignments.items()
    ]
    evaluated = _lb_run_bound_sql(
        df, spark, lambda view: "SELECT *, " + ", ".join(expressions) + " FROM " + view
    )
    return evaluated.withColumns({target: F.col(name) for target, name in temporary.items()}).drop(*temporary.values())

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Coerce a user-provided value to the target column's type.

    UI always feeds text per the operator spec. try_cast, not cast, so a
    value that does not fit the column's type yields NULL instead of
    aborting the query under ANSI mode -- the empty default fill value
    against a numeric column then leaves that column's nulls in place, as
    the fill-null UI promises, rather than raising CAST_INVALID_INPUT.
    """
    if value is None:
        return F.lit(None).cast(target_dtype)
    escaped = str(value).replace(chr(92), chr(92) * 2).replace("'", chr(92) + "'")
    return F.expr("try_cast('" + escaped + "' as " + target_dtype + ")")

def _apply_formula(df, action: Dict[str, Any], spark):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, {target: expression})
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any], spark):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = rendered if action.get("__lb_bind_expression") else F.expr(rendered)
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, assignments)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action, spark)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action, spark)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "id": "a_u4Yzy99s",
            "enabled": True,
            "target": "quality_check_reason",
            "expression": "CONCAT_WS('; ',\n  CASE WHEN transaction_id IS NULL THEN 'transaction_id is missing' END,\n  CASE WHEN customer_id IS NULL THEN 'customer_id is missing' END,\n  CASE WHEN product_id IS NULL THEN 'product_id is missing' END,\n  CASE WHEN quantity IS NULL THEN 'quantity is missing' END,\n  CASE WHEN total_amount IS NULL THEN 'total_amount is missing' END,\n  CASE WHEN order_timestamp IS NULL THEN 'order_timestamp is missing' END,\n  CASE WHEN quantity < 1 THEN 'quantity is less than 1' END,\n  CASE WHEN total_amount < 0 THEN 'total_amount is negative' END\n)\n"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["guardrails_f5b28d4a.failed"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_a7d03e6b.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_a7d03e6b" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_a7d03e6b.prepared_data"])
if "prepare_a7d03e6b" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_a7d03e6b", "counts": {"prepared_data": int(ctx["prepare_a7d03e6b.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: output_6d8a2f15
template: output
templateVersion: 4.0.0
name: customer_usa clean
parentId: group_5d2e8a1f
position:
  x: 630
  y: 78
description:
  text: Append input data to an existing table with fast CSV overwrite enabled.
  hash: 873a7ed4
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792940647159
config:
  output_type: table
  catalog: workspace
  schema: visual_data_prep
  table_name: customer_usa_clean
  write_mode: append
checkpoint:
  mode: auto
input:
  - node: guardrails_e3f05a6d
    input_port: data
    output_port: passed
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "visual_data_prep",
    "table_name": "customer_usa_clean",
    "write_mode": "append",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["guardrails_e3f05a6d.passed"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM workspace.visual_data_prep.customer_usa_clean").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_6d8a2f15.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_6d8a2f15" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_6d8a2f15.result"])
if "output_6d8a2f15" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_6d8a2f15", "counts": {"result": int(ctx["output_6d8a2f15.result"].count())}}}, raw=True)

In [0]:
"""
id: prepare_1b9c4e72
template: prepare
templateVersion: 2.0.0
name: customer_usa failure reasons
parentId: group_5d2e8a1f
position:
  x: 630
  y: 233
description:
  text: Create a new column listing reasons for missing or invalid customer data.
  hash: 8bfe6980
previewCodeHash: 1e196d9787f84cdf
codegenHash: 7b3643f91c41b910
updatedAt: 1792940647159
config:
  actions:
    - type: formula
      id: a_j-T0xvN4
      enabled: true
      target: quality_check_reason
      expression: |
        CONCAT_WS('; ',
          CASE WHEN customer_id IS NULL THEN 'customer_id is missing' END,
          CASE WHEN first_name IS NULL THEN 'first_name is missing' END,
          CASE WHEN last_name IS NULL THEN 'last_name is missing' END,
          CASE WHEN email IS NULL THEN 'email is missing' END,
          CASE WHEN phone IS NULL THEN 'phone is missing' END,
          CASE WHEN NOT email RLIKE '^[^@]+@[^@]+\\.[^@]+$' THEN 'email is invalid' END
        )
checkpoint:
  mode: auto
input:
  - node: guardrails_e3f05a6d
    input_port: data
    output_port: failed
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
from uuid import uuid4
import pyspark.sql.functions as F
import pyspark.sql.types as T

def _lb_run_bound_sql(df, spark, make_query):
    view_name = "_lb_input_" + uuid4().hex
    df.createOrReplaceTempView(view_name)
    try:
        query = make_query(view_name)
        args = {}
        if "dbutils" in globals() and hasattr(dbutils, "widgets") and hasattr(dbutils.widgets, "getAll"):
            args = dict(dbutils.widgets.getAll())
        evaluated = spark.sql(query, args=args)
    except Exception:
        spark.catalog.dropTempView(view_name)
        raise
    globals().setdefault("_lb_views", set()).add(view_name)
    return evaluated

def _lb_apply_bound_expressions(df, spark, assignments):
    occupied = set(df.columns) | set(assignments)
    temporary = {}
    for target in assignments:
        name = "_lb_expression_" + uuid4().hex
        while name in occupied:
            name = "_lb_expression_" + uuid4().hex
        occupied.add(name)
        temporary[target] = name
    expressions = [
        "(" + expression + "\n) AS " + chr(96) + temporary[target] + chr(96)
        for target, expression in assignments.items()
    ]
    evaluated = _lb_run_bound_sql(
        df, spark, lambda view: "SELECT *, " + ", ".join(expressions) + " FROM " + view
    )
    return evaluated.withColumns({target: F.col(name) for target, name in temporary.items()}).drop(*temporary.values())

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Coerce a user-provided value to the target column's type.

    UI always feeds text per the operator spec. try_cast, not cast, so a
    value that does not fit the column's type yields NULL instead of
    aborting the query under ANSI mode -- the empty default fill value
    against a numeric column then leaves that column's nulls in place, as
    the fill-null UI promises, rather than raising CAST_INVALID_INPUT.
    """
    if value is None:
        return F.lit(None).cast(target_dtype)
    escaped = str(value).replace(chr(92), chr(92) * 2).replace("'", chr(92) + "'")
    return F.expr("try_cast('" + escaped + "' as " + target_dtype + ")")

def _apply_formula(df, action: Dict[str, Any], spark):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, {target: expression})
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any], spark):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = rendered if action.get("__lb_bind_expression") else F.expr(rendered)
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, assignments)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action, spark)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action, spark)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "id": "a_j-T0xvN4",
            "enabled": True,
            "target": "quality_check_reason",
            "expression": "CONCAT_WS('; ',\n  CASE WHEN customer_id IS NULL THEN 'customer_id is missing' END,\n  CASE WHEN first_name IS NULL THEN 'first_name is missing' END,\n  CASE WHEN last_name IS NULL THEN 'last_name is missing' END,\n  CASE WHEN email IS NULL THEN 'email is missing' END,\n  CASE WHEN phone IS NULL THEN 'phone is missing' END,\n  CASE WHEN NOT email RLIKE '^[^@]+@[^@]+\\\\.[^@]+$' THEN 'email is invalid' END\n)\n"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["guardrails_e3f05a6d.failed"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_1b9c4e72.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_1b9c4e72" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_1b9c4e72.prepared_data"])
if "prepare_1b9c4e72" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_1b9c4e72", "counts": {"prepared_data": int(ctx["prepare_1b9c4e72.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: output_b7e1a364
template: output
templateVersion: 4.0.0
name: customer_uk clean
parentId: group_8c3f1d6a
position:
  x: 630
  y: 78
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792940615068
config:
  output_type: table
  catalog: workspace
  schema: visual_data_prep
  table_name: customer_uk_clean
  write_mode: append
checkpoint:
  mode: auto
input:
  - node: guardrails_4a7c0e8b
    input_port: data
    output_port: passed
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "visual_data_prep",
    "table_name": "customer_uk_clean",
    "write_mode": "append",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["guardrails_4a7c0e8b.passed"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM workspace.visual_data_prep.customer_uk_clean").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_b7e1a364.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_b7e1a364" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_b7e1a364.result"])
if "output_b7e1a364" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_b7e1a364", "counts": {"result": int(ctx["output_b7e1a364.result"].count())}}}, raw=True)

In [0]:
"""
id: prepare_92b5d1fc
template: prepare
templateVersion: 2.0.0
name: customer_uk failure reasons
parentId: group_8c3f1d6a
position:
  x: 630
  y: 233
codegenHash: 7b3643f91c41b910
updatedAt: 1792940615068
config:
  actions:
    - type: formula
      id: a_kzMbWLIe
      enabled: true
      target: quality_check_reason
      expression: |
        CONCAT_WS('; ',
          CASE WHEN customer_id IS NULL THEN 'customer_id is missing' END,
          CASE WHEN first_name IS NULL THEN 'first_name is missing' END,
          CASE WHEN last_name IS NULL THEN 'last_name is missing' END,
          CASE WHEN email IS NULL THEN 'email is missing' END,
          CASE WHEN phone IS NULL THEN 'phone is missing' END,
          CASE WHEN NOT email RLIKE '^[^@]+@[^@]+\\.[^@]+$' THEN 'email is invalid' END
        )
checkpoint:
  mode: auto
input:
  - node: guardrails_4a7c0e8b
    input_port: data
    output_port: failed
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
from uuid import uuid4
import pyspark.sql.functions as F
import pyspark.sql.types as T

def _lb_run_bound_sql(df, spark, make_query):
    view_name = "_lb_input_" + uuid4().hex
    df.createOrReplaceTempView(view_name)
    try:
        query = make_query(view_name)
        args = {}
        if "dbutils" in globals() and hasattr(dbutils, "widgets") and hasattr(dbutils.widgets, "getAll"):
            args = dict(dbutils.widgets.getAll())
        evaluated = spark.sql(query, args=args)
    except Exception:
        spark.catalog.dropTempView(view_name)
        raise
    globals().setdefault("_lb_views", set()).add(view_name)
    return evaluated

def _lb_apply_bound_expressions(df, spark, assignments):
    occupied = set(df.columns) | set(assignments)
    temporary = {}
    for target in assignments:
        name = "_lb_expression_" + uuid4().hex
        while name in occupied:
            name = "_lb_expression_" + uuid4().hex
        occupied.add(name)
        temporary[target] = name
    expressions = [
        "(" + expression + "\n) AS " + chr(96) + temporary[target] + chr(96)
        for target, expression in assignments.items()
    ]
    evaluated = _lb_run_bound_sql(
        df, spark, lambda view: "SELECT *, " + ", ".join(expressions) + " FROM " + view
    )
    return evaluated.withColumns({target: F.col(name) for target, name in temporary.items()}).drop(*temporary.values())

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Coerce a user-provided value to the target column's type.

    UI always feeds text per the operator spec. try_cast, not cast, so a
    value that does not fit the column's type yields NULL instead of
    aborting the query under ANSI mode -- the empty default fill value
    against a numeric column then leaves that column's nulls in place, as
    the fill-null UI promises, rather than raising CAST_INVALID_INPUT.
    """
    if value is None:
        return F.lit(None).cast(target_dtype)
    escaped = str(value).replace(chr(92), chr(92) * 2).replace("'", chr(92) + "'")
    return F.expr("try_cast('" + escaped + "' as " + target_dtype + ")")

def _apply_formula(df, action: Dict[str, Any], spark):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, {target: expression})
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any], spark):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = rendered if action.get("__lb_bind_expression") else F.expr(rendered)
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, assignments)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action, spark)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action, spark)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "id": "a_kzMbWLIe",
            "enabled": True,
            "target": "quality_check_reason",
            "expression": "CONCAT_WS('; ',\n  CASE WHEN customer_id IS NULL THEN 'customer_id is missing' END,\n  CASE WHEN first_name IS NULL THEN 'first_name is missing' END,\n  CASE WHEN last_name IS NULL THEN 'last_name is missing' END,\n  CASE WHEN email IS NULL THEN 'email is missing' END,\n  CASE WHEN phone IS NULL THEN 'phone is missing' END,\n  CASE WHEN NOT email RLIKE '^[^@]+@[^@]+\\\\.[^@]+$' THEN 'email is invalid' END\n)\n"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["guardrails_4a7c0e8b.failed"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_92b5d1fc.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_92b5d1fc" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_92b5d1fc.prepared_data"])
if "prepare_92b5d1fc" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_92b5d1fc", "counts": {"prepared_data": int(ctx["prepare_92b5d1fc.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: output_9a0f3c58
template: output
templateVersion: 4.0.0
name: bad_records_customer_ind
parentId: group_0
position:
  x: 930
  y: 233
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792940614985
config:
  output_type: table
  catalog: workspace
  schema: visual_data_prep
  table_name: bad_records_customer_ind
  write_mode: append
checkpoint:
  mode: auto
input:
  - node: prepare_c81a5f03
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "visual_data_prep",
    "table_name": "bad_records_customer_ind",
    "write_mode": "append",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_c81a5f03.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM workspace.visual_data_prep.bad_records_customer_ind").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_9a0f3c58.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_9a0f3c58" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_9a0f3c58.result"])
if "output_9a0f3c58" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_9a0f3c58", "counts": {"result": int(ctx["output_9a0f3c58.result"].count())}}}, raw=True)

In [0]:
"""
id: output_d49f1a2c
template: output
templateVersion: 4.0.0
name: bad_records_products
parentId: group_e7a14c3b
position:
  x: 930
  y: 233
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792940615159
config:
  output_type: table
  catalog: workspace
  schema: visual_data_prep
  table_name: bad_records_products
  write_mode: append
checkpoint:
  mode: auto
input:
  - node: prepare_6e2a7f41
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "visual_data_prep",
    "table_name": "bad_records_products",
    "write_mode": "append",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_6e2a7f41.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM workspace.visual_data_prep.bad_records_products").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_d49f1a2c.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_d49f1a2c" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_d49f1a2c.result"])
if "output_d49f1a2c" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_d49f1a2c", "counts": {"result": int(ctx["output_d49f1a2c.result"].count())}}}, raw=True)

In [0]:
"""
id: output_e8f6a2d3
template: output
templateVersion: 4.0.0
name: bad_records_transactions
parentId: group_2b6f9e0d
position:
  x: 930
  y: 233
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792940615249
config:
  output_type: table
  catalog: workspace
  schema: visual_data_prep
  table_name: bad_records_transactions
  write_mode: append
checkpoint:
  mode: auto
input:
  - node: prepare_a7d03e6b
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "visual_data_prep",
    "table_name": "bad_records_transactions",
    "write_mode": "append",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_a7d03e6b.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM workspace.visual_data_prep.bad_records_transactions").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_e8f6a2d3.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_e8f6a2d3" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_e8f6a2d3.result"])
if "output_e8f6a2d3" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_e8f6a2d3", "counts": {"result": int(ctx["output_e8f6a2d3.result"].count())}}}, raw=True)

In [0]:
"""
id: output_f2c47b3e
template: output
templateVersion: 4.0.0
name: bad_records_customer_usa
parentId: group_5d2e8a1f
position:
  x: 930
  y: 233
description:
  text: Add data by appending to an existing table in specified workspace schema, creating the table if needed.
  hash: 1b906845
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792940647159
config:
  output_type: table
  catalog: workspace
  schema: visual_data_prep
  table_name: bad_records_customer_usa
  write_mode: append
checkpoint:
  mode: auto
input:
  - node: prepare_1b9c4e72
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "visual_data_prep",
    "table_name": "bad_records_customer_usa",
    "write_mode": "append",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_1b9c4e72.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM workspace.visual_data_prep.bad_records_customer_usa").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_f2c47b3e.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_f2c47b3e" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_f2c47b3e.result"])
if "output_f2c47b3e" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_f2c47b3e", "counts": {"result": int(ctx["output_f2c47b3e.result"].count())}}}, raw=True)

In [0]:
"""
id: combine_31
template: combine
templateVersion: 2.0.0
name: good_records_combine_all_countries
position:
  x: 1260
  y: 836
description:
  text: Combine multiple inputs by union, matching columns by name and allowing missing columns.
  hash: e3fee704
previewCodeHash: 1a1ae5b4f94b62c9
codegenHash: fcd9d3979c7d02f2
updatedAt: 1792941445795
config:
  operator: UNION
  quantifier: ALL
  union_by_name: true
  allow_missing_columns: true
checkpoint:
  mode: auto
input:
  - node: output_2e6b9d47
    input_port: data
    output_port: result
  - node: output_b7e1a364
    input_port: data
    output_port: result
  - node: output_6d8a2f15
    input_port: data
    output_port: result
"""

# generated from the system
from functools import reduce
from typing import Any, Callable, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = list(inputs.get("data") or [])
    if len(dfs) < 1:
        raise ValueError(
            f"Combine requires at least 1 input, got {len(dfs)}. "
            "Wire one or more upstream operators into the 'data' port."
        )

    operator = config.get("operator", "UNION")
    quantifier = config.get("quantifier", "DISTINCT")
    union_by_name = bool(config.get("union_by_name", False))
    allow_missing_columns = bool(config.get("allow_missing_columns", False))

    pair: Callable[[Any, Any], Any]
    if operator == "UNION":
        if union_by_name:
            def pair(a: Any, b: Any) -> Any:
                return a.unionByName(b, allowMissingColumns=allow_missing_columns)
        else:
            def pair(a: Any, b: Any) -> Any:
                return a.union(b)
    elif operator == "INTERSECT":
        if quantifier == "ALL":
            def pair(a: Any, b: Any) -> Any:
                return a.intersectAll(b)
        else:
            def pair(a: Any, b: Any) -> Any:
                return a.intersect(b)
    elif operator in ("EXCEPT", "MINUS"):
        if quantifier == "ALL":
            def pair(a: Any, b: Any) -> Any:
                return a.exceptAll(b)
        else:
            def pair(a: Any, b: Any) -> Any:
                return a.subtract(b)
    else:
        raise ValueError(f"Unsupported combine operator: {operator}")

    result = reduce(pair, dfs)
    if quantifier == "DISTINCT" and (operator == "UNION" or len(dfs) == 1):
        result = result.distinct()
    return {"combined_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "operator": "UNION",
    "quantifier": "ALL",
    "union_by_name": True,
    "allow_missing_columns": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": [
        ctx["output_2e6b9d47.result"],
        ctx["output_b7e1a364.result"],
        ctx["output_6d8a2f15.result"]
    ],
    "data__sources": [
        {
            "node": "output_2e6b9d47",
            "output_port": "result",
            "name": "customer_ind clean",
            "df_name": "customer_ind_clean"
        },
        {
            "node": "output_b7e1a364",
            "output_port": "result",
            "name": "customer_uk clean",
            "df_name": "customer_uk_clean"
        },
        {
            "node": "output_6d8a2f15",
            "output_port": "result",
            "name": "customer_usa clean",
            "df_name": "customer_usa_clean"
        }
    ]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["combine_31.combined_data"] = out["combined_data"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["combine_31.combined_data"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["combine_31.combined_data"]
        try:
            ctx["combine_31.combined_data"] = ctx["combine_31.combined_data"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["combine_31.combined_data"] is not _lb_cp_original_0:
                try:
                    ctx["combine_31.combined_data"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["combine_31.combined_data"] = _lb_cp_original_0
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "combine_31" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["combine_31.combined_data"])
if "combine_31" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "combine_31", "counts": {"combined_data": int(ctx["combine_31.combined_data"].count())}}}, raw=True)

In [0]:
"""
id: output_0c5f8e29
template: output
templateVersion: 4.0.0
name: bad_records_customer_uk
parentId: group_8c3f1d6a
position:
  x: 930
  y: 233
codegenHash: 3ffb65aedbea9b19
updatedAt: 1792940615068
config:
  output_type: table
  catalog: workspace
  schema: visual_data_prep
  table_name: bad_records_customer_uk
  write_mode: append
checkpoint:
  mode: auto
input:
  - node: prepare_92b5d1fc
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import json
import uuid
from typing import Dict, Any, List

DELTA_COLUMN_MAPPING_PROP = "'delta.columnMapping.mode' = 'id'"

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    + DELTA_COLUMN_MAPPING_PROP
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _table_exists(spark, qualified_name: str) -> bool:
    try:
        return spark.catalog.tableExists(qualified_name)
    except Exception:
        return False

def _existing_column_mapping_mode(spark, qualified_name: str) -> str:
    if spark is None or not qualified_name:
        return ""
    try:
        rows = spark.sql(
            f"SHOW TBLPROPERTIES {qualified_name} ('delta.columnMapping.mode')"
        ).collect()
        if rows:
            value = str(rows[0]["value"])
            if value in ("name", "id"):
                return value
    except Exception:
        pass
    return ""

def _column_mapping_prop(spark, full_name: str) -> str:
    if spark is None or not full_name:
        return DELTA_COLUMN_MAPPING_PROP
    if not _table_exists(spark, full_name):
        return DELTA_COLUMN_MAPPING_PROP
    existing = _existing_column_mapping_mode(spark, full_name)
    if existing:
        return f"'delta.columnMapping.mode' = '{existing}'"
    return ""

def _tbl_properties_clause(
    user_props: str, format_value: str, spark=None, full_name: str = ""
) -> str:
    parts: List[str] = []
    table_exists = bool(
        spark is not None and full_name and _table_exists(spark, full_name)
    )
    column_mapping = _column_mapping_prop(spark, full_name)
    if format_value == "uniform":
        uniform_base = (
            "'delta.enableIcebergCompatV2' = 'true', "
            "'delta.universalFormat.enabledFormats' = 'iceberg'"
        )
        if column_mapping:
            parts.append(uniform_base + ", " + column_mapping)
        elif not table_exists:
            parts.append(UNIFORM_PROPS)
        else:
            parts.append(uniform_base)
    elif format_value in ("delta", "default") and column_mapping:
        parts.append(column_mapping)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json", "excel"}

FILE_EXTENSIONS = {"csv": "csv", "json": "json", "excel": "xlsx"}

MAX_SINGLE_FILE_ROWS = 1_000_000

MAX_FAST_CSV_CELLS = 50_000_000

MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES = 2 * 1024 * 1024 * 1024

_FAST_CSV_BYTES_PER_CELL_OVERHEAD = 16

MAX_EXCEL_CELLS = 5_000_000

MAX_SPLIT_GROUPS = 50

def _excel_row_cap(num_cols: int) -> int:
    return min(MAX_SINGLE_FILE_ROWS, max(1, MAX_EXCEL_CELLS // max(1, num_cols)))

_FORMULA_TRIGGERS = "=+-@\t\r"

def _neutralize_formula(value):
    if isinstance(value, str) and value and value[0] in _FORMULA_TRIGGERS:
        return "'" + value
    return value

def _pandas_line_terminator_parameter(pd) -> str:
    import inspect

    return (
        "lineterminator"
        if "lineterminator" in inspect.signature(pd.DataFrame.to_csv).parameters
        else "line_terminator"
    )

def _write_csv_overwrite(df, path: str, include_header: bool = True) -> bool:
    import os
    import shutil
    import tempfile

    from pyspark.sql import functions as F
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return False

    supported_type_names = {
        "boolean",
        "byte",
        "char",
        "date",
        "decimal",
        "double",
        "float",
        "integer",
        "long",
        "short",
        "string",
        "timestamp",
        "timestamp_ntz",
        "varchar",
        "void",
    }
    normalized_names = [name.casefold() for name in df.columns]
    if (
        not df.columns
        or len(set(normalized_names)) != len(normalized_names)
        or any(field.dataType.typeName() not in supported_type_names for field in df.schema.fields)
    ):
        return False

    spark_output_directory = os.path.isdir(path) and os.path.isfile(
        os.path.join(path, "_SUCCESS")
    )
    if os.path.isdir(path) and not spark_output_directory:
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )

    from databricks.sdk import WorkspaceClient

    client = WorkspaceClient()

    expressions = []
    output_column_names = []
    nan_markers = []
    for index, field in enumerate(df.schema.fields):
        escaped_name = field.name.replace("`", "``")
        column = F.col(f"`{escaped_name}`")
        output_column_name = f"__lb_csv_column_{index}"
        output_column_names.append(field.name)
        type_name = field.dataType.typeName()
        if type_name in {"char", "string", "varchar"}:
            first_character = F.substring(column, 1, 1)
            column = F.when(
                first_character.isin(*_FORMULA_TRIGGERS),
                F.concat(F.lit("'"), column),
            ).otherwise(column)
        elif type_name in {"byte", "integer", "long", "short"}:
            column = column.cast("string")
        elif type_name in {"double", "float"}:
            marker = f"__lb_csv_nan_{index}"
            nan_markers.append((output_column_name, marker))
            expressions.append(F.isnan(column).alias(marker))
        elif type_name == "boolean":
            column = F.when(column, F.lit("True")).when(~column, F.lit("False"))
        elif type_name in {"date", "decimal"}:
            column = column.cast("string")
        elif type_name in {"timestamp", "timestamp_ntz"}:
            column = F.regexp_replace(
                F.date_format(column, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
                r"[.]000000$",
                "",
            )
        expressions.append(column.alias(output_column_name))

    fast_csv_row_cap = min(
        MAX_SINGLE_FILE_ROWS,
        max(1, MAX_FAST_CSV_CELLS // len(df.columns)),
    )
    prepared_df = df.select(*expressions).limit(fast_csv_row_cap + 1).persist()
    try:
        payload_size = F.lit(0)
        for name in prepared_df.columns:
            escaped_name = name.replace("`", "``")
            payload_size = payload_size + F.length(
                F.encode(
                    F.coalesce(F.col(f"`{escaped_name}`").cast("string"), F.lit("")),
                    "UTF-8",
                )
            )
        stats = prepared_df.agg(
            F.count(F.lit(1)).alias("row_count"),
            F.sum(payload_size).alias("payload_bytes"),
        ).collect()[0]
        row_count = stats["row_count"]
        if row_count > fast_csv_row_cap:
            if fast_csv_row_cap < MAX_SINGLE_FILE_ROWS:
                raise ValueError(
                    f"Output: result too wide for safe single-file CSV export "
                    f"(over {MAX_FAST_CSV_CELLS:,} cells) — write to a table instead."
                )
            raise ValueError(
                f"Output: result too large for single-file export "
                f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
            )
        estimated_driver_bytes = (stats["payload_bytes"] or 0) + (
            row_count * len(prepared_df.columns) * _FAST_CSV_BYTES_PER_CELL_OVERHEAD
        )
        if estimated_driver_bytes > MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES:
            raise ValueError(
                f"Output: result too large for safe single-file CSV export "
                f"(estimated driver memory exceeds "
                f"{MAX_FAST_CSV_ESTIMATED_DRIVER_BYTES // (1024 * 1024):,} MiB) — "
                f"write to a table instead."
            )
        pdf = prepared_df.toPandas()
    finally:
        prepared_df.unpersist()
    for column_name, marker in nan_markers:
        nan_mask = pdf[marker].fillna(False)
        finite_mask = ~pdf[column_name].isna()
        pdf[column_name] = pdf[column_name].astype(object)
        pdf.loc[finite_mask, column_name] = pdf.loc[finite_mask, column_name].map(
            lambda value: repr(float(value))
        )
        pdf.loc[nan_mask, column_name] = "nan"
        pdf.loc[~nan_mask & ~finite_mask, column_name] = None
        pdf.drop(columns=[marker], inplace=True)
    pdf.columns = output_column_names
    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        pdf.to_csv(
            local_tmp,
            index=False,
            header=include_header,
            **{_pandas_line_terminator_parameter(pd): "\r\n"},
        )
        if spark_output_directory:
            shutil.rmtree(path)
        with open(local_tmp, "rb") as f:
            client.files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)
    return True

def _file_path(
    catalog: str, schema: str, volume: str, file_name: str, directory_path: str = ""
) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        if directory_path:
            parts = directory_path.split("/")
            if (
                directory_path.startswith("/")
                or directory_path.endswith("/")
                or chr(92) in directory_path
                or any(part in ("", ".", "..") for part in parts)
            ):
                raise ValueError(
                    f"Output: invalid 'directory_path' {directory_path!r}: it must be "
                    "a relative path beneath the selected volume."
                )
        schema_segment = schema.replace(".", "/")
        directory_segment = f"/{directory_path}" if directory_path else ""
        return f"/Volumes/{catalog}/{schema_segment}/{volume}{directory_segment}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _sanitize_split_value(value) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(c for c in text if c not in "/\\").strip()
    cleaned = cleaned.replace("..", ".")
    cleaned = cleaned.strip(". ")
    return cleaned or "none"

def _split_file_name(file_name: str, value, file_type: str, used: set) -> str:
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = file_name
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    base = f"{stem}_{_sanitize_split_value(value)}"
    candidate = base
    n = 2
    while candidate.lower() in used:
        candidate = f"{base}_{n}"
        n += 1
    used.add(candidate.lower())
    return candidate

_INVALID_SHEET_CHARS = "[]:*?/\\"
_SPLIT_MAP_SHEET = "_lb_split_map"

def _normalize_user_sheet_name(sheet_name: str) -> str:
    if not sheet_name:
        return ""
    cleaned = "".join(
        c for c in sheet_name if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    return cleaned[:31]

def _sanitize_sheet_name(value, used: set) -> str:
    text = "none" if value is None else str(value)
    cleaned = "".join(
        c for c in text if c not in _INVALID_SHEET_CHARS and ord(c) >= 32
    ).strip()
    cleaned = cleaned.strip("'") or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    n = 2
    while candidate.lower() in used:
        tag = f"_{n}"
        candidate = cleaned[: 31 - len(tag)] + tag
        n += 1
    used.add(candidate.lower())
    return candidate

def _split_value_key(value) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return f"b:{int(value)}"
    if isinstance(value, int):
        return f"i:{value}"
    if isinstance(value, float):
        return f"f:{value}"
    return f"s:{value}"

def _assign_split_names(values, name_for, prior_names=None):
    used: set = set()
    assigned: Dict[int, str] = {}
    priors = list(prior_names) if prior_names is not None else [None] * len(values)
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        prior = priors[i] if i < len(priors) else None
        if prior:
            assigned[i] = prior
            used.add(prior.lower())
    for i in sorted(range(len(values)), key=lambda j: str(values[j])):
        if i in assigned:
            continue
        assigned[i] = name_for(values[i], used)
    return [assigned[i] for i in range(len(values))]

def _split_sheet_names(values, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(values, _sanitize_sheet_name, prior_names)

def _split_file_names(file_name: str, values, file_type: str, prior_by_key=None):
    prior_by_key = prior_by_key or {}
    prior_names = [prior_by_key.get(_split_value_key(v)) for v in values]
    return _assign_split_names(
        values,
        lambda value, used: _split_file_name(file_name, value, file_type, used),
        prior_names,
    )

def _load_sheet_split_map(workbook):
    if _SPLIT_MAP_SHEET not in workbook.sheetnames:
        return {}
    mapping = {}
    for row in workbook[_SPLIT_MAP_SHEET].iter_rows(min_row=2, values_only=True):
        if not row or row[0] is None or row[1] is None:
            continue
        mapping[str(row[0])] = str(row[1])
    return mapping

def _write_sheet_split_map(workbook, mapping) -> None:
    if _SPLIT_MAP_SHEET in workbook.sheetnames:
        del workbook[_SPLIT_MAP_SHEET]
    worksheet = workbook.create_sheet(_SPLIT_MAP_SHEET)
    worksheet.sheet_state = "hidden"
    worksheet.append(["value_key", "sheet_name"])
    for key in sorted(mapping):
        worksheet.append([key, mapping[key]])

def _split_manifest_path(directory: str, base_file_name: str, file_type: str) -> str:
    import os

    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    return os.path.join(directory, f"{stem}.lb-split-manifest.json")

def _load_file_split_manifest(directory: str, base_file_name: str, file_type: str):
    import json
    import os

    path = _split_manifest_path(directory, base_file_name, file_type)
    if not os.path.isfile(path):
        return {}
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    groups = payload.get("groups") if isinstance(payload, dict) else None
    if not isinstance(groups, dict):
        return {}
    return {str(k): str(v) for k, v in groups.items()}

def _write_file_split_manifest(directory: str, base_file_name: str, file_type: str, mapping) -> None:
    import json
    import os

    if not directory:
        return
    os.makedirs(directory, exist_ok=True)
    path = _split_manifest_path(directory, base_file_name, file_type)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump({"version": 1, "groups": mapping}, handle, indent=2, sort_keys=True)

def _staged_publisher():
    import os
    import shutil
    import tempfile

    pending: List[Any] = []

    def _discard() -> None:
        for local_tmp, _ in pending:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    def stage_for(dest: str):
        def _stage(write_body) -> None:
            fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
            os.close(fd)
            try:
                write_body(local_tmp)
            except BaseException:
                if os.path.isfile(local_tmp):
                    os.remove(local_tmp)
                raise
            pending.append((local_tmp, dest))

        return _stage

    def publish():
        from databricks.sdk import WorkspaceClient

        client = WorkspaceClient()
        remote_staging: List[Any] = []
        try:
            for local_tmp, dest in pending:
                staging_dest = f"{dest}.lb-publishing"
                if os.path.isdir(staging_dest):
                    shutil.rmtree(staging_dest)
                with open(local_tmp, "rb") as f:
                    client.files.upload(staging_dest, f, overwrite=True)
                remote_staging.append((staging_dest, dest))
            for staging_dest, dest in remote_staging:
                if os.path.isdir(dest):
                    shutil.rmtree(dest)
                with open(staging_dest, "rb") as f:
                    client.files.upload(dest, f, overwrite=True)
        finally:
            for staging_dest, _ in remote_staging:
                if os.path.isfile(staging_dest):
                    os.remove(staging_dest)
            _discard()
        return [dest for _, dest in pending]

    return stage_for, publish, _discard

def _remove_stale_split_files(directory: str, base_file_name: str, file_type: str, keep) -> None:
    import os

    if not directory or not os.path.isdir(directory):
        return
    suffix = f".{FILE_EXTENSIONS.get(file_type, file_type)}"
    stem = os.path.basename(base_file_name)
    if stem.lower().endswith(suffix):
        stem = stem[: -len(suffix)]
    if not stem:
        return
    prefix = f"{stem}_"
    keep_names = {os.path.basename(p) for p in keep}
    for name in sorted(os.listdir(directory)):
        if name in keep_names:
            continue
        if not name.startswith(prefix) or not name.lower().endswith(suffix.lower()):
            continue
        if len(name) <= len(prefix) + len(suffix):
            continue
        stale = os.path.join(directory, name)
        if os.path.isfile(stale):
            os.remove(stale)

def _align_source_keys(existing_header, columns, safe, where: str):
    pending: Dict[str, List[Any]] = {}
    for c in columns:
        pending.setdefault(str(safe(c)), []).append(c)
    source_keys: List[Any] = []
    for h in existing_header:
        bucket = pending.get(str(h))
        if not bucket:
            raise ValueError(
                f"Output: cannot append to {where}: the existing column {str(h)!r} "
                f"occurs more times than the data provides."
            )
        source_keys.append(bucket.pop(0))
    return source_keys

def _group_rows(rows, split_column: str):
    groups: List[Any] = []
    index: Dict[Any, int] = {}
    for row in rows:
        key = row[split_column]
        if key not in index:
            if len(groups) >= MAX_SPLIT_GROUPS:
                raise ValueError(
                    f"Output: splitting by {split_column!r} would produce more than "
                    f"{MAX_SPLIT_GROUPS} sheets/files. Pick a lower-cardinality column "
                    f"or write to a table instead."
                )
            index[key] = len(groups)
            groups.append((key, []))
        groups[index[key]][1].append(row)
    return groups

def _cell_reference(ref: str):
    from openpyxl.utils.cell import coordinate_to_tuple

    try:
        return coordinate_to_tuple(ref.strip().upper())
    except Exception as exc:
        raise ValueError(
            f"Output: invalid cell reference {ref!r} in the range — use A1 notation."
        ) from exc

def _write_single_file(
    rows, columns, file_type: str, path: str, append: bool, include_header: bool, stage=None
) -> None:
    import csv
    import json
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    _stage = stage if stage is not None else _upload_stage

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and sorted(header) != sorted(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                elif include_header:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([_neutralize_formula(row[c]) for c in header])

        _stage(_write_csv)
        return

    if file_type == "excel":
        _write_excel_single(
            rows, columns, path, appending, include_header, "", _stage
        )
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
        if records:
            existing_columns = (
                list(records[0].keys()) if isinstance(records[0], dict) else []
            )
            if existing_columns and sorted(existing_columns) != sorted(columns):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{sorted(columns)} do not match the existing file's "
                    f"columns {sorted(existing_columns)}."
                )
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _excel_illegal_re():
    import re as _re

    try:
        from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE as _lb_illegal
    except ImportError:
        _lb_illegal = _re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
    return _lb_illegal

def _require_openpyxl():
    try:
        import openpyxl
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Writing Excel files requires the 'openpyxl' package, which "
            "is not installed in this environment. Add "
            "openpyxl==3.1.5 to the notebook environment "
            "and apply it, then run again."
        ) from exc

def _write_excel_single(rows, columns, path, appending, include_header, sheet_name, stage) -> None:
    import io

    _require_openpyxl()
    import openpyxl
    import pandas as pd

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""

    if appending:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            if target_sheet in workbook.sheetnames:
                worksheet = workbook[target_sheet]
            else:
                worksheet = workbook.create_sheet(target_sheet)
        else:
            worksheet = workbook.active

        first = next(worksheet.iter_rows(values_only=True), None)
        if first is not None:
            header = list(first)
            if sorted(str(c) for c in header) != sorted(
                str(_excel_safe(c)) for c in columns
            ):
                raise ValueError(
                    f"Output: cannot append to {path}: the data's columns "
                    f"{[str(_excel_safe(c)) for c in columns]} do not match the existing "
                    f"file's columns {[str(c) for c in header]}."
                )
            source_keys = _align_source_keys(header, columns, _excel_safe, path)
            existing_rows = max(0, (worksheet.max_row or 0) - 1)
            row_cap = _excel_row_cap(len(header))
            if existing_rows + len(rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(header):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[k])) for k in source_keys]
                )
        else:
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            row_cap = _excel_row_cap(len(columns))
            if len(rows) >= row_cap:
                raise ValueError(
                    f"Output: writing {len(rows):,} rows would grow {path} past "
                    f"the single-file excel export limit ({MAX_EXCEL_CELLS:,} cells "
                    f"at {len(columns):,} columns) — write to a table instead."
                )
            for row in rows:
                worksheet.append(
                    [_neutralize_formula(_excel_safe(row[c])) for c in columns]
                )

        def _write_excel(tmp_path: str) -> None:
            workbook.save(tmp_path)

        stage(_write_excel)
        return

    header = list(columns)
    source_keys = list(columns)
    combined = [
        [_neutralize_formula(_excel_safe(row[k])) for k in source_keys] for row in rows
    ]
    new_df = pd.DataFrame(combined, columns=[_excel_safe(c) for c in header])
    to_excel_kwargs = {"index": False, "engine": "openpyxl", "header": include_header}
    if target_sheet:
        to_excel_kwargs["sheet_name"] = target_sheet

    def _write_excel(tmp_path: str) -> None:
        buffer = io.BytesIO()
        new_df.to_excel(buffer, **to_excel_kwargs)
        with open(tmp_path, "wb") as out:
            out.write(buffer.getvalue())

    stage(_write_excel)

def _write_excel_sheets(groups, columns, path, include_header, append) -> None:
    import os
    import shutil
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    if os.path.isdir(path):
        if os.path.isfile(os.path.join(path, "_SUCCESS")):
            shutil.rmtree(path)
        else:
            raise ValueError(
                f"Output: cannot write file to {path}: a directory already exists at that path."
            )

    appending = append and os.path.isfile(path)

    if appending:
        workbook = openpyxl.load_workbook(path)
    else:
        workbook = openpyxl.Workbook()
        workbook.remove(workbook.active)

    effective_groups = groups if groups else [(None, [])]
    values = [value for value, _ in effective_groups]
    prior_map = _load_sheet_split_map(workbook) if appending else {}
    prior_for_assign = {
        key: name
        for key, name in prior_map.items()
        if name in workbook.sheetnames
    }
    sheet_names = _split_sheet_names(values, prior_for_assign if appending else None)
    updated_map = dict(prior_map) if appending else {}
    for value, target_name in zip(values, sheet_names):
        updated_map[_split_value_key(value)] = target_name
    row_cap = _excel_row_cap(len(columns))
    for (value, group_rows), target_name in zip(effective_groups, sheet_names):
        if appending and target_name in workbook.sheetnames:
            worksheet = workbook[target_name]
            if include_header:
                first = next(worksheet.iter_rows(values_only=True), None)
                if first is not None:
                    existing_header = [str(c) for c in first]
                    if sorted(existing_header) != sorted(str(_excel_safe(c)) for c in columns):
                        raise ValueError(
                            f"Output: cannot append to sheet {target_name!r} in {path}: "
                            f"the data's columns {[str(_excel_safe(c)) for c in columns]} do not "
                            f"match the sheet's columns {existing_header}."
                        )
                    source_keys = _align_source_keys(
                        existing_header, columns, _excel_safe, f"sheet {target_name!r} in {path}"
                    )
                    existing_rows = max(0, (worksheet.max_row or 0) - 1)
                else:
                    source_keys = list(columns)
                    existing_rows = 0
            else:
                source_keys = list(columns)
                existing_rows = worksheet.max_row or 0
            if existing_rows + len(group_rows) >= row_cap:
                raise ValueError(
                    f"Output: appending {len(group_rows):,} rows would grow sheet "
                    f"{target_name!r} in {path} past the single-file excel export limit "
                    f"({MAX_EXCEL_CELLS:,} cells at {len(columns):,} columns) — write to "
                    f"a table instead."
                )
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[k])) for k in source_keys])
        else:
            worksheet = workbook.create_sheet(target_name)
            if include_header:
                worksheet.append([_excel_safe(c) for c in columns])
            for row in group_rows:
                worksheet.append([_neutralize_formula(_excel_safe(row[c])) for c in columns])

    if not workbook.sheetnames or workbook.sheetnames == [_SPLIT_MAP_SHEET]:
        workbook.create_sheet("Sheet")
    _write_sheet_split_map(workbook, updated_map)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _write_excel_range(rows, columns, path, sheet_name, range_start, range_end, include_header) -> None:
    import os
    import tempfile

    _require_openpyxl()
    import openpyxl

    _lb_illegal = _excel_illegal_re()

    def _excel_safe(value):
        if isinstance(value, str):
            return _lb_illegal.sub("", value)
        if isinstance(value, (int, float, bool, type(None))):
            return value
        return _lb_illegal.sub("", str(value))

    start_row, start_col = _cell_reference(range_start)
    needed_rows = len(rows) + (1 if include_header else 0)
    needed_cols = len(columns)
    if range_end:
        end_row, end_col = _cell_reference(range_end)
        if end_row < start_row or end_col < start_col:
            raise ValueError(
                f"Output: range end {range_end!r} must be at or after range start {range_start!r}."
            )
        available_rows = end_row - start_row + 1
        available_cols = end_col - start_col + 1
        if needed_rows > available_rows or needed_cols > available_cols:
            raise ValueError(
                f"Output: the data ({len(rows):,} rows x {needed_cols} columns"
                f"{' plus a header row' if include_header else ''}) does not fit the "
                f"range {range_start}:{range_end} ({available_rows} rows x {available_cols} "
                f"columns). Widen the range or write to a table instead."
            )
    else:
        end_row = start_row + max(needed_rows, 1) - 1
        end_col = start_col + max(needed_cols, 1) - 1

    if os.path.isdir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    target_sheet = _normalize_user_sheet_name(sheet_name) if sheet_name else ""
    exists = os.path.isfile(path)
    if exists:
        workbook = openpyxl.load_workbook(path)
        if target_sheet:
            worksheet = (
                workbook[target_sheet]
                if target_sheet in workbook.sheetnames
                else workbook.create_sheet(target_sheet)
            )
        else:
            worksheet = workbook.active
    else:
        workbook = openpyxl.Workbook()
        worksheet = workbook.active
        if target_sheet:
            worksheet.title = target_sheet

    if range_end:
        clear_end_row, clear_end_col = end_row, end_col
    else:
        clear_end_row = max(end_row, worksheet.max_row)
        probe = start_col
        while probe < worksheet.max_column:
            if all(
                worksheet.cell(row=r, column=probe + 1).value is None
                for r in range(start_row, clear_end_row + 1)
            ):
                break
            probe += 1
        clear_end_col = max(end_col, probe)

    from openpyxl.utils.cell import range_boundaries

    removed_merges = []
    for merged in list(worksheet.merged_cells.ranges):
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(str(merged))
        overlaps = (
            m_min_row <= clear_end_row
            and m_max_row >= start_row
            and m_min_col <= clear_end_col
            and m_max_col >= start_col
        )
        if overlaps:
            removed_merges.append(str(merged))
            worksheet.unmerge_cells(str(merged))

    for r in range(start_row, clear_end_row + 1):
        for c in range(start_col, clear_end_col + 1):
            worksheet.cell(row=r, column=c).value = None

    target_row = start_row
    if include_header:
        for offset, column in enumerate(columns):
            worksheet.cell(row=target_row, column=start_col + offset, value=_excel_safe(column))
        target_row += 1
    for row in rows:
        for offset, column in enumerate(columns):
            worksheet.cell(
                row=target_row,
                column=start_col + offset,
                value=_neutralize_formula(_excel_safe(row[column])),
            )
        target_row += 1

    write_end_row = target_row - 1
    write_end_col = start_col + max(needed_cols, 1) - 1
    for merged in removed_merges:
        m_min_col, m_min_row, m_max_col, m_max_row = range_boundaries(merged)
        overlaps_write = (
            m_min_row <= write_end_row
            and m_max_row >= start_row
            and m_min_col <= write_end_col
            and m_max_col >= start_col
        )
        if not overlaps_write:
            worksheet.merge_cells(merged)

    fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
    os.close(fd)
    try:
        workbook.save(local_tmp)
        from databricks.sdk import WorkspaceClient

        with open(local_tmp, "rb") as f:
            WorkspaceClient().files.upload(path, f, overwrite=True)
    finally:
        if os.path.isfile(local_tmp):
            os.remove(local_tmp)

def _collect_capped(df, file_type: str):
    if file_type == "excel":
        row_cap = _excel_row_cap(len(df.columns))
        rows = df.limit(row_cap + 1).collect()
        if len(rows) > row_cap:
            raise ValueError(
                f"Output: result too large for single-file excel export "
                f"(over the {MAX_EXCEL_CELLS:,}-cell limit at {len(df.columns):,} "
                f"columns) — write to a table instead."
            )
        return rows
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    return rows

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv, json, or excel."
        )
    split_by_column = config.get("split_by_column", "") or ""
    split_mode = config.get("split_mode", "sheets")
    sheet_name = config.get("sheet_name", "") or ""
    include_header = config.get("include_header", True)
    range_start = (config.get("range_start", "") or "").strip()
    range_end = (config.get("range_end", "") or "").strip()
    write_mode = config.get("write_mode", "overwrite")
    columns = df.columns

    if range_end and not range_start:
        raise ValueError(
            "Output: 'range_start' is required to write a cell range "
            "('range_end' alone is not a valid range)."
        )
    if range_start:
        if file_type != "excel":
            raise ValueError("Output: a cell range is only supported for excel file output.")
        if split_by_column:
            raise ValueError(
                "Output: a cell range cannot be combined with split_by_column — "
                "clear one of them."
            )
        if write_mode == "append":
            raise ValueError(
                "Output: a cell range cannot be combined with append — a range writes "
                "into a fixed region, so use write_mode overwrite."
            )

    if split_by_column:
        if split_mode not in ("sheets", "files"):
            raise ValueError(
                f"Output: split_mode {split_mode!r} is not supported. Use 'sheets' or 'files'."
            )
        if split_by_column not in columns:
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} is not a column of the input."
            )
        from pyspark.sql.types import ArrayType, MapType, StructType

        split_type = df.schema[split_by_column].dataType
        if isinstance(split_type, (ArrayType, MapType, StructType)):
            raise ValueError(
                f"Output: split_by_column {split_by_column!r} has type "
                f"{split_type.simpleString()}, which cannot be used to group rows. "
                f"Pick a column with a scalar type (string, number, boolean, date)."
            )
        if split_mode == "sheets" and file_type != "excel":
            raise ValueError(
                "Output: split_mode 'sheets' is only supported for excel file output; "
                "use 'files' for csv/json."
            )

    base_file_name = _with_extension(config.get("file_name", ""), file_type)
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        base_file_name,
        config.get("directory_path", ""),
    )

    if (
        file_type == "csv"
        and not split_by_column
        and write_mode == "overwrite"
        and config.get("__fast_csv_overwrite", False)
        and _write_csv_overwrite(df, path, include_header)
    ):
        return

    rows = _collect_capped(df, file_type)
    append = write_mode == "append"

    if range_start:
        _write_excel_range(
            rows, columns, path, sheet_name, range_start, range_end, include_header
        )
        return

    if split_by_column and split_mode == "sheets":
        groups = _group_rows(rows, split_by_column)
        _write_excel_sheets(groups, columns, path, include_header, append)
        return

    if split_by_column:
        import os

        groups = _group_rows(rows, split_by_column)
        directory = os.path.dirname(path)
        prior_map = (
            _load_file_split_manifest(directory, base_file_name, file_type)
            if append
            else {}
        )
        prior_for_assign = {}
        if append:
            for key, name in prior_map.items():
                candidate = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(name, file_type),
                    config.get("directory_path", ""),
                )
                if os.path.isfile(candidate):
                    prior_for_assign[key] = name
        group_names = _split_file_names(
            config.get("file_name", ""),
            [value for value, _ in groups],
            file_type,
            prior_for_assign if append else None,
        )
        updated_map = dict(prior_map) if append else {}
        for value, group_name in zip([value for value, _ in groups], group_names):
            updated_map[_split_value_key(value)] = group_name
        stage_for, publish, discard = _staged_publisher()
        try:
            for (_, group_rows), group_name in zip(groups, group_names):
                group_path = _file_path(
                    config.get("catalog", ""),
                    config.get("schema", ""),
                    config.get("volume", ""),
                    _with_extension(group_name, file_type),
                    config.get("directory_path", ""),
                )
                if file_type == "excel":
                    _write_excel_single_at(
                        group_rows,
                        columns,
                        group_path,
                        append,
                        include_header,
                        sheet_name,
                        stage_for(group_path),
                    )
                else:
                    _write_single_file(
                        group_rows,
                        columns,
                        file_type,
                        group_path,
                        append,
                        include_header,
                        stage_for(group_path),
                    )
        except BaseException:
            discard()
            raise
        written = publish()
        _write_file_split_manifest(directory, base_file_name, file_type, updated_map)
        if not append:
            _remove_stale_split_files(
                directory, base_file_name, file_type, written
            )
        return

    if file_type == "excel":
        _write_excel_single_at(rows, columns, path, append, include_header, sheet_name)
        return
    _write_single_file(rows, columns, file_type, path, append, include_header)

def _write_excel_single_at(rows, columns, path, append, include_header, sheet_name, stage=None) -> None:
    import os
    import shutil
    import tempfile

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already exists at that path."
        )

    def _upload_stage(write_body) -> None:
        from databricks.sdk import WorkspaceClient

        fd, local_tmp = tempfile.mkstemp(suffix=".lb-output-stage")
        os.close(fd)
        try:
            write_body(local_tmp)
            if os.path.isdir(path) and _is_spark_output_dir(path):
                _remove(path)
            with open(local_tmp, "rb") as f:
                WorkspaceClient().files.upload(path, f, overwrite=True)
        finally:
            if os.path.isfile(local_tmp):
                os.remove(local_tmp)

    appending = append and os.path.isfile(path)
    _write_excel_single(
        rows,
        columns,
        path,
        appending,
        include_header,
        sheet_name,
        stage if stage is not None else _upload_stage,
    )

def _read_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    path_literal = json.dumps(path)
    if file_type == "excel":
        sheet_name = (config.get("sheet_name", "") or "").strip()
        range_start = (config.get("range_start", "") or "").strip()
        range_end = (config.get("range_end", "") or "").strip()
        cell_range = (
            f"{range_start}:{range_end}" if range_start and range_end else range_start
        )
        data_address = (
            f"{sheet_name}!{cell_range}" if sheet_name and cell_range else sheet_name or cell_range
        )
        options = ['format=>"excel"']
        if data_address:
            options.append(f"dataAddress=>{json.dumps(data_address)}")
        options.extend(
            [
                f'headerRows=>{1 if config.get("include_header", True) else 0}',
                'schemaEvolutionMode=>"none"',
            ]
        )
        return spark.sql(f"SELECT * FROM read_files({path_literal}, {', '.join(options)})")
    if file_type == "csv":
        header = "true" if config.get("include_header", True) else "false"
        return spark.sql(
            f'SELECT * FROM read_files({path_literal}, format=>"csv", header=>{header})'
        )
    return spark.sql(f'SELECT * FROM read_files({path_literal}, format=>"{file_type}")')

READ_BACK_MISSING_PATH_CONDITIONS = ("CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES", "PATH_NOT_FOUND")

READ_BACK_RETRY_DELAYS_SECONDS = (0.5, 1.0, 2.0, 4.0, 8.0)

def _spark_error_condition(error):
    condition_getter = getattr(error, "getCondition", None)
    condition = condition_getter() if callable(condition_getter) else None
    if not condition:
        legacy_getter = getattr(error, "getErrorClass", None)
        condition = legacy_getter() if callable(legacy_getter) else None
    return condition

def _read_written_file_result(path: str, file_type: str, config: Dict[str, Any], spark):
    import time

    for delay in (*READ_BACK_RETRY_DELAYS_SECONDS, None):
        try:
            result = _read_file_result(path, file_type, config, spark)
            result.schema
            return result
        except Exception as error:
            if delay is None or _spark_error_condition(error) not in READ_BACK_MISSING_PATH_CONDITIONS:
                raise
            time.sleep(delay)

def _display_write_result(write_result) -> None:
    if write_result is not None and globals().get("ld_display_outputs", False):
        display(write_result)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
):
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    return spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        file_type = config.get("file_type", "csv")
        split_col = (config.get("split_by_column") or "").strip()
        if split_col:
            return {"result": df}
        path = _file_path(
            config.get("catalog", ""),
            config.get("schema", ""),
            config.get("volume", ""),
            _with_extension(config.get("file_name", ""), file_type),
            config.get("directory_path", ""),
        )
        return {"result": _read_written_file_result(path, file_type, config, spark)}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _display_write_result(_run_materialized_view(config, source_view, spark))
        finally:
            spark.catalog.dropTempView(source_view)
        view_full_name = _qualified(catalog, schema, config.get("view_name", ""))
        return {"result": spark.sql(f"SELECT * FROM {view_full_name}")}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(
            table_properties, format_value, spark=spark, full_name=full_name
        )

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            write_result = (
                spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
            )
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            write_result = (
                spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
            )
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            write_result = spark.sql(stmt, args=args) if args else spark.sql(stmt)
        _display_write_result(write_result)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {"result": spark.sql(f"SELECT * FROM {full_name}")}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "workspace",
    "schema": "visual_data_prep",
    "table_name": "bad_records_customer_uk",
    "write_mode": "append",
    "__fast_csv_overwrite": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["prepare_92b5d1fc.prepared_data"]
}
if config["meta_state"]["is_disabled"]:
    try:
        _lb_disabled_persisted_result = spark.sql("SELECT * FROM workspace.visual_data_prep.bad_records_customer_uk").limit(0)
        _lb_disabled_persisted_result.schema
    except Exception as _lb_disabled_persisted_error:
        _lb_disabled_persisted_condition_getter = getattr(_lb_disabled_persisted_error, "getCondition", None)
        _lb_disabled_persisted_error_class = _lb_disabled_persisted_condition_getter() if callable(_lb_disabled_persisted_condition_getter) else None
        if not _lb_disabled_persisted_error_class:
            _lb_disabled_persisted_legacy_error_class_getter = getattr(_lb_disabled_persisted_error, "getErrorClass", None)
            _lb_disabled_persisted_error_class = _lb_disabled_persisted_legacy_error_class_getter() if callable(_lb_disabled_persisted_legacy_error_class_getter) else None
        if _lb_disabled_persisted_error_class not in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND"):
            if config.get("output_type") != "file" or _lb_disabled_persisted_error_class != "CF_PATH_DOES_NOT_EXIST_FOR_READ_FILES":
                raise
        _lb_disabled_persisted_input = inputs.get("data")
        _lb_disabled_persisted_result = _lb_disabled_persisted_input.limit(0) if hasattr(_lb_disabled_persisted_input, "limit") else _lb_disabled_persisted_input
    out = {"result": _lb_disabled_persisted_result}
else:
    out = run(config, inputs, spark)
ctx["output_0c5f8e29.result"] = out["result"]
if globals().get("ld_display_outputs", False) or "output_0c5f8e29" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["output_0c5f8e29.result"])
if "output_0c5f8e29" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "output_0c5f8e29", "counts": {"result": int(ctx["output_0c5f8e29.result"].count())}}}, raw=True)

In [0]:
"""
id: prepare_32
template: prepare
templateVersion: 2.0.0
name: Cleanse combined customers
position:
  x: 2040
  y: 836
description:
  text: Fill country based on customer ID prefix if missing; format phone numbers by country; default gender and acquisition channel; replace blanks with 'N/A'; title case some fields; uppercase status and loyalty tier; trim spaces in names and address.
  hash: 6aa7a7b3
previewCodeHash: 66caca2a3f986844
codegenHash: 7b3643f91c41b910
updatedAt: 1792943291048
config:
  actions:
    - type: formula
      id: a_uYR7m4da
      enabled: true
      target: country
      expression: |
        CASE
          WHEN country IS NULL OR TRIM(country) = '' THEN
            CASE
              WHEN UPPER(customer_id) LIKE 'IND%' THEN 'India'
              WHEN UPPER(customer_id) LIKE 'USA%' THEN 'United States'
              WHEN UPPER(customer_id) LIKE 'UK%' THEN 'United Kingdom'
              ELSE country
            END
          ELSE country
        END
    - type: formula
      id: a_t9h1kb-I
      enabled: true
      target: phone
      expression: |
        CASE
          WHEN UPPER(customer_id) LIKE 'IND%' THEN
            CONCAT('+91 ', REGEXP_REPLACE(REGEXP_REPLACE(phone, '^(\\+91|091|91)\\s*', ''), '(\\d{5})(\\d{5})', '$1 $2'))
          WHEN UPPER(customer_id) LIKE 'USA%' THEN
            CONCAT('+1 ', REGEXP_REPLACE(REGEXP_REPLACE(phone, '^(\\+1|1)\\s*', ''), '(\\d{3})(\\d{3})(\\d{4})', '($1) $2-$3'))
          WHEN UPPER(customer_id) LIKE 'UK%' THEN
            CONCAT('+44 ', REGEXP_REPLACE(REGEXP_REPLACE(phone, '^(\\+44|044|44|0)\\s*', ''), '(\\d{4})(\\d{6})', '$1 $2'))
          ELSE phone
        END
    - type: formula
      id: a_DLve-X_t
      enabled: true
      target: gender
      expression: CASE WHEN gender IS NULL OR TRIM(gender) = '' THEN 'N' ELSE gender END
    - type: formula
      id: a_DyKjqBgP
      enabled: true
      target: acquisition_channel
      expression: |
        CASE
          WHEN acquisition_channel IS NULL OR TRIM(acquisition_channel) = '' THEN 'Unknown'
          ELSE acquisition_channel
        END
    - type: multi_column_formula
      id: a_UWq9rVwA
      enabled: true
      columns:
        - first_name
        - last_name
        - email
        - phone
        - address
        - country
        - age
        - gender
        - status
        - loyalty_tier
        - registration_date
        - acquisition_channel
      expression: CASE WHEN {current_column} IS NULL OR TRIM(CAST({current_column} AS STRING)) = '' THEN 'N/A' ELSE {current_column} END
      prefix: ""
      suffix: ""
    - type: text_case
      id: a_gcdojoVs
      enabled: true
      columns:
        - first_name
        - last_name
        - gender
        - acquisition_channel
      case: title
    - type: text_case
      id: a_6v9cJKxC
      enabled: true
      columns:
        - status
        - loyalty_tier
      case: upper
    - type: trim
      id: a_6Zj6hlKF
      enabled: true
      columns:
        - first_name
        - last_name
        - address
      side: both
checkpoint:
  mode: auto
input:
  - node: combine_31
    input_port: data
    output_port: combined_data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
from uuid import uuid4
import pyspark.sql.functions as F
import pyspark.sql.types as T

def _lb_run_bound_sql(df, spark, make_query):
    view_name = "_lb_input_" + uuid4().hex
    df.createOrReplaceTempView(view_name)
    try:
        query = make_query(view_name)
        args = {}
        if "dbutils" in globals() and hasattr(dbutils, "widgets") and hasattr(dbutils.widgets, "getAll"):
            args = dict(dbutils.widgets.getAll())
        evaluated = spark.sql(query, args=args)
    except Exception:
        spark.catalog.dropTempView(view_name)
        raise
    globals().setdefault("_lb_views", set()).add(view_name)
    return evaluated

def _lb_apply_bound_expressions(df, spark, assignments):
    occupied = set(df.columns) | set(assignments)
    temporary = {}
    for target in assignments:
        name = "_lb_expression_" + uuid4().hex
        while name in occupied:
            name = "_lb_expression_" + uuid4().hex
        occupied.add(name)
        temporary[target] = name
    expressions = [
        "(" + expression + "\n) AS " + chr(96) + temporary[target] + chr(96)
        for target, expression in assignments.items()
    ]
    evaluated = _lb_run_bound_sql(
        df, spark, lambda view: "SELECT *, " + ", ".join(expressions) + " FROM " + view
    )
    return evaluated.withColumns({target: F.col(name) for target, name in temporary.items()}).drop(*temporary.values())

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Coerce a user-provided value to the target column's type.

    UI always feeds text per the operator spec. try_cast, not cast, so a
    value that does not fit the column's type yields NULL instead of
    aborting the query under ANSI mode -- the empty default fill value
    against a numeric column then leaves that column's nulls in place, as
    the fill-null UI promises, rather than raising CAST_INVALID_INPUT.
    """
    if value is None:
        return F.lit(None).cast(target_dtype)
    escaped = str(value).replace(chr(92), chr(92) * 2).replace("'", chr(92) + "'")
    return F.expr("try_cast('" + escaped + "' as " + target_dtype + ")")

def _apply_formula(df, action: Dict[str, Any], spark):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, {target: expression})
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any], spark):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = rendered if action.get("__lb_bind_expression") else F.expr(rendered)
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, assignments)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action, spark)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action, spark)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "id": "a_uYR7m4da",
            "enabled": True,
            "target": "country",
            "expression": "CASE\n  WHEN country IS NULL OR TRIM(country) = '' THEN\n    CASE\n      WHEN UPPER(customer_id) LIKE 'IND%' THEN 'India'\n      WHEN UPPER(customer_id) LIKE 'USA%' THEN 'United States'\n      WHEN UPPER(customer_id) LIKE 'UK%' THEN 'United Kingdom'\n      ELSE country\n    END\n  ELSE country\nEND\n"
        },
        {
            "type": "formula",
            "id": "a_t9h1kb-I",
            "enabled": True,
            "target": "phone",
            "expression": "CASE\n  WHEN UPPER(customer_id) LIKE 'IND%' THEN\n    CONCAT('+91 ', REGEXP_REPLACE(REGEXP_REPLACE(phone, '^(\\\\+91|091|91)\\\\s*', ''), '(\\\\d{5})(\\\\d{5})', '$1 $2'))\n  WHEN UPPER(customer_id) LIKE 'USA%' THEN\n    CONCAT('+1 ', REGEXP_REPLACE(REGEXP_REPLACE(phone, '^(\\\\+1|1)\\\\s*', ''), '(\\\\d{3})(\\\\d{3})(\\\\d{4})', '($1) $2-$3'))\n  WHEN UPPER(customer_id) LIKE 'UK%' THEN\n    CONCAT('+44 ', REGEXP_REPLACE(REGEXP_REPLACE(phone, '^(\\\\+44|044|44|0)\\\\s*', ''), '(\\\\d{4})(\\\\d{6})', '$1 $2'))\n  ELSE phone\nEND\n"
        },
        {
            "type": "formula",
            "id": "a_DLve-X_t",
            "enabled": True,
            "target": "gender",
            "expression": "CASE WHEN gender IS NULL OR TRIM(gender) = '' THEN 'N' ELSE gender END"
        },
        {
            "type": "formula",
            "id": "a_DyKjqBgP",
            "enabled": True,
            "target": "acquisition_channel",
            "expression": "CASE\n  WHEN acquisition_channel IS NULL OR TRIM(acquisition_channel) = '' THEN 'Unknown'\n  ELSE acquisition_channel\nEND\n"
        },
        {
            "type": "multi_column_formula",
            "id": "a_UWq9rVwA",
            "enabled": True,
            "columns": [
                "first_name",
                "last_name",
                "email",
                "phone",
                "address",
                "country",
                "age",
                "gender",
                "status",
                "loyalty_tier",
                "registration_date",
                "acquisition_channel"
            ],
            "expression": "CASE WHEN {current_column} IS NULL OR TRIM(CAST({current_column} AS STRING)) = '' THEN 'N/A' ELSE {current_column} END",
            "prefix": "",
            "suffix": ""
        },
        {
            "type": "text_case",
            "id": "a_gcdojoVs",
            "enabled": True,
            "columns": [
                "first_name",
                "last_name",
                "gender",
                "acquisition_channel"
            ],
            "case": "title"
        },
        {
            "type": "text_case",
            "id": "a_6v9cJKxC",
            "enabled": True,
            "columns": [
                "status",
                "loyalty_tier"
            ],
            "case": "upper"
        },
        {
            "type": "trim",
            "id": "a_6Zj6hlKF",
            "enabled": True,
            "columns": [
                "first_name",
                "last_name",
                "address"
            ],
            "side": "both"
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["combine_31.combined_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_32.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_32" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_32.prepared_data"])
if "prepare_32" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_32", "counts": {"prepared_data": int(ctx["prepare_32.prepared_data"].count())}}}, raw=True)

In [0]:
"""
id: combine_30
template: combine
templateVersion: 2.0.0
name: bad_records_combine_all_countries
position:
  x: 1260
  y: 1112.5
description:
  text: Combine multiple datasets by including all rows and matching columns by name.
  hash: d61256d4
codegenHash: fcd9d3979c7d02f2
updatedAt: 1792941365087
config:
  operator: UNION
  quantifier: ALL
  union_by_name: true
  allow_missing_columns: true
checkpoint:
  mode: auto
input:
  - node: output_9a0f3c58
    input_port: data
    output_port: result
  - node: output_f2c47b3e
    input_port: data
    output_port: result
  - node: output_0c5f8e29
    input_port: data
    output_port: result
"""

# generated from the system
from functools import reduce
from typing import Any, Callable, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = list(inputs.get("data") or [])
    if len(dfs) < 1:
        raise ValueError(
            f"Combine requires at least 1 input, got {len(dfs)}. "
            "Wire one or more upstream operators into the 'data' port."
        )

    operator = config.get("operator", "UNION")
    quantifier = config.get("quantifier", "DISTINCT")
    union_by_name = bool(config.get("union_by_name", False))
    allow_missing_columns = bool(config.get("allow_missing_columns", False))

    pair: Callable[[Any, Any], Any]
    if operator == "UNION":
        if union_by_name:
            def pair(a: Any, b: Any) -> Any:
                return a.unionByName(b, allowMissingColumns=allow_missing_columns)
        else:
            def pair(a: Any, b: Any) -> Any:
                return a.union(b)
    elif operator == "INTERSECT":
        if quantifier == "ALL":
            def pair(a: Any, b: Any) -> Any:
                return a.intersectAll(b)
        else:
            def pair(a: Any, b: Any) -> Any:
                return a.intersect(b)
    elif operator in ("EXCEPT", "MINUS"):
        if quantifier == "ALL":
            def pair(a: Any, b: Any) -> Any:
                return a.exceptAll(b)
        else:
            def pair(a: Any, b: Any) -> Any:
                return a.subtract(b)
    else:
        raise ValueError(f"Unsupported combine operator: {operator}")

    result = reduce(pair, dfs)
    if quantifier == "DISTINCT" and (operator == "UNION" or len(dfs) == 1):
        result = result.distinct()
    return {"combined_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "operator": "UNION",
    "quantifier": "ALL",
    "union_by_name": True,
    "allow_missing_columns": True
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": [
        ctx["output_9a0f3c58.result"],
        ctx["output_f2c47b3e.result"],
        ctx["output_0c5f8e29.result"]
    ],
    "data__sources": [
        {
            "node": "output_9a0f3c58",
            "output_port": "result",
            "name": "bad_records_customer_ind",
            "df_name": "bad_records_customer_ind"
        },
        {
            "node": "output_f2c47b3e",
            "output_port": "result",
            "name": "bad_records_customer_usa",
            "df_name": "bad_records_customer_usa"
        },
        {
            "node": "output_0c5f8e29",
            "output_port": "result",
            "name": "bad_records_customer_uk",
            "df_name": "bad_records_customer_uk"
        }
    ]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["combine_30.combined_data"] = out["combined_data"]
if globals().get("ld_display_outputs", False) or "combine_30" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["combine_30.combined_data"])
if "combine_30" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "combine_30", "counts": {"combined_data": int(ctx["combine_30.combined_data"].count())}}}, raw=True)

In [0]:
"""
id: join_33
template: join
templateVersion: 3.0.0
name: join_33
position:
  x: 2820
  y: 340.5
description:
  text: Join three tables on specified keys and keep all matching rows with corresponding non-matching rows from each side.
  hash: 96aac636
previewCodeHash: d7b0411f0bf781f1
codegenHash: ea5821d113ebbb3b
updatedAt: 1792943531423
config:
  join_type: inner
  match_case: false
  multi_table: true
  conditions:
    - left_input: 1
      right_input: 0
      join_keys:
        - left: customer_id
          right: customer_id
      join_conditions: ""
    - left_input: 2
      right_input: 0
      join_keys:
        - left: product_id
          right: product_id
      join_conditions: ""
  columns:
    - edits: []
      ordered: []
    - edits: []
      ordered: []
    - edits: []
      ordered: []
checkpoint:
  mode: auto
input:
  - node: output_4c1e9b78
    input_port: data
    output_port: result
  - node: prepare_32
    input_port: data
    output_port: prepared_data
  - node: output_3b8e5c90
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Any, Dict, List, Optional
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(alias: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + alias + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(frames: List[Any]):
    out = []
    seen: Dict[str, int] = {}
    n = len(frames)
    for idx, (alias, df, cfg) in enumerate(frames):
        edits = _edits_by_column(cfg)
        for name in _ordered_names(list(df.columns), cfg):
            edit = edits.get(name)
            if edit is not None and not _is_checked(edit):
                continue
            col = _qualified_col(alias, name)
            user_alias = edit.get("alias") if edit else None
            if user_alias:
                out.append(col.alias(user_alias))
                seen[user_alias.lower()] = seen.get(user_alias.lower(), 0) + 1
                continue
            key = name.lower()
            if key not in seen:
                seen[key] = 1
                out.append(col.alias(name))
                continue
            new_name = ("right_" + name) if (n == 2 and idx == 1) else (name + "_" + str(idx + 1))
            while new_name.lower() in seen:
                new_name = new_name + "_dup"
            seen[new_name.lower()] = 1
            out.append(col.alias(new_name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    dfs: List[Any] = []
    for port in ("left", "right"):
        df = inputs.get(port)
        if df is not None:
            dfs.append(df)
    dfs.extend(inputs.get("data") or [])
    if len(dfs) < 2:
        raise ValueError(
            f"Join requires at least 2 inputs, got {len(dfs)}. "
            "Wire 2 or more upstream operators into this Join."
        )

    n = len(dfs)
    join_type = config.get("join_type") or "split_join"
    match_case = bool(config.get("match_case", False))
    conditions: List[Dict[str, Any]] = config.get("conditions") or []
    columns: List[Dict[str, Any]] = config.get("columns") or []

    aliased = [df.alias("t" + str(i)) for i, df in enumerate(dfs)]
    types = [{f.name.lower(): f.dataType for f in df.schema} for df in aliased]

    def key_col(idx: int, name: str):
        col = _qualified_col("t" + str(idx), name)
        if not match_case and isinstance(types[idx].get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    def edge_predicate(i: int, j: int, edge: Dict[str, Any]) -> Optional[Any]:
        predicates = []
        for key in edge.get("join_keys") or []:
            predicates.append(key_col(i, key["left"]) == key_col(j, key["right"]))
        condition = edge.get("join_conditions") or ""
        if condition:
            predicates.append(F.expr(condition))
        join_expr = None
        for predicate in predicates:
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    edges: List[Any] = [
        (int(edge.get("left_input", 0)), int(edge.get("right_input", 0)), edge)
        for edge in conditions
    ]

    predicates_by_step: Dict[int, List[Any]] = {}
    for i, j, edge in edges:
        predicate = edge_predicate(i, j, edge)
        if predicate is not None:
            predicates_by_step.setdefault(max(i, j), []).append(predicate)

    def predicate_at(step: int) -> Optional[Any]:
        join_expr = None
        for predicate in predicates_by_step.get(step, []):
            join_expr = predicate if join_expr is None else join_expr & predicate
        return join_expr

    def cfg(i: int) -> Dict[str, Any]:
        return columns[i] if i < len(columns) else {}

    is_two = n == 2
    is_split = is_two and join_type == "split_join"

    if is_split:
        join_expr = predicate_at(1)
        left_df, right_df = aliased[0], aliased[1]
        if join_expr is None:
            matched = left_df.join(right_df, how="inner")
            left_unmatched = left_df.join(right_df, how="left_anti")
            right_unmatched = right_df.join(left_df, how="left_anti")
        else:
            matched = left_df.join(right_df, join_expr, how="inner")
            left_unmatched = left_df.join(right_df, join_expr, how="left_anti")
            right_unmatched = right_df.join(left_df, join_expr, how="left_anti")
        projection = _projection([("t0", left_df, cfg(0)), ("t1", right_df, cfg(1))])
        if projection:
            matched = matched.select(*projection)
        return {
            "joined_data": matched,
            "left_unmatched": left_unmatched,
            "right_unmatched": right_unmatched,
        }

    how = join_type if is_two else "inner"
    if is_two:
        acc = aliased[0]
        predicate = predicate_at(1)
        if predicate is None:
            acc = acc.join(aliased[1], how=how)
        else:
            acc = acc.join(aliased[1], predicate, how=how)
    else:
        adjacency: Dict[int, List[int]] = {}
        for idx, (i, j, _edge) in enumerate(edges):
            adjacency.setdefault(i, []).append(idx)
            adjacency.setdefault(j, []).append(idx)
        visited = [False] * n
        visited[0] = True
        acc = aliased[0]
        used_edges = set()
        queue: List[int] = [0]
        while queue:
            cur = queue.pop(0)
            for idx in adjacency.get(cur, []):
                i, j, edge = edges[idx]
                nxt = j if i == cur else i
                if visited[nxt]:
                    continue
                visited[nxt] = True
                used_edges.add(idx)
                predicate = edge_predicate(i, j, edge)
                if predicate is None:
                    acc = acc.join(aliased[nxt], how=how)
                else:
                    acc = acc.join(aliased[nxt], predicate, how=how)
                queue.append(nxt)
        for idx, (i, j, edge) in enumerate(edges):
            if idx in used_edges:
                continue
            predicate = edge_predicate(i, j, edge)
            if predicate is not None:
                acc = acc.where(predicate)

    projection = _projection([("t" + str(i), aliased[i], cfg(i)) for i in range(n)])
    if projection:
        acc = acc.select(*projection)

    return {
        "joined_data": acc,
        "left_unmatched": spark.createDataFrame([], aliased[0].schema),
        "right_unmatched": spark.createDataFrame([], aliased[n - 1].schema),
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "match_case": False,
    "multi_table": True,
    "conditions": [
        {
            "left_input": 1,
            "right_input": 0,
            "join_keys": [
                {
                    "left": "customer_id",
                    "right": "customer_id"
                }
            ],
            "join_conditions": ""
        },
        {
            "left_input": 2,
            "right_input": 0,
            "join_keys": [
                {
                    "left": "product_id",
                    "right": "product_id"
                }
            ],
            "join_conditions": ""
        }
    ],
    "columns": [
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        },
        {
            "edits": [],
            "ordered": []
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": [
        ctx["output_4c1e9b78.result"],
        ctx["prepare_32.prepared_data"],
        ctx["output_3b8e5c90.result"]
    ],
    "data__sources": [
        {
            "node": "output_4c1e9b78",
            "output_port": "result",
            "name": "transactions clean",
            "df_name": "transactions_clean"
        },
        {
            "node": "prepare_32",
            "output_port": "prepared_data",
            "name": "Cleanse combined customers",
            "df_name": "Cleanse_combined_customers"
        },
        {
            "node": "output_3b8e5c90",
            "output_port": "result",
            "name": "products clean",
            "df_name": "products_clean"
        }
    ]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["join_33.joined_data"] = out["joined_data"]
ctx["join_33.left_unmatched"] = out["left_unmatched"]
ctx["join_33.right_unmatched"] = out["right_unmatched"]
# BEGIN LAKEBUILDER OUTPUT CHECKPOINT
_lb_cp_materialized = False
if not (config.get("meta_state", {}).get("is_disabled", False)):
    if callable(getattr(ctx["join_33.joined_data"], "localCheckpoint", None)) and callable(getattr(ctx["join_33.left_unmatched"], "localCheckpoint", None)) and callable(getattr(ctx["join_33.right_unmatched"], "localCheckpoint", None)):
        _lb_cp_original_0 = ctx["join_33.joined_data"]
        _lb_cp_original_1 = ctx["join_33.left_unmatched"]
        _lb_cp_original_2 = ctx["join_33.right_unmatched"]
        try:
            ctx["join_33.joined_data"] = ctx["join_33.joined_data"].localCheckpoint(eager=True)
            ctx["join_33.left_unmatched"] = ctx["join_33.left_unmatched"].localCheckpoint(eager=True)
            ctx["join_33.right_unmatched"] = ctx["join_33.right_unmatched"].localCheckpoint(eager=True)
            _lb_cp_materialized = True
        except Exception:
            if ctx["join_33.joined_data"] is not _lb_cp_original_0:
                try:
                    ctx["join_33.joined_data"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["join_33.joined_data"] = _lb_cp_original_0
            if ctx["join_33.left_unmatched"] is not _lb_cp_original_1:
                try:
                    ctx["join_33.left_unmatched"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["join_33.left_unmatched"] = _lb_cp_original_1
            if ctx["join_33.right_unmatched"] is not _lb_cp_original_2:
                try:
                    ctx["join_33.right_unmatched"].unpersist(blocking=False)
                except Exception:
                    pass
            ctx["join_33.right_unmatched"] = _lb_cp_original_2
# END LAKEBUILDER OUTPUT CHECKPOINT
if globals().get("ld_display_outputs", False) or "join_33" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["join_33.joined_data"])
    display(ctx["join_33.left_unmatched"])
    display(ctx["join_33.right_unmatched"])
if "join_33" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "join_33", "counts": {"joined_data": int(ctx["join_33.joined_data"].count()), "left_unmatched": int(ctx["join_33.left_unmatched"].count()), "right_unmatched": int(ctx["join_33.right_unmatched"].count())}}}, raw=True)

In [0]:
"""
id: prepare_34
template: prepare
templateVersion: 2.0.0
name: Standardize to USD & clean all columns
position:
  x: 3120
  y: 340.5
description:
  text: Convert currency values to USD, adjust prices and amounts accordingly, standardize text case for several columns, normalize weight units to kilograms, and replace missing text values with 'N/A'.
  hash: 83e344b8
previewCodeHash: 9c399c3e3b11ddd1
codegenHash: 7b3643f91c41b910
updatedAt: 1792944049815
config:
  actions:
    - type: formula
      id: a_yJ-atED9
      enabled: true
      target: usd_rate
      expression: |
        CASE
          WHEN UPPER(currency) IN ('GBP', '£') THEN 1.27
          WHEN UPPER(currency) = 'INR' THEN 0.012
          WHEN UPPER(currency) = 'USD' THEN 1.0
          ELSE 1.0
        END
    - type: formula
      id: a_ZDSARBxY
      enabled: true
      target: unit_price
      expression: ROUND(unit_price * usd_rate, 2)
    - type: formula
      id: a_lz6nPzLr
      enabled: true
      target: subtotal
      expression: ROUND(subtotal * usd_rate, 2)
    - type: formula
      id: a_TwtilTnh
      enabled: true
      target: discount_amount
      expression: ROUND(discount_amount * usd_rate, 2)
    - type: formula
      id: a_j8zl1fvw
      enabled: true
      target: tax_amount
      expression: ROUND(tax_amount * usd_rate, 2)
    - type: formula
      id: a_m3cz6RAc
      enabled: true
      target: total_amount
      expression: ROUND(total_amount * usd_rate, 2)
    - type: formula
      id: a_bwlpkwRg
      enabled: true
      target: unit_price_3
      expression: ROUND(unit_price_3 * usd_rate, 2)
    - type: formula
      id: a_Q9bPxO7I
      enabled: true
      target: list_price
      expression: ROUND(list_price * usd_rate, 2)
    - type: formula
      id: a_WFNNo_1Q
      enabled: true
      target: currency
      expression: "'USD'"
    - type: text_case
      id: a_OBQSWBV9
      enabled: true
      columns:
        - order_status
        - payment_method
        - channel
        - shipping_method
      case: title
    - type: text_case
      id: a_oxuMPagX
      enabled: true
      columns:
        - condition
        - availability_status
      case: upper
    - type: formula
      id: a_iieZyUfK
      enabled: true
      target: weight_value
      expression: |
        CASE
          WHEN LOWER(weight_unit) = 'g' THEN ROUND(weight_value / 1000.0, 3)
          WHEN LOWER(weight_unit) = 'oz' THEN ROUND(weight_value * 0.02835, 3)
          WHEN LOWER(weight_unit) = 'lbs' THEN ROUND(weight_value * 0.4536, 3)
          WHEN LOWER(weight_unit) = 'kg' THEN weight_value
          ELSE weight_value
        END
    - type: formula
      id: a_180hHKNw
      enabled: true
      target: weight_unit
      expression: CASE WHEN weight_unit IS NOT NULL AND TRIM(weight_unit) != '' THEN 'kg' ELSE weight_unit END
    - type: multi_column_formula
      id: a_qcXkiKLr
      enabled: true
      columns:
        - transaction_id
        - customer_id
        - product_id
        - payment_method
        - order_status
        - channel
        - shipping_method
        - coupon_code
        - warranty
        - product_name
        - category
        - brand
        - description
        - stock_quantity
        - availability_status
        - weight_unit
        - condition
      expression: CASE WHEN {current_column} IS NULL OR TRIM(CAST({current_column} AS STRING)) = '' THEN 'N/A' ELSE {current_column} END
      prefix: ""
      suffix: ""
checkpoint:
  mode: auto
input:
  - node: join_33
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Any, Callable, Dict, List, Optional, Tuple
import json
import re
from uuid import uuid4
import pyspark.sql.functions as F
import pyspark.sql.types as T

def _lb_run_bound_sql(df, spark, make_query):
    view_name = "_lb_input_" + uuid4().hex
    df.createOrReplaceTempView(view_name)
    try:
        query = make_query(view_name)
        args = {}
        if "dbutils" in globals() and hasattr(dbutils, "widgets") and hasattr(dbutils.widgets, "getAll"):
            args = dict(dbutils.widgets.getAll())
        evaluated = spark.sql(query, args=args)
    except Exception:
        spark.catalog.dropTempView(view_name)
        raise
    globals().setdefault("_lb_views", set()).add(view_name)
    return evaluated

def _lb_apply_bound_expressions(df, spark, assignments):
    occupied = set(df.columns) | set(assignments)
    temporary = {}
    for target in assignments:
        name = "_lb_expression_" + uuid4().hex
        while name in occupied:
            name = "_lb_expression_" + uuid4().hex
        occupied.add(name)
        temporary[target] = name
    expressions = [
        "(" + expression + "\n) AS " + chr(96) + temporary[target] + chr(96)
        for target, expression in assignments.items()
    ]
    evaluated = _lb_run_bound_sql(
        df, spark, lambda view: "SELECT *, " + ", ".join(expressions) + " FROM " + view
    )
    return evaluated.withColumns({target: F.col(name) for target, name in temporary.items()}).drop(*temporary.values())

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

CURRENT_COLUMN_PLACEHOLDER = "{current_column}"

JSON_PARSE_MODES = ("name_value", "typed_value", "flatten_array", "unnest")

JSON_PARSE_OUTPUT_SUFFIXES: Dict[str, List[str]] = {
    "name_value": ["_Name", "_ValueString"],
    "typed_value": [
        "_Name",
        "_ValueString",
        "_ValueInt",
        "_ValueFloat",
        "_ValueBool",
    ],
    "flatten_array": ["_flatten", "_idx"],
}

JSON_PARSE_SEPARATOR = "_"

JSON_VARIANT_DTYPE = "variant"

JSON_UNNEST_MAX_KEYS = 1024

INT64_MIN = -(2 ** 63)
INT64_MAX = 2 ** 63 - 1

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _action_columns(action: Dict[str, Any], action_type: str) -> List[str]:
    """Resolve an action's target columns from either accepted key.

    The "columns" list wins when it names anything, else the scalar
    "column". An empty list therefore can't mask a valid scalar. Empty
    names are dropped and repeats collapsed, since applying an action
    twice to a column is a no-op. Names are used verbatim (not
    stripped): a column identifier can legally carry edge spaces, so
    trimming would look up a different column than the one saved.
    Mirrors readColumnSelection in the notebook parser so the UI and
    the job resolve the same selection.
    """

    def _names(raw: Any) -> List[str]:
        if isinstance(raw, str):
            raw = [raw]
        if not isinstance(raw, (list, tuple)):
            raise ValueError(
                f"{action_type}: 'columns' must be a list of column names, "
                f"got {type(raw).__name__}"
            )
        names: List[str] = []
        for entry in raw:
            if not isinstance(entry, str):
                raise ValueError(
                    f"{action_type}: column names must be strings, got {entry!r}"
                )
            if entry and entry not in names:
                names.append(entry)
        return names

    columns = _names(action.get("columns", []))
    if not columns:
        columns = _names(action.get("column", ""))
    if not columns:
        raise ValueError(f"{action_type}: 'column' or 'columns' is required")
    return columns

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Coerce a user-provided value to the target column's type.

    UI always feeds text per the operator spec. try_cast, not cast, so a
    value that does not fit the column's type yields NULL instead of
    aborting the query under ANSI mode -- the empty default fill value
    against a numeric column then leaves that column's nulls in place, as
    the fill-null UI promises, rather than raising CAST_INVALID_INPUT.
    """
    if value is None:
        return F.lit(None).cast(target_dtype)
    escaped = str(value).replace(chr(92), chr(92) * 2).replace("'", chr(92) + "'")
    return F.expr("try_cast('" + escaped + "' as " + target_dtype + ")")

def _apply_formula(df, action: Dict[str, Any], spark):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, {target: expression})
    return df.withColumn(target, F.expr(expression))

def _quoted_or_comment_end(expression: str, start: int) -> int:
    end = len(expression)
    opening = expression[start]
    if opening in ("'", '"', "`"):
        index = start + 1
        while index < end:
            if expression[index] == "\\" and opening != "`":
                index += 2
                continue
            if expression[index] == opening:
                if index + 1 < end and expression[index + 1] == opening:
                    index += 2
                    continue
                return index + 1
            index += 1
        return end
    if expression.startswith("--", start):
        line_end = expression.find("\n", start)
        return end if line_end == -1 else line_end
    if expression.startswith("/*", start):
        comment_end = expression.find("*/", start + 2)
        return end if comment_end == -1 else comment_end + 2
    return start

def _placeholder_positions(expression: str) -> List[int]:
    positions: List[int] = []
    index = 0
    end = len(expression)
    while index < end:
        if expression.startswith(CURRENT_COLUMN_PLACEHOLDER, index):
            positions.append(index)
            index += len(CURRENT_COLUMN_PLACEHOLDER)
            continue
        skip_to = _quoted_or_comment_end(expression, index)
        index = skip_to if skip_to > index else index + 1
    return positions

def _expand_placeholders(
    expression: str, positions: List[int], quoted_column: str
) -> str:
    rendered: List[str] = []
    cursor = 0
    for position in positions:
        rendered.append(expression[cursor:position])
        rendered.append(quoted_column)
        cursor = position + len(CURRENT_COLUMN_PLACEHOLDER)
    rendered.append(expression[cursor:])
    return "".join(rendered)

def _apply_multi_column_formula(df, action: Dict[str, Any], spark):
    columns = action.get("columns") or []
    expression = action.get("expression", "")
    prefix = str(action.get("prefix") or "").strip()
    suffix = str(action.get("suffix") or "").strip()
    if isinstance(columns, str) or not isinstance(columns, (list, tuple)):
        raise ValueError(
            "multi_column_formula: 'columns' must be a list of column names, "
            f"got {type(columns).__name__}"
        )
    if not columns:
        raise ValueError("multi_column_formula: 'columns' is required")
    if not expression:
        raise ValueError("multi_column_formula: 'expression' is required")
    positions = _placeholder_positions(expression)
    if not positions:
        raise ValueError(
            "multi_column_formula: 'expression' must contain the "
            + CURRENT_COLUMN_PLACEHOLDER
            + " placeholder outside of string literals, quoted identifiers "
            + "and comments, e.g. '"
            + CURRENT_COLUMN_PLACEHOLDER
            + " * 1.1'"
        )
    assignments: Dict[str, Any] = {}
    for raw_column in columns:
        column = "" if raw_column is None else str(raw_column).strip()
        if not column:
            raise ValueError(
                "multi_column_formula: 'columns' must not contain blank entries"
            )
        _require_column(df, column, "multi_column_formula")
        rendered = _expand_placeholders(
            expression, positions, _quoted_ident(column)
        )
        assignments[f"{prefix}{column}{suffix}"] = rendered if action.get("__lb_bind_expression") else F.expr(rendered)
    if action.get("__lb_bind_expression"):
        return _lb_apply_bound_expressions(df, spark, assignments)
    return df.withColumns(assignments)

def _apply_cast(df, action: Dict[str, Any], column: str):
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return F.expr(f"try_cast({_quoted_ident(column)} as {to_type})")
    return _col(column).cast(to_type)

def _apply_replace_value(df, action: Dict[str, Any], column: str):
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return F.when(predicate, replacement).otherwise(_col(column))

def _apply_fill_null(df, action: Dict[str, Any], column: str):
    with_val = action.get("with", "")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return F.when(_col(column).isNull(), replacement).otherwise(_col(column))

def _apply_text_case(df, action: Dict[str, Any], column: str):
    case = action.get("case", "")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_trim(df, action: Dict[str, Any], column: str):
    side = action.get("side", "both")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return fn(_col(column))

def _apply_regex_replace(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    return F.regexp_replace(_col(column), pattern, replacement)

def _apply_extract(df, action: Dict[str, Any], column: str):
    pattern = action.get("pattern", "")
    group_raw = action.get("group", 0)
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return F.regexp_extract(_col(column), pattern, group_idx)

def _extract_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    """Destination for one extract application.

    Several extractions cannot share one destination, so "target" is
    honoured for a single source and ignored beyond that, where every
    source is rewritten in place.
    """
    if len(columns) > 1:
        return column
    return action.get("target") or column

def _apply_parse_date(df, action: Dict[str, Any], column: str):
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return F.try_to_timestamp(_col(column), F.lit(fmt))
            return F.try_to_timestamp(_col(column))
        if fmt:
            return F.to_date(_col(column), fmt)
        return F.expr(f"try_cast({_quoted_ident(column)} as date)")
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return fn(_col(column), fmt)
    return fn(_col(column))

def _unique_temp_name(df, base: str) -> str:
    """Name for an intermediate column that cannot shadow user data."""
    existing = set(df.columns)
    if base not in existing:
        return base
    suffix = 2
    while base + "_" + str(suffix) in existing:
        suffix += 1
    return base + "_" + str(suffix)

JSON_DECODE_ERRORS = (ValueError, TypeError, RecursionError)

JSON_PARSE_CONTAINERS = {
    "flatten_array": (list, "array"),
    "unnest": (dict, "object"),
}

def _json_container(text: Any, column: str, mode: str, strict: bool):
    """Decode one document that must be the container the mode expects.
    Returns None when the text is missing, undecodable or the wrong shape;
    raises for the latter two when strict."""
    expected, label = JSON_PARSE_CONTAINERS[mode]
    if _json_document_missing(text):
        return None
    try:
        document = json.loads(text)
    except JSON_DECODE_ERRORS as exc:
        if strict:
            raise ValueError(
                "parse_json: invalid JSON in column '" + column + "': " + str(exc)
            ) from exc
        return None
    if isinstance(document, expected):
        return document
    if strict:
        raise ValueError(
            "parse_json: mode '" + mode + "' expects a JSON " + label
            + " in column '" + column + "', got " + type(document).__name__
        )
    return None

def _json_leaf_pairs(text: str) -> List[Tuple[str, Any]]:
    """Depth-first (path, scalar) pairs for one JSON document.

    Paths use dotted notation with array positions as segments
    ("phoneNumber.0.type"). An empty object or array yields a single
    (path, None) pair so the row still records that the key existed.

    The walk is iterative rather than recursive so that traversal depth is
    bounded by memory, not by the interpreter's recursion limit. (json's own
    decoder still recurses, so documents nested deeper than that limit are
    reported as undecodable — see JSON_DECODE_ERRORS.)
    """
    pairs: List[Tuple[str, Any]] = []
    stack: List[Tuple[str, Any]] = [("", json.loads(text))]
    while stack:
        path, node = stack.pop()
        if isinstance(node, dict):
            if not node:
                pairs.append((path, None))
                continue
            for key in reversed(list(node.keys())):
                child = str(key) if not path else path + "." + str(key)
                stack.append((child, node[key]))
        elif isinstance(node, list):
            if not node:
                pairs.append((path, None))
                continue
            for index in range(len(node) - 1, -1, -1):
                child = str(index) if not path else path + "." + str(index)
                stack.append((child, node[index]))
        else:
            pairs.append((path, node))
    return pairs

def _json_text(value: Any) -> Optional[str]:
    """Render a JSON value as text: strings verbatim, everything else as
    compact JSON so booleans read "true"/"false" and containers keep their
    original form."""
    if value is None:
        return None
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, str):
        return value
    return json.dumps(value, separators=(",", ":"), ensure_ascii=False)

def _reject_json_constant(name: str) -> Any:
    raise ValueError("parse_json: refusing to read " + name + " as a number")

JSON_SCALAR_LEADS = "-0123456789tf"

def _json_scalar_from_text(text: str) -> Any:
    """Reinterpret a quoted scalar as the value it spells, or None when it
    spells nothing.

    Plenty of documents quote everything ({"active": "true", "age": "25"}),
    which would otherwise pile every leaf into the string column and leave
    the typed ones permanently empty. The test is JSON's own literal
    grammar, applied to the whole text: "25" is a number because JSON says
    so, while "007", "+5", "1_000" and " 25 " are not, and so keep their
    exact text rather than being silently rewritten. Only bool/int/float
    count -- "null" stays the word, never a missing value, and Infinity/NaN
    are refused outright since JSON has no such literals.
    """
    if not text or text[0] not in JSON_SCALAR_LEADS or text[-1].isspace():
        return None
    try:
        parsed = json.loads(text, parse_constant=_reject_json_constant)
    except JSON_DECODE_ERRORS:
        return None
    if parsed is True or parsed is False or isinstance(parsed, (int, float)):
        return parsed
    return None

def _json_typed(value: Any) -> Tuple[Any, Any, Any, Any]:
    """Spread a JSON scalar across (string, int, float, bool).

    Exactly one slot is filled, picked from the value's type, so an integer
    never also appears as a float. Integers too wide for BIGINT fall back to
    the string slot instead of overflowing. A quoted scalar is routed by
    what it spells rather than by its quoting, so {"a": true} and
    {"a": "true"} land in the same column -- see _json_scalar_from_text.
    """
    if value is None:
        return (None, None, None, None)
    if value is True or value is False:
        return (None, None, None, bool(value))
    if isinstance(value, str):
        spelled = _json_scalar_from_text(value)
        if spelled is None:
            return (value, None, None, None)
        return _json_typed(spelled)
    if isinstance(value, int):
        if INT64_MIN <= value <= INT64_MAX:
            return (None, value, None, None)
        return (str(value), None, None, None)
    if isinstance(value, float):
        return (None, None, value, None)
    return (_json_text(value), None, None, None)

def _json_document_missing(text: Any) -> bool:
    """NULL and blank text mean "no document" rather than "bad document", so
    they yield NULL outputs even when on_error is "error"."""
    return text is None or (isinstance(text, str) and not text.strip())

def _prefixed_output(prefix: str, suffix: str) -> str:
    """Name a generated column. With no prefix the separator goes too, so
    the column is "Name" rather than "_Name"."""
    if prefix:
        return prefix + suffix
    if suffix.startswith(JSON_PARSE_SEPARATOR):
        return suffix[len(JSON_PARSE_SEPARATOR):]
    return suffix

def _json_source_column(df, column: str):
    """The source column, as the JSON text a mode's UDF expects.

    A VARIANT column holds an already-parsed document rather than its text,
    so render it back with to_json. Any other type is handed over untouched:
    string is text already, and nothing else is offered as a source.
    """
    if _column_dtype(df, column) == JSON_VARIANT_DTYPE:
        return F.to_json(_col(column))
    return _col(column)

def _require_free_columns(outputs: List[str], retained: List[str]) -> None:
    collisions = [name for name in outputs if name in retained]
    if collisions:
        raise ValueError(
            f"parse_json: generated column(s) {collisions} would overwrite existing "
            f"columns. Set a different 'prefix'."
        )

def _parse_json_pairs(df, column: str, source, mode: str, outputs: List[str],
                      retained: List[str], strict: bool):
    typed = mode == "typed_value"
    fields = [
        T.StructField("name", T.StringType()),
        T.StructField("value_string", T.StringType()),
    ]
    if typed:
        fields.extend([
            T.StructField("value_int", T.LongType()),
            T.StructField("value_float", T.DoubleType()),
            T.StructField("value_bool", T.BooleanType()),
        ])

    def to_pairs(text):
        if _json_document_missing(text):
            return None
        try:
            pairs = _json_leaf_pairs(text)
        except JSON_DECODE_ERRORS as exc:
            if strict:
                raise ValueError(
                    "parse_json: invalid JSON in column '" + column + "': " + str(exc)
                ) from exc
            return None
        if typed:
            return [(path,) + _json_typed(value) for path, value in pairs]
        return [(path, _json_text(value)) for path, value in pairs]

    pairs_udf = F.udf(to_pairs, T.ArrayType(T.StructType(fields)))
    exploded = _unique_temp_name(df, "__parse_json")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.explode_outer(pairs_udf(source)).alias(exploded),
    ).select(
        *kept,
        *[
            _col(exploded).getField(field.name).alias(output)
            for field, output in zip(fields, outputs)
        ],
    )

def _parse_json_flatten_array(df, column: str, source, outputs: List[str],
                              retained: List[str], strict: bool):
    def to_elements(text):
        document = _json_container(text, column, "flatten_array", strict)
        if document is None:
            return None
        return [_json_text(element) for element in document]

    elements_udf = F.udf(to_elements, T.ArrayType(T.StringType()))
    position = _unique_temp_name(df, "__parse_json_pos")
    element = _unique_temp_name(df, "__parse_json_val")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        F.posexplode_outer(elements_udf(source)).alias(position, element),
    ).select(
        *kept,
        _col(element).alias(outputs[0]),
        (_col(position) + F.lit(1)).cast("int").alias(outputs[1]),
    )

def _parse_json_unnest(df, column: str, source, prefix: str,
                       retained: List[str], strict: bool):
    """Alteryx 'Unnest JSON field': one column per top-level key, each value
    kept as JSON text. One level only -- unnest again to go deeper.

    Alone among the modes, the output columns come from the data, so this
    runs one extra Spark job to discover the key names before it can build
    the projection. Keys are ordered by their first position within a
    document (ties broken by name) so the resulting schema is document-order
    like Alteryx's, yet stable across runs and partitionings.
    """
    def to_keys(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [(str(key), index) for index, key in enumerate(document.keys())]

    keys_udf = F.udf(
        to_keys,
        T.ArrayType(
            T.StructType([
                T.StructField("key", T.StringType()),
                T.StructField("position", T.IntegerType()),
            ])
        ),
    )
    found = _unique_temp_name(df, "__parse_json_key")
    discovered = (
        df.select(F.explode(keys_udf(source)).alias(found))
        .select(
            _col(found).getField("key").alias("key"),
            _col(found).getField("position").alias("position"),
        )
        .groupBy("key")
        .agg(F.min("position").alias("position"))
        .limit(JSON_UNNEST_MAX_KEYS + 1)
        .collect()
    )
    if len(discovered) > JSON_UNNEST_MAX_KEYS:
        raise ValueError(
            f"parse_json: mode 'unnest' found more than {JSON_UNNEST_MAX_KEYS} "
            f"distinct keys in column '{column}'. Keys that vary per row (ids, "
            f"timestamps) belong in rows, not columns -- use mode 'name_value' "
            f"for those."
        )
    keys = [
        row["key"]
        for row in sorted(discovered, key=lambda row: (row["position"], row["key"]))
    ]
    if not keys:
        raise ValueError(
            f"parse_json: mode 'unnest' found no keys to unnest in column '{column}'. "
            f"Every value was NULL, blank, or not a JSON object."
        )
    outputs = [prefix + "." + key if prefix else key for key in keys]
    _require_free_columns(outputs, retained)

    def to_values(text):
        document = _json_container(text, column, "unnest", strict)
        if document is None:
            return None
        return [_json_text(document.get(key)) for key in keys]

    values_udf = F.udf(to_values, T.ArrayType(T.StringType()))
    values = _unique_temp_name(df, "__parse_json_values")
    kept = [_col(name) for name in retained]
    return df.select(
        *kept,
        values_udf(source).alias(values),
    ).select(
        *kept,
        *[
            _col(values).getItem(index).alias(output)
            for index, output in enumerate(outputs)
        ],
    )

def _apply_parse_json(df, action: Dict[str, Any]):
    """Explode JSON into rows or columns, mirroring Alteryx's JSON Parse
    tool.

    Python UDFs on purpose: from_json / schema_of_json would need the
    document's schema at plan time, and inferring it means eagerly scanning
    the data, which would both cost a job per action and make the operator's
    output schema depend on the current data. The "unnest" mode accepts
    exactly that cost because its column names ARE the data; every other
    mode stays lazy. VARIANT does not change this -- see the parse_json
    design notes at the top of this file for why each mode reads text.
    """
    column = action.get("column", "")
    mode = action.get("mode", "name_value")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_json")
    if mode not in JSON_PARSE_MODES:
        raise ValueError(
            f"parse_json: unsupported mode '{mode}'. "
            f"Choose one of: {sorted(JSON_PARSE_MODES)}"
        )
    if on_error not in ("null", "error"):
        raise ValueError(
            f"parse_json: unsupported on_error '{on_error}'. Choose null or error."
        )
    prefix = (action.get("prefix") or "").strip()
    keep_source = action.get("keep_source", True) is not False
    retained = [name for name in df.columns if keep_source or name != column]
    strict = on_error == "error"
    source = _json_source_column(df, column)
    if mode == "unnest":
        return _parse_json_unnest(df, column, source, prefix, retained, strict)
    outputs = [
        _prefixed_output(prefix, suffix)
        for suffix in JSON_PARSE_OUTPUT_SUFFIXES[mode]
    ]
    _require_free_columns(outputs, retained)
    if mode == "flatten_array":
        return _parse_json_flatten_array(df, column, source, outputs, retained, strict)
    return _parse_json_pairs(df, column, source, mode, outputs, retained, strict)

VALID_RENAME_TRANSFORMS = {
    "add_prefix",
    "add_suffix",
    "lower",
    "upper",
    "trim",
    "remove_whitespace",
    "snake_case",
    "title_case",
    "sentence_case",
    "camel_case",
    "replace",
}

def _rename_words(name: str) -> List[str]:
    return [word for word in re.split(r"[^A-Za-z0-9]+", name) if word]

def _capitalize_word(word: str) -> str:
    return word[:1].upper() + word[1:].lower()

def _renamed(name: str, action: Dict[str, Any]) -> str:
    transform = action.get("transform", "")
    if transform == "add_prefix":
        return action.get("text", "") + name
    if transform == "add_suffix":
        return name + action.get("text", "")
    if transform == "lower":
        return name.lower()
    if transform == "upper":
        return name.upper()
    if transform == "trim":
        return name.strip()
    if transform == "remove_whitespace":
        return re.sub(r"\s+", "", name)
    if transform == "snake_case":
        return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    if transform == "title_case":
        return " ".join(_capitalize_word(word) for word in _rename_words(name))
    if transform == "sentence_case":
        words = _rename_words(name)
        if not words:
            return ""
        return " ".join([_capitalize_word(words[0])] + [word.lower() for word in words[1:]])
    if transform == "camel_case":
        words = _rename_words(name)
        if not words:
            return ""
        return words[0].lower() + "".join(_capitalize_word(word) for word in words[1:])
    if transform == "replace":
        find = action.get("find", "")
        if not find:
            return name
        return name.replace(find, action.get("replace_with", ""))
    return name

def _require_no_merged_columns(
    old_names: List[str], new_names: List[str]
) -> None:
    sources_by_name: Dict[str, List[str]] = {}
    display_by_name: Dict[str, str] = {}
    for old, new in zip(old_names, new_names):
        key = new.lower()
        sources_by_name.setdefault(key, []).append(old)
        display_by_name.setdefault(key, new)
    merged = [
        (display_by_name[key], sorted(sources))
        for key, sources in sources_by_name.items()
        if len({source.lower() for source in sources}) > 1
    ]
    if merged:
        details = "; ".join(
            f"{sources} -> {new!r}" for new, sources in sorted(merged)
        )
        raise ValueError(
            "rename_columns: the rename would give distinct columns the "
            f"same name ({details}). Spark matches column names "
            "case-insensitively. Narrow 'columns' or pick another transform."
        )

def _apply_rename_columns(df, action: Dict[str, Any]):
    transform = action.get("transform", "")
    if transform not in VALID_RENAME_TRANSFORMS:
        raise ValueError(
            f"rename_columns: unsupported transform '{transform}'. "
            f"Choose one of: {sorted(VALID_RENAME_TRANSFORMS)}"
        )
    targets = action.get("columns") or []
    if not isinstance(targets, list):
        raise ValueError(
            "rename_columns: 'columns' must be a list of column names"
        )
    old_names = list(df.columns)
    missing = [name for name in targets if name not in old_names]
    if missing:
        raise ValueError(
            f"rename_columns: column(s) {missing} not found in input data. "
            f"Available columns: {old_names}"
        )
    selected = set(targets)
    new_names = [
        _renamed(name, action) if not selected or name in selected else name
        for name in old_names
    ]
    blank = [old for old, new in zip(old_names, new_names) if not new]
    if blank:
        raise ValueError(
            f"rename_columns: transform '{transform}' leaves column(s) "
            f"{blank} with an empty name."
        )
    _require_no_merged_columns(old_names, new_names)
    if new_names == old_names:
        return df
    return df.toDF(*new_names)

COLUMN_ACTION_DISPATCH: Dict[str, Callable] = {
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def _in_place_target(action: Dict[str, Any], column: str, columns: List[str]) -> str:
    return column

ACTION_TARGET_RESOLVERS: Dict[str, Callable] = {
    "extract": _extract_target,
}

# formula names its own output, multi_column_formula writes one output per
# selected column, parse_json reshapes rows/columns, and rename_columns
# rewrites the column names, so all four sit outside the single-expression,
# in-place column dispatch and are routed on their own; every other action
# targets one column or several via the shared column dispatch below.
SUPPORTED_ACTION_TYPES = sorted(
    [
        "formula",
        "multi_column_formula",
        "parse_json",
        "rename_columns",
        *COLUMN_ACTION_DISPATCH,
    ]
)

def _apply_column_action(df, action: Dict[str, Any], action_type: str):
    """Apply one column-scoped action across all of its target columns.

    Every expression is built against the same input DataFrame and
    applied in one withColumns pass. Equivalent to chaining a withColumn
    per column, since no builder reads a column another target in the
    same action rewrites, but it adds one Project instead of N.
    """
    columns = _action_columns(action, action_type)
    build_expression = COLUMN_ACTION_DISPATCH[action_type]
    resolve_target = ACTION_TARGET_RESOLVERS.get(action_type, _in_place_target)
    updates: Dict[str, Any] = {}
    for column in columns:
        _require_column(df, column, action_type)
        target = resolve_target(action, column, columns)
        updates[target] = build_expression(df, action, column)
    return df.withColumns(updates)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        if action_type == "formula":
            df = _apply_formula(df, action, spark)
        elif action_type == "multi_column_formula":
            df = _apply_multi_column_formula(df, action, spark)
        elif action_type == "parse_json":
            df = _apply_parse_json(df, action)
        elif action_type == "rename_columns":
            df = _apply_rename_columns(df, action)
        elif action_type in COLUMN_ACTION_DISPATCH:
            df = _apply_column_action(df, action, action_type)
        else:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {SUPPORTED_ACTION_TYPES}"
            )
    return {"prepared_data": df}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        _ld_param = dbutils.widgets.getAll().get("ld_display_outputs")
        if _ld_param is not None and str(_ld_param).strip() != "":
            globals()["ld_display_outputs"] = str(_ld_param).strip().lower() not in ("false", "0", "no", "off")
        else:
            try:
                from dbruntime.databricks_repl_context import get_context
                globals()["ld_display_outputs"] = not get_context().isInJob
            except Exception:
                globals()["ld_display_outputs"] = True
    except Exception:
        globals()["ld_display_outputs"] = False
if "ld_display_outputs_for" not in globals():
    try:
        _ld_for = dbutils.widgets.getAll().get("ld_display_outputs_for")
        globals()["ld_display_outputs_for"] = frozenset(_p.strip() for _p in str(_ld_for).split(",") if _p.strip()) if _ld_for is not None else frozenset()
    except Exception:
        globals()["ld_display_outputs_for"] = frozenset()
if "_lb_collect_row_counts" not in globals():
    try:
        _lb_count_param = dbutils.widgets.getAll().get("_lb_collect_row_counts")
        globals()["_lb_collect_row_counts"] = _lb_count_param is not None and str(_lb_count_param).strip().lower() in ("true", "1", "yes", "on")
    except Exception:
        globals()["_lb_collect_row_counts"] = False
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "id": "a_yJ-atED9",
            "enabled": True,
            "target": "usd_rate",
            "expression": "CASE\n  WHEN UPPER(currency) IN ('GBP', '£') THEN 1.27\n  WHEN UPPER(currency) = 'INR' THEN 0.012\n  WHEN UPPER(currency) = 'USD' THEN 1.0\n  ELSE 1.0\nEND\n"
        },
        {
            "type": "formula",
            "id": "a_ZDSARBxY",
            "enabled": True,
            "target": "unit_price",
            "expression": "ROUND(unit_price * usd_rate, 2)"
        },
        {
            "type": "formula",
            "id": "a_lz6nPzLr",
            "enabled": True,
            "target": "subtotal",
            "expression": "ROUND(subtotal * usd_rate, 2)"
        },
        {
            "type": "formula",
            "id": "a_TwtilTnh",
            "enabled": True,
            "target": "discount_amount",
            "expression": "ROUND(discount_amount * usd_rate, 2)"
        },
        {
            "type": "formula",
            "id": "a_j8zl1fvw",
            "enabled": True,
            "target": "tax_amount",
            "expression": "ROUND(tax_amount * usd_rate, 2)"
        },
        {
            "type": "formula",
            "id": "a_m3cz6RAc",
            "enabled": True,
            "target": "total_amount",
            "expression": "ROUND(total_amount * usd_rate, 2)"
        },
        {
            "type": "formula",
            "id": "a_bwlpkwRg",
            "enabled": True,
            "target": "unit_price_3",
            "expression": "ROUND(unit_price_3 * usd_rate, 2)"
        },
        {
            "type": "formula",
            "id": "a_Q9bPxO7I",
            "enabled": True,
            "target": "list_price",
            "expression": "ROUND(list_price * usd_rate, 2)"
        },
        {
            "type": "formula",
            "id": "a_WFNNo_1Q",
            "enabled": True,
            "target": "currency",
            "expression": "'USD'"
        },
        {
            "type": "text_case",
            "id": "a_OBQSWBV9",
            "enabled": True,
            "columns": [
                "order_status",
                "payment_method",
                "channel",
                "shipping_method"
            ],
            "case": "title"
        },
        {
            "type": "text_case",
            "id": "a_oxuMPagX",
            "enabled": True,
            "columns": [
                "condition",
                "availability_status"
            ],
            "case": "upper"
        },
        {
            "type": "formula",
            "id": "a_iieZyUfK",
            "enabled": True,
            "target": "weight_value",
            "expression": "CASE\n  WHEN LOWER(weight_unit) = 'g' THEN ROUND(weight_value / 1000.0, 3)\n  WHEN LOWER(weight_unit) = 'oz' THEN ROUND(weight_value * 0.02835, 3)\n  WHEN LOWER(weight_unit) = 'lbs' THEN ROUND(weight_value * 0.4536, 3)\n  WHEN LOWER(weight_unit) = 'kg' THEN weight_value\n  ELSE weight_value\nEND\n"
        },
        {
            "type": "formula",
            "id": "a_180hHKNw",
            "enabled": True,
            "target": "weight_unit",
            "expression": "CASE WHEN weight_unit IS NOT NULL AND TRIM(weight_unit) != '' THEN 'kg' ELSE weight_unit END"
        },
        {
            "type": "multi_column_formula",
            "id": "a_qcXkiKLr",
            "enabled": True,
            "columns": [
                "transaction_id",
                "customer_id",
                "product_id",
                "payment_method",
                "order_status",
                "channel",
                "shipping_method",
                "coupon_code",
                "warranty",
                "product_name",
                "category",
                "brand",
                "description",
                "stock_quantity",
                "availability_status",
                "weight_unit",
                "condition"
            ],
            "expression": "CASE WHEN {current_column} IS NULL OR TRIM(CAST({current_column} AS STRING)) = '' THEN 'N/A' ELSE {current_column} END",
            "prefix": "",
            "suffix": ""
        }
    ]
}
config["meta_state"] = {"is_preview": False, "is_focused_preview": False, "is_disabled": False}
inputs = {
    "data": ctx["join_33.joined_data"]
}
if config["meta_state"]["is_disabled"]:
    def _lb_limit_zero(value):
        if hasattr(value, "limit"):
            return value.limit(0)
        if isinstance(value, list):
            return [_lb_limit_zero(item) for item in value]
        return value
    inputs = {k: _lb_limit_zero(v) for k, v in inputs.items()}
out = run(config, inputs, spark)
if config["meta_state"]["is_disabled"]:
    out = {k: _lb_limit_zero(v) for k, v in out.items()}
ctx["prepare_34.prepared_data"] = out["prepared_data"]
if globals().get("ld_display_outputs", False) or "prepare_34" in globals().get("ld_display_outputs_for", frozenset()):
    display(ctx["prepare_34.prepared_data"])
if "prepare_34" in globals().get("ld_display_outputs_for", frozenset()) and globals().get("_lb_collect_row_counts", False):
    __import__("IPython.display", fromlist=["display"]).display({"application/vnd.databricks.lakeflow-designer.row-counts+json": {"node": "prepare_34", "counts": {"prepared_data": int(ctx["prepare_34.prepared_data"].count())}}}, raw=True)

---
id: group_0
template: group
name: Customers – India (customer_ind)
position:
  x: 0
  y: 155
dimensions:
  width: 1160
  height: 358
---

---
id: group_5d2e8a1f
template: group
name: Customers – USA (customer_usa)
position:
  x: 0
  y: 573
dimensions:
  width: 1160
  height: 358
---

---
id: group_8c3f1d6a
template: group
name: Customers – UK (customer_uk)
position:
  x: 0
  y: 991
dimensions:
  width: 1160
  height: 358
---

---
id: group_e7a14c3b
template: group
name: Products (products)
position:
  x: 1560
  y: 0
dimensions:
  width: 1160
  height: 358
---

---
id: group_2b6f9e0d
template: group
name: Transactions (transactions)
position:
  x: 1560
  y: 418
dimensions:
  width: 1160
  height: 358
---